In [1]:

# This script implements the Trading Advisor Bot pipeline used in the thesis
# It supports both single-ticker and cross-sectional workfolws, including
# feature generation, time-based splits, model training, calibration,
# band based signal generation, entry-only gating and backtesting

# --- Imports & setup ---
import warnings, sys, os, re, random, platform, math, pickle
import numpy as np, pandas as pd, torch
import json


warnings.filterwarnings("ignore", message="Metric '.*' requires frequency to be set")
warnings.filterwarnings("ignore", message="Object has multiple columns.*")

import yfinance as yf
import vectorbt as vbt

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression

# Ensure sklearn keeps feature names / returns DataFrames where possible (silences warnings)
try:
    from sklearn import set_config
    set_config(transform_output="pandas")
except Exception:
    pass

from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from pandas.tseries.offsets import BDay  # for next-business-day timestamps

# Reproducibility settings:
# fix random seeds across Python, NumPy and PyTorch so repeated runs
# produce more stable and comparable results during testing
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
try:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

# ---------- Notification helper ----------
# used to surface important signal events locally during runs.
# if desktop notifications are unavailable, it falls back to console output.
def notify(title: str, message: str):
    sent = False
    try:
        from player import notification
        notification.notify(title=title, message=message, app_name="TF Signal", timeout=8)
        sent = True
    except Exception:
        pass
    if not sent:
        try:
            if platform.system() == "Windows":
                import winsound; winsound.MessageBeep()
        except Exception:
            pass
        print(f"[NOTIFY] {title}: {message}")

# Converts a ticker symbol into a filesystem-safe string
# so artifacts such as saved band can be written safely to disk.
def _safe_ticker_for_path(t: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", t)

# ---------- Yahoo daily normalization ----------
def _normalize_daily_index_us(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize yfinance daily index to US/Eastern calendar days (naive midnight) and drop dups.
    Prevents mixed T vs T-1 dates caused by timezone offsets.
    """
    df = df.copy()
    idx = pd.DatetimeIndex(df.index)
    if idx.tz is not None:
        idx = idx.tz_convert('America/New_York').tz_localize(None)
    idx = pd.to_datetime(idx.date)   # force to calendar date (midnight)
    df.index = idx
    df = df[~df.index.duplicated(keep='last')]  # keep the latest row if duplicates
    return df

# Determines the most recent usable US trading date for after-close logic.
# Thus avoids treating the current day as complete before the market close buffer has passed.
def latest_us_trading_date(buffer_min: int = 5) -> pd.Timestamp:
    """
    Return yesterday's US trading date (US/Eastern) unless today's close + buffer has passed.
    """
    now_et = pd.Timestamp.now(tz='America/New_York')
    d = now_et.normalize()
    # roll back weekends
    while d.weekday() >= 5:
        d = d - BDay(1)
    close_ts = d + pd.Timedelta(hours=16)
    if now_et < close_ts + pd.Timedelta(minutes=buffer_min):
        d = d - BDay(1)
    return d.tz_localize(None)

# ---------- Technical helpers ----------
# Average True Range helper used as a volatility proxy
# This is later used in both feature engineering and some target logic.
def _atr(df: pd.DataFrame, n=14):
    high, low, close = df['High'], df['Low'], df['Close']
    prev_close = close.shift(1)
    tr1 = (high - low).abs()
    tr2 = (high - prev_close).abs()
    tr3 = (low - prev_close).abs()
    tr = np.maximum(tr1, np.maximum(tr2, tr3))
    atr = tr.rolling(n).mean()
    return atr

# ---------- Features (precision-oriented & lean) ----------
# Feature engineeringn step:
# build a compact set of technical and return-based inputs
# that the model uses for daily directional prediction.
def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().ffill().bfill()
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].astype(float)

    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / loss.where(loss != 0, np.nan)
    df['RSI'] = 100 - (100 / (1 + rs))

    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26
    df['SignalLine'] = df['MACD'].ewm(span=9, adjust=False).mean()

    ma20 = df['Close'].rolling(20).mean()
    std20 = df['Close'].rolling(20).std()
    df['BB_upper'] = ma20 + 2 * std20
    df['BB_lower'] = ma20 - 2 * std20

    df['LogReturn'] = np.log(df['Close'] / df['Close'].shift(1))
    df['r1'] = df['Close'].pct_change(1)
    df['r5'] = df['Close'].pct_change(5)
    df['r10'] = df['Close'].pct_change(10)

    atr = _atr(df, n=14)
    df['ATRp'] = atr / df['Close']

    rng = df['BB_upper'] - df['BB_lower']
    numerator = df['Close'] - df['BB_lower']
    df['pctB'] = numerator / rng.where(rng != 0, np.nan)

    df['MA50'] = df['Close'].rolling(50).mean()
    df['distMA50'] = df['Close'] / df['MA50'] - 1.0
    df['MA200'] = df['Close'].rolling(200).mean()

    for col in ['RSI', 'MACD', 'SignalLine', 'BB_upper', 'BB_lower', 'LogReturn',
                'r1', 'r5', 'r10', 'ATRp', 'pctB', 'MA50', 'distMA50', 'MA200']:
        if isinstance(df[col], pd.DataFrame):
            raise ValueError(f"Column {col} is a DataFrame with shape {df[col].shape}, expected Series")
        elif not isinstance(df[col], pd.Series):
            raise ValueError(f"Column {col} is not a Series, got {type[df[col]]}")

    df.dropna(inplace=True)
    return df

# ---------- Splits ----------
# Simple chronological split by fractions.
# Data is kept in time order rather than shuffled to better match real deployment.
def time_splits(df: pd.DataFrame, val_frac=0.15, test_frac=0.15, purge_days=2):
    n = len(df)
    n_test = int(round(n * test_frac))
    n_val = int(round(n * val_frac))
    n_train = max(0, n - n_val - n_test)
    if n_train < 60:
        raise ValueError("Not enough data after splits; reduce seq_len or fractions.")
    train = df.iloc[:n_train].copy()
    val = df.iloc[n_train:n_train+n_val].copy()
    test = df.iloc[n_train+n_val:].copy()
    if purge_days > 0:
        if len(val) > purge_days: val = val.iloc[purge_days:]
        if len(test) > purge_days: test = test.iloc[purge_days:]
    return train, val, test

# Date-based chronological split.
# keeps train, validation and test periods seperated in time to reduce leakage
# and preserve a realistic train on the past, test on the future workflow.
def time_splits_by_date(df: pd.DataFrame, test_years=2, val_years=1, purge_days=2):
    last = df.index.max()
    test_start = last - pd.DateOffset(years=test_years)
    val_start = test_start - pd.DateOffset(years=val_years)

    train = df.loc[:val_start - pd.Timedelta(days=1)].copy()
    val = df.loc[val_start:test_start - pd.Timedelta(days=1)].copy()
    test = df.loc[test_start:].copy()

    if purge_days > 0:
        if len(val) > purge_days: val = val.iloc[purge_days:]
        if len(test) > purge_days: test = test.iloc[purge_days:]

    # Ensure test includes the latest date in df
    if test.index.max() < df.index.max():
        test = df.loc[test_start:df.index.max()].copy()
        if purge_days > 0 and len(test) > purge_days:
            test = test.iloc[purge_days:]

    if min(map(len, [train, val, test])) < 60:
        raise ValueError("One of the splits is too short; reduce seq_len or adjust years.")
    
    print(f"Train range: {train.index.min()} to {train.index.max()}")
    print(f"Val range: {val.index.min()} to {val.index.max()}")
    print(f"Test range: {test.index.min()} to {test.index.max()}")
    return train, val, test

# ---------- Targets ----------
# Target construction:
# the pipeline supports several labelling schemes depending on the experiment,
# including fixed-horizon, tripple-barrier, risk-adjusted and excess-return targets.
def add_targets_horizon_inplace(df: pd.DataFrame, H=3, eps=0.003):
    df['FwdRet_H'] = np.log(df['Close'].shift(-H) / df['Close'])
    y = np.where(df['FwdRet_H'] > eps, 1,
                 np.where(df['FwdRet_H'] < -eps, 0, np.nan))
    df['Target'] = y
    df.dropna(subset=['Target'], inplace=True)
    df['Target'] = df['Target'].astype(int)

def add_targets_triple_barrier_inplace(df: pd.DataFrame, H=5, atr_mult=1.0):
    """
    Robust daily triple-barrier labeling.
    - Upper/lower barriers: Close ± atr_mult * ATR(14)
    - Horizon H (in bars) ahead.
    - If both barriers are touched on the same bar, label is set to NaN (tie / unknown order).
    """
    df = df.copy()
    atr = _atr(df, n=14)
    up = df['Close'] + atr_mult * atr
    dn = df['Close'] - atr_mult * atr

    highs = df['High'].values
    lows  = df['Low'].values
    up_v  = up.values
    dn_v  = dn.values

    n = len(df)
    tgt = np.full(n, np.nan, dtype=float)

    for i in range(n - H):
        hi_seq = highs[i+1:i+H+1]
        lo_seq = lows[i+1:i+H+1]
        up_hit_idx = np.where(hi_seq >= up_v[i])[0]
        dn_hit_idx = np.where(lo_seq <= dn_v[i])[0]

        if up_hit_idx.size == 0 and dn_hit_idx.size == 0:
            continue
        if up_hit_idx.size > 0 and (dn_hit_idx.size == 0 or up_hit_idx[0] < dn_hit_idx[0]):
            tgt[i] = 1.0
        elif dn_hit_idx.size > 0 and (up_hit_idx.size == 0 or dn_hit_idx[0] < up_hit_idx[0]):
            tgt[i] = 0.0
        else:
            # same-bar hit: ambiguous on daily data -> skip (remain NaN)
            pass

    df['Target'] = tgt
    df.dropna(subset=['Target'], inplace=True)
    df['Target'] = df['Target'].astype(int)
    df['FwdRet_H'] = np.log(df['Close'].shift(-H) / df['Close'])
    # write columns back into the original frame
    for col in ['Target', 'FwdRet_H']:
        df_col = df[col]
        df_col = df_col.reindex(df.index)
        try:
            pass
        except Exception:
            pass

# ---------- NEW TARGETS (extras) ----------
# Small helper to compute forward log return over chosen horizon.
# Used by alternative target-generation methods.
def _forward_log_return(df: pd.DataFrame, H: int) -> pd.Series:
    return np.log(df['Close'].shift(-H) / df['Close'])

def add_targets_risk_adjusted_sign_inplace(df: pd.DataFrame,
                                           H_min=5, H_max=10,
                                           risk_window=20,
                                           eps=0.0):
    """
    Label by the sign of average forward Sharpe-like return over horizons H_min..H_max.
    FwdRiskAdj = mean_H [ log(C_{t+H}/C_t) / (roll_std(LogReturn, risk_window) * sqrt(H)) ]

    Returns a new DataFrame with ['FwdRiskAdj','Target'] added.
    """
    df = df.copy()
    r = df['LogReturn'] if 'LogReturn' in df.columns else np.log(df['Close']/df['Close'].shift(1))
    vol = r.rolling(risk_window).std()

    sharps = []
    for H in range(H_min, H_max + 1):
        f = _forward_log_return(df, H)
        sharps.append(f / (vol * np.sqrt(H)))
    F = pd.concat(sharps, axis=1).mean(axis=1)
    df['FwdRiskAdj'] = F
    y = np.where(F > eps, 1, np.where(F < -eps, 0, np.nan))
    df['Target'] = y
    df.dropna(subset=['Target'], inplace=True)
    df['Target'] = df['Target'].astype(int)
    for col in ['FwdRiskAdj', 'Target']:
        df[col] = df[col].astype(float) if col == 'FwdRiskAdj' else df[col]
    return df

def add_targets_excess_sign_inplace(df: pd.DataFrame, market_close: pd.Series,
                                    H=10, eps=0.0):
    """
    Label by sign of forward EXCESS return vs market (e.g., SPY) over H days.
    Returns a new DataFrame (copy) with ['FwdExcessRet','Target'].
    """
    df = df.copy()
    mrk = market_close.reindex(df.index).ffill()
    f_stock = _forward_log_return(df, H)
    f_mkt = np.log(mrk.shift(-H) / mrk)
    excess = f_stock - f_mkt
    df['FwdExcessRet'] = excess
    y = np.where(excess > eps, 1, np.where(excess < -eps, 0, np.nan))
    df['Target'] = y
    df.dropna(subset=['Target'], inplace=True)
    df['Target'] = df['Target'].astype(int)
    return df

# ---------- Scaling ----------
# Scaling is fitted on the train split only and then applied to validation and test.
# This avoids leaking information from later periods into earlier model preparation.
def scale_by_train(train, val, test, features):
    scaler = RobustScaler()
    try:
        scaler.set_output(transform="pandas")
    except Exception:
        pass
    train[features] = scaler.fit_transform(train[features])
    val[features] = scaler.transform(val[features])
    test[features] = scaler.transform(test[features])
    return scaler, train, val, test

# ---------- Dataset ----------
# Converts tabular daily data into rolling sequence for the Transformer model.
# Each sample uses seq_len rows of features and the label anchored at the sequence end.
class TimeSeriesDataset(Dataset):
    def __init__(self, df, features, seq_len=30):
        self.seq_len = seq_len
        self.features = features
        X_list, y_list, idx = [], [], []
        if len(df) <= seq_len:
            self.X = torch.empty(0, seq_len, len(features))
            self.y = torch.empty(0, dtype=torch.long)
            self.index = pd.DatetimeIndex([])
            return
        for i in range(len(df) - seq_len):
            end = i + seq_len - 1                        # last feature row (decision time t)
            X_list.append(df[features].iloc[i:i+seq_len].values)
            y_list.append(int(df['Target'].iloc[end]))   # label anchored at t
            idx.append(df.index[end])                    # index = decision time t

        self.X = torch.tensor(np.stack(X_list), dtype=torch.float32)
        self.y = torch.tensor(np.array(y_list), dtype=torch.long)
        self.index = pd.Index(idx)

    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

# ---------- Datasets with regime features & cross-sectional ----------
# Extension of the rollingn dataset that also attaches regime-level context
# for the same decision timestamp, used by the cross-sectional MoE model.
class TimeSeriesDatasetWithRegime(Dataset):
    def __init__(self, df, features, regime_df, seq_len=30):
        self.seq_len = seq_len
        self.features = features
        self.regime_df = regime_df
        X_list, y_list, r_list, idx = [], [], [], []
        if len(df) <= seq_len:
            self.X = torch.empty(0, seq_len, len(features))
            self.R = torch.empty(0, regime_df.shape[1])
            self.y = torch.empty(0, dtype=torch.long)
            self.index = pd.DatetimeIndex([])
            return
        for i in range(len(df) - seq_len):
            end = i + seq_len - 1            # decision time t
            tgt_ts = df.index[end]
            X_list.append(df[features].iloc[i:i+seq_len].values)
            y_list.append(int(df['Target'].iloc[end]))
            r_list.append(regime_df.reindex([tgt_ts]).iloc[0].values)
            idx.append(tgt_ts)
        self.X = torch.tensor(np.stack(X_list), dtype=torch.float32)
        self.R = torch.tensor(np.stack(r_list), dtype=torch.float32)
        self.y = torch.tensor(np.array(y_list), dtype=torch.long)
        self.index = pd.Index(idx)

    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.R[idx], self.y[idx]

# Aggregate per-ticker rolling datasets into one pooled cross-sectional dataset
# so shared model can be trained across many names at once.
class CrossSectionalDataset(Dataset):
    def __init__(self, per_ticker_ds: dict):
        self.samples = []
        for t, ds in per_ticker_ds.items():
            for i in range(len(ds)):
                self.samples.append((t, ds.X[i], ds.R[i], ds.y[i], ds.index[i]))
        if len(self.samples) == 0:
            self.X = torch.empty(0,1,1); self.R = torch.empty(0,1); self.y = torch.empty(0, dtype=torch.long)
            self.tickers = []; self.index = pd.DatetimeIndex([]); return
        self.X = torch.stack([s[1] for s in self.samples])
        self.R = torch.stack([s[2] for s in self.samples])
        self.y = torch.stack([s[3] for s in self.samples])
        self.tickers = [s[0] for s in self.samples]
        self.index = pd.Index([s[4] for s in self.samples])
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.R[idx], self.y[idx]

# ---------- Mixture-of-Experts model (regime-aware head) ----------
# Standard sinusoidal positional encoding so the Transformer can distinguish
# where each row sits inside the rollong input window.
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1024):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        S = x.size(1)
        return x + self.pe[:, :S, :]

# Regime-aware mixture of experts classifier.
# A shared sequence backbone produces a latent representation, while
# regime inputs determine how expert outputs are weighted for the final decision.
class RegimeMoEClassifier(nn.Module):
    def __init__(self, in_features, seq_len=30, d_model=64, nhead=4, num_layers=2,
                 r_features=4, n_experts=3):
        super().__init__()
        self.seq_len = seq_len
        self.input_proj = nn.Linear(in_features, d_model)
        self.pos = PositionalEncoding(d_model, max_len=seq_len)
        enc = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=128, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers)

        rep_dim = seq_len * d_model
        self.backbone_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(rep_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.experts = nn.ModuleList([nn.Linear(128, 2) for _ in range(n_experts)])
        self.gate = nn.Sequential(
            nn.Linear(r_features, 16),
            nn.ReLU(),
            nn.Linear(16, n_experts)
        )

    def forward(self, x, r):
        x = self.input_proj(x)
        x = self.pos(x)
        x = self.transformer(x)
        h = self.backbone_head(x)
        weights = torch.softmax(self.gate(r), dim=1)
        logits_stack = torch.stack([exp(h) for exp in self.experts], dim=1)  # (B, n_exp, 2)
        logits = (logits_stack * weights.unsqueeze(-1)).sum(dim=1)           # (B, 2)
        return logits

# Simpler single ticker Transformer classifier used for the single name pipeline.
# It predicts a binary directional outcome from the rolling feature window alone.
class TransformerClassifier(nn.Module):
    def __init__(self, in_features, seq_len=30, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.seq_len = seq_len
        self.input_proj = nn.Linear(in_features, d_model)
        self.pos = PositionalEncoding(d_model, max_len=seq_len)
        enc = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=128, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(seq_len*d_model, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos(x)
        x = self.transformer(x)
        return self.classifier(x)

# ---------- Loss: Focal ----------
# Focal loss places more emphasis on harder examples.
# it can help when classes are imbalanced or when easy cases dominate training.
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super().__init__(); self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(logits, targets, reduction='none')
        p = torch.softmax(logits, dim=1).gather(1, targets.view(-1,1)).squeeze()
        loss = self.alpha * (1 - p).pow(self.gamma) * ce
        return loss.mean()

# ---------- Train / Early stopping ----------
# Training loop with early stopping.
# The best model weights are kept based on validation loss to reduce overfitting.
def train_with_early_stopping(model, Xtr, ytr, Xva, yva, epochs=150, lr=1e-4, patience=10,
                              seed=SEED, batch_size=64, weights_tr: torch.Tensor=None,
                              use_focal=True):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    base_loss = FocalLoss() if use_focal else nn.CrossEntropyLoss(reduction='none')
    g = torch.Generator().manual_seed(SEED)

    if weights_tr is None:
        weights_tr = torch.ones(len(Xtr), dtype=torch.float32)
    ds = TensorDataset(Xtr, ytr, weights_tr)
    tr_loader = DataLoader(ds, batch_size=batch_size, shuffle=True, generator=g)

    best_w, best_val = None, float('inf'); bad = 0
    for ep in range(1, epochs+1):
        model.train(); tot = 0.0; batches = 0
        for xb, yb, wb in tr_loader:
            xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
            opt.zero_grad()
            logits = model(xb)
            if isinstance(base_loss, FocalLoss):
                loss = base_loss(logits, yb)
            else:
                loss_vec = base_loss(logits, yb)
                loss = (loss_vec * wb).mean()
            loss.backward(); opt.step()
            tot += loss.item(); batches += 1
        model.eval()
        with torch.no_grad():
            v_logits = model(Xva.to(device))
            vloss = (FocalLoss().to(device)(v_logits, yva.to(device))
                     if isinstance(base_loss, FocalLoss)
                     else nn.CrossEntropyLoss()(v_logits, yva.to(device)))
        print(f"Epoch {ep:03d} | train_loss={tot/max(batches,1):.4f} | val_loss={vloss:.4f}")
        if vloss < best_val:
            best_val, bad = vloss.item() if torch.is_tensor(vloss) else float(vloss), 0
            best_w = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad > patience:
                print(f"Early stopping at epoch {ep}")
                break
    if best_w is not None:
        model.load_state_dict(best_w)
    return model

# ---------- Threshold selection (precision) ----------
# Searches for a treshold that improves precision while keeping
# at least a minimum number of positive signals.
def pick_threshold_for_precision(p, y, min_signals=12):
    thrs = np.linspace(0.60, 0.90, 61)
    best = (0.0, 0.50, 0)
    for t in thrs:
        yhat = (p >= t).astype(int)
        n = int(yhat.sum())
        if n < min_signals:
            continue
        prec = precision_score(y, yhat, zero_division=0)
        if prec > best[0]:
            best = (prec, float(t), n)
    return best

def pick_threshold_from_val(model, Xva, yva):
    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        p = torch.softmax(model(Xva.to(device)), dim=1)[:, 1].cpu().numpy()
    thr_grid = np.linspace(0.45, 0.65, 41)
    scores = [f1_score(yva.numpy(), (p >= t).astype(int)) for t in thr_grid]
    t_best = float(thr_grid[int(np.argmax(scores))])
    return t_best

# ---------- Platt calibration ----------
# Platt calibration converts raw model probabilities into calibrated probabilities
# using a logistic regression fitted on validation outputs.
class PlattCalibrator:
    def __init__(self):
        self.lr = None
    def fit(self, p_val: np.ndarray, y_val: np.ndarray):
        y_unique = np.unique(y_val)
        if len(y_unique) < 2:
            self.lr = None
            return self
        x = np.clip(p_val.reshape(-1,1), 1e-6, 1-1e-6)
        self.lr = LogisticRegression(solver='lbfgs')
        self.lr.fit(x, y_val)
        return self
    def transform(self, p: np.ndarray) -> np.ndarray:
        if self.lr is None:
            return p
        x = np.clip(p.reshape(-1,1), 1e-6, 1-1e-6)
        return self.lr.predict_proba(x)[:,1]

def platt_fit_apply(p_val: np.ndarray, y_val: np.ndarray, p_test: np.ndarray):
    cal = PlattCalibrator().fit(p_val, y_val)
    return cal.transform(p_test), cal

# ---------- Helper: robust label -> numpy ----------
def _as_numpy_1d(y):
    if isinstance(y, np.ndarray):
        arr = y
    elif hasattr(y, "detach"):
        arr = y.detach().cpu().numpy()
    elif hasattr(y, "numpy"):
        arr = y.numpy()
    else:
        arr = np.asarray(y)
    return arr.astype(int).reshape(-1)

# ---------- Eval helpers ----------
def eval_classification(model, X, y, thr):
    y_np = _as_numpy_1d(y)
    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        p = torch.softmax(model(X.to(device)), dim=1)[:, 1].cpu().numpy()
    yhat = (p >= thr).astype(int)
    return {
        "acc": accuracy_score(y_np, yhat),
        "prec": precision_score(y_np, yhat, zero_division=0),
        "rec": recall_score(y_np, yhat, zero_division=0),
        "f1": f1_score(y_np, yhat, zero_division=0),
        "probs": p,
        "yhat": yhat
    }

# ---------- "Today" inference ----------
# Convenience helper for obtaining the latest model probability
# fom the most recent seq_len rows of available feature data.
def infer_today_prob(df_full_feats: pd.DataFrame, features, scaler: RobustScaler, model, seq_len=30):
    window = df_full_feats[features].iloc[-seq_len:].copy()
    if len(window) < seq_len:
        return None, None
    window_scaled = scaler.transform(window)
    if isinstance(window_scaled, pd.DataFrame):
        window_scaled = window_scaled.to_numpy()
    X_live = torch.tensor(window_scaled, dtype=torch.float32).unsqueeze(0)
    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        prob_up = torch.softmax(model(X_live.to(device)), dim=1)[:, 1].item()
    return df_full_feats.index[-1], float(prob_up)

# --- Live inference on arbitrary dates (single-ticker model) ---
def infer_live_probs_basic(df_full_feats: pd.DataFrame, features, scaler: RobustScaler,
                           model, seq_len: int, dates: pd.DatetimeIndex) -> pd.Series:
    dates = pd.DatetimeIndex(dates)
    device = next(model.parameters()).device
    out = []
    for d in dates:
        if d not in df_full_feats.index:
            out.append(np.nan); continue
        end_loc = df_full_feats.index.get_loc(d)
        if isinstance(end_loc, slice):
            end_loc = end_loc.stop - 1
        start = end_loc - (seq_len - 1)
        if start < 0:
            out.append(np.nan); continue
        window = df_full_feats[features].iloc[start:end_loc+1]
        X = scaler.transform(window)
        if isinstance(X, pd.DataFrame):
            X = X.to_numpy()
        X = torch.tensor(X, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            p = torch.softmax(model(X), dim=1)[:, 1].item()
        out.append(p)
    return pd.Series(out, index=dates, name="prob_up").dropna()

# --- Live inference on arbitrary dates (cross-sectional MoE model + regime) ---
def infer_live_probs_cs(df_full_feats: pd.DataFrame, features, scaler: RobustScaler,
                        model, regime_df: pd.DataFrame, seq_len: int,
                        dates: pd.DatetimeIndex) -> pd.Series:
    dates = pd.DatetimeIndex(dates)
    device = next(model.parameters()).device
    out = []
    for d in dates:
        if d not in df_full_feats.index:
            out.append(np.nan); continue
        end_loc = df_full_feats.index.get_loc(d)
        if isinstance(end_loc, slice):
            end_loc = end_loc.stop - 1
        start = end_loc - (seq_len - 1)
        if start < 0:
            out.append(np.nan); continue
        window = df_full_feats[features].iloc[start:end_loc+1]
        X = scaler.transform(window)
        if isinstance(X, pd.DataFrame):
            X = X.to_numpy()
        X = torch.tensor(X, dtype=torch.float32).unsqueeze(0).to(device)
        r = regime_df.reindex([d]).iloc[0].values.reshape(1, -1)
        R = torch.tensor(r, dtype=torch.float32).to(device)
        with torch.no_grad():
            p = torch.softmax(model(X, R), dim=1)[:, 1].item()
        out.append(p)
    return pd.Series(out, index=dates, name="prob_up").dropna()

# ---------- Band (hysteresis) ----------
# Hysteresis band logic.
# entring a position requires a stronger signal than continuing to hold one
# This reduces frequent flipping caused by small probability changes.
def state_from_band(probs: np.ndarray, idx: pd.Index, thr_entry=0.78, thr_exit=0.48):
    state = np.zeros_like(probs, dtype=bool)
    for i, p in enumerate(probs):
        if i == 0:
            state[i] = (p >= thr_entry)
        else:
            state[i] = (p >= (thr_exit if state[i-1] else thr_entry))
    return pd.Series(state, index=idx, name='state')

def eval_classification_band(probs, y, idx, thr_entry, thr_exit):
    y_np = _as_numpy_1d(y)
    s = state_from_band(probs, idx, thr_entry, thr_exit)
    yhat = s.astype(int).values
    return {
        "acc": accuracy_score(y_np, yhat),
        "prec": precision_score(y_np, yhat, zero_division=0),
        "rec": recall_score(y_np, yhat, zero_division=0),
        "f1": f1_score(y_np, yhat, zero_division=0),
        "probs": probs,
        "yhat": yhat,
        "state": s
    }

# ---------- Band artifacts + entry-only gating + vol gate helpers ----------
def _safe_band_path(ticker: str, out_dir: str) -> str:
    os.makedirs(out_dir, exist_ok=True)
    return os.path.join(out_dir, f"{_safe_ticker_for_path(ticker)}_band.json")

def save_band(ticker: str, best: dict, out_dir: str):
    """
    best = {"entry": float, "exit": float, ...}
    """
    path = _safe_band_path(ticker, out_dir)
    with open(path, "w") as f:
        json.dump(best, f, indent=2)

def load_band(ticker: str, out_dir: str):
    path = _safe_band_path(ticker, out_dir)
    with open(path, "r") as f:
        j = json.load(f)
    return float(j["entry"]), float(j["exit"]), j

# Entry only gate
# risk filters can block a new entry, but they do not automatically close
# a position that is already active
def entry_only_gate(prev_final: bool, raw_curr: bool,
                    trend_ok_today: bool, vol_ok_today: bool,
                    override_used: bool) -> bool:
    """
    Gates checked only on entries (flat -> long).
    """
    is_entry = (not prev_final) and raw_curr
    if is_entry:
        return bool(raw_curr and ((trend_ok_today and vol_ok_today) or override_used))
    else:
        return bool(raw_curr)  # once in, gates don't force exit

# Applies entry only gating across a full state series.
# The result is final tradable state after risk filters are considered.
def apply_entry_only_gates(raw_state: pd.Series,
                           trend_gate: pd.Series,
                           vol_gate: pd.Series,
                           override: pd.Series) -> pd.Series:
    """
    Build final state where gates can only *block new entries*.
    """
    idx = raw_state.index
    trend_gate = trend_gate.reindex(idx).fillna(False)
    vol_gate   = vol_gate.reindex(idx).fillna(False)
    override   = override.reindex(idx).fillna(False)

    out = []
    prev_final = False
    for i, d in enumerate(idx):
        raw_curr = bool(raw_state.iloc[i])
        if (not prev_final) and raw_curr:
            passed = bool((trend_gate.iloc[i] and vol_gate.iloc[i]) or override.iloc[i])
            final = raw_curr and passed
        else:
            final = raw_curr
        out.append(final)
        prev_final = final
    return pd.Series(out, index=idx, name="state")

# Rolling realized volatility estimate used as a practical volatility filter
# when deciding whether new entries should be allowed.
def realized_vol(close: pd.Series, lookback: int = 60) -> pd.Series:
    """Annualized rolling realized volatility from daily pct returns."""
    ret = close.pct_change()
    return (ret.rolling(lookback, min_periods=lookback//2).std() * np.sqrt(252.0)).rename("rv")

def fit_vol_gate_threshold(vol_train: pd.Series, q_high: float = 0.80) -> float:
    vol_train = vol_train.dropna()
    return float(vol_train.quantile(q_high)) if len(vol_train) else np.inf


# ---------- Backtest ----------
# Backtest helper.
# evaluates the state series as a trading strategy and supports next open execution, 
# which matches the after close signal generation design used in the project.
def backtest_state(price_s: pd.Series, state: pd.Series,
                   init_cash=10_000, fees=0.001, slippage=0.001,
                   execution: str = "next_open"):  # "next_open" or "same_close"
    state = state.astype(bool)
    prev = state.shift(1, fill_value=False)
    entries = (state & ~prev)
    exits   = (~state & prev)

    if execution == "next_open":
        entries = entries.shift(1, fill_value=False)
        exits   = exits.shift(1, fill_value=False)

    idx = price_s.index.union(entries.index).union(exits.index)
    price_s = price_s.reindex(idx).astype(float)
    entries = entries.reindex(idx, fill_value=False).astype(bool)
    exits   = exits.reindex(idx, fill_value=False).astype(bool)

    if entries.any():
        last_true_idx = entries.index[entries].max()
        if not exits.any() or exits.index[exits].max() < last_true_idx:
            exits.iloc[-1] = True

    pf = vbt.Portfolio.from_signals(price_s, entries, exits,
                                    init_cash=init_cash, fees=fees, slippage=slippage)

    def _get_returns_and_curve(_pf):
        ret_s, curve_s = None, None
        for cand in ('returns',):
            obj = getattr(_pf, cand, None)
            if obj is None: continue
            try:
                ret_s = obj() if callable(obj) else obj
                if isinstance(ret_s, (pd.Series, pd.DataFrame)):
                    ret_s = ret_s.squeeze()
                    break
            except Exception:
                pass
        for cand in ('value', 'asset_value', 'portfolio_value'):
            obj = getattr(_pf, cand, None)
            if obj is None: continue
            try:
                curve_s = obj() if callable(obj) else obj
                if isinstance(curve_s, (pd.Series, pd.DataFrame)):
                    curve_s = curve_s.squeeze()
                    break
            except Exception:
                pass
        if (ret_s is None) and (curve_s is not None):
            try:
                ret_s = curve_s.pct_change().dropna()
            except Exception:
                pass
        return ret_s, curve_s

    try:
        stats = pf.stats(freq='1D')
    except TypeError:
        stats = pf.stats()
        rets, curve = _get_returns_and_curve(pf)
        if ('Volatility [%]' not in stats.index) or pd.isna(stats.get('Volatility [%]', np.nan)):
            if isinstance(rets, pd.Series) and len(rets) > 0:
                vol_pct = float(rets.std(ddof=0) * math.sqrt(252.0) * 100.0)
                stats.loc['Volatility [%]'] = vol_pct
        if ('Max Drawdown [%]' not in stats.index) or pd.isna(stats.get('Max Drawdown [%]', np.nan)):
            try:
                if isinstance(curve, pd.Series) and len(curve) > 1:
                    dd = (curve / curve.cummax() - 1.0).min() * 100.0
                    stats.loc['Max Drawdown [%]'] = float(dd)
            except Exception:
                pass
        wr = stats.get('Win Rate [%]', np.nan)
        if pd.isna(wr):
            try:
                rec = pf.trades.records_readable
                wins = int((rec['PnL'] > 0).sum()); total = int(len(rec))
                stats.loc['Win Rate [%]'] = (100.0 * wins / max(total, 1)) if total > 0 else np.nan
            except Exception:
                pass

    return pf, stats

# ---------- Auto-tune band on validation ----------
# Searches for entry/exit band combinations on the validation period.
# The selected band is later reused in testing and live-style evaluation.
def pick_band_on_val(p_va, idx_va, close_series, y_va=None, min_trades=5,
                     entry_grid=np.linspace(0.62, 0.90, 29), gap=0.25,
                     init_cash=10_000, fees=0.001, slippage=0.001,
                     objective="winrate"):
    best = None
    close_va = close_series.reindex(idx_va).dropna()
    for e in entry_grid:
        x = max(0.40, e - gap)
        state = state_from_band(p_va, idx_va, thr_entry=e, thr_exit=x)
        pf, stats = backtest_state(close_va, state, init_cash=init_cash, fees=fees, slippage=slippage)
        trades = float(stats.get('Total Trades', 0.0))
        print(f"Testing band: entry={e:.3f}, exit={x:.3f}, trades={trades}")
        if trades < min_trades:
            continue

        if objective == "winrate":
            wr = stats.get('Win Rate [%]', np.nan)
            if pd.isna(wr):
                try:
                    rec = pf.trades.records_readable
                    wins = int((rec['PnL'] > 0).sum()); total = int(len(rec))
                    wr = 100.0 * wins / max(total, 1)
                except Exception:
                    wr = -np.inf
            score = wr
        elif objective == "precision":
            if y_va is None:
                continue
            m_va = eval_classification_band(p_va, y_va, idx_va, e, x)
            score = 100.0 * m_va['prec']
        else:  # return
            score = float(stats['Total Return [%]'])
        cand = (score, e, x, trades, stats)
        if (best is None) or (cand[0] > best[0]):
            best = cand
    return best

# ---------- Risk-aware band search with constraints ----------
def _risk_score_from_stats(stats, lam=0.3, gamma=0.2):
    ret = float(stats.get('Total Return [%]', 0.0))
    vol = float(stats.get('Volatility [%]', stats.get('Volatility', 0.0)))
    mdd = float(stats.get('Max Drawdown [%]', stats.get('Max Drawdown', 0.0)))
    if np.isnan(ret): ret = 0.0
    if np.isnan(vol): vol = 0.0
    if np.isnan(mdd): mdd = 0.0
    return ret - lam * vol - gamma * abs(mdd)

def _rr_precheck_by_atr(df_feat: pd.DataFrame, entries_state: pd.Series,
                        target_mult=1.5, stop_mult=1.0, atr_col='ATRp',
                        max_fail_rate=0.5):
    entries = entries_state[entries_state].index
    if len(entries) < 3:
        return False
    atrp = df_feat[atr_col].reindex(entries).dropna()
    if len(atrp) == 0:
        return False
    ok1 = (target_mult > stop_mult * 1.1)
    ok2 = (atrp.median() < 0.06)
    ok3 = (atrp.quantile(0.9) < 0.12)
    fail_rate = 1.0 - float((atrp < 0.08).mean())
    return bool(ok1 and ok2 and ok3 and (fail_rate <= max_fail_rate))

# Risk aware band search.
# selects validation bands using a return versus risk objective
# rather than only pure classification quality.
def pick_band_on_val_riskaware(p_va, idx_va, close_series, df_feat_val,
                               min_trades=20, enforce_entry_min=0.60,
                               gap=0.25, lam=0.3, gamma=0.2,
                               use_trend_gate=True, use_vol_gate=True,
                               trend_gate_s=None, vol_gate_s=None,
                               gate_relax_penalty=10.0,
                               init_cash=10_000, fees=0.001, slippage=0.001):
    best = None
    close_va = close_series.reindex(idx_va).dropna()
    if len(close_va) == 0:
        return None
    gate_penalty = 0.0
    if not use_trend_gate or not use_vol_gate:
        gate_penalty += gate_relax_penalty

    for e in np.linspace(max(0.50, enforce_entry_min-0.05), 0.90, 36):
        x = max(0.40, e - gap)
        state = state_from_band(p_va, idx_va, thr_entry=e, thr_exit=x)

        if e < enforce_entry_min:
            entries = (state & ~state.shift(1, fill_value=False))
            if not _rr_precheck_by_atr(df_feat_val, entries, target_mult=1.6, stop_mult=1.0):
                continue

        gated_state = state.copy()
        if use_trend_gate and trend_gate_s is not None:
            gated_state &= trend_gate_s.reindex(gated_state.index).fillna(False)
        if use_vol_gate and vol_gate_s is not None:
            gated_state &= vol_gate_s.reindex(gated_state.index).fillna(False)

        pf, stats = backtest_state(close_va, gated_state,
                                   init_cash=init_cash, fees=fees, slippage=slippage,
                                   execution="next_open")
        trades = float(stats.get('Total Trades', 0.0))
        if trades < min_trades:
            continue
        score = _risk_score_from_stats(stats, lam=lam, gamma=gamma) - gate_penalty
        cand = (score, e, x, trades, stats)
        print(f"[riskaware] entry={e:.3f} exit={x:.3f} trades={trades:.0f} score={score:.2f}")
        if (best is None) or (cand[0] > best[0]):
            best = cand
    return best

# ---------- Baseline: MA momentum + ATR-style exit ----------
# Baseline strategy used for comparison with ML strategy
# This provides a simpler rule based reference point in evaluation.
def baseline_ma_atr_signals(df_feat: pd.DataFrame,
                            fast=50, slow=200,
                            atr_mult_exit=1.5):
    ma_fast = df_feat['MA50']
    ma_slow = df_feat['MA200']
    atr = df_feat['ATRp'] * df_feat['Close']
    momentum_state = (ma_fast > ma_slow) & (ma_slow.pct_change(20) > 0)

    exit_line = ma_fast - atr_mult_exit * (atr.fillna(0.0))
    early_exit = df_feat['Close'] < exit_line

    st = pd.Series(False, index=df_feat.index)
    prev = False
    for i in range(len(df_feat)):
        if not prev and momentum_state.iloc[i]:
            prev = True
            st.iloc[i] = True
        elif prev:
            if (not momentum_state.iloc[i]) or early_exit.iloc[i]:
                prev = False
                st.iloc[i] = False
            else:
                st.iloc[i] = True
    return st.rename('baseline_state')

# ---------- Pretty helpers for per-ticker reporting ----------
def _print_prob_stats(name, probs):
    if probs is None or len(probs) == 0:
        print(f"[{name}] No probabilities to summarize.")
        return
    print(f"[{name}] prob_up range: min={np.min(probs):.3f} | max={np.max(probs):.3f} | mean={np.mean(probs):.3f}")

def _print_signal_summary(ticker, state_series: pd.Series):
    if state_series is None or len(state_series) == 0:
        print(f"[{ticker}] No state to summarize.")
        return
    entries = (state_series & ~state_series.shift(1, fill_value=False))
    exits   = (~state_series & state_series.shift(1, fill_value=False))
    long_ratio = float(state_series.mean())
    print(f"[{ticker}] signals: entries={int(entries.sum())}, exits={int(exits.sum())}, long-day%={100*long_ratio:.1f}%")

def _print_bt_subset(ticker, stats):
    fields = [
        'Start', 'End', 'Period', 'Start Value', 'End Value',
        'Total Return [%]', 'Max Drawdown [%]', 'Volatility [%]',
        'Total Trades', 'Win Rate [%]', 'Profit Factor', 'Expectancy'
    ]
    print(f"\n--- {ticker} backtest summary (freq=1D) ---")
    for f in fields:
        if f in stats.index:
            print(f"{f:>20}: {stats[f]}")

# ---------- NEW: After-close decision helpers ----------
def _next_business_day(d: pd.Timestamp) -> pd.Timestamp:
    try:
        return (pd.Timestamp(d) + BDay(1)).normalize()
    except Exception:
        return pd.Timestamp(d) + pd.Timedelta(days=1)

# Converts latest model and gate state into a user facing instruction
# such as enter next open, hold, exit next open or remain flat.
def _format_after_close_message(ticker: str,
                                live_date: pd.Timestamp,
                                prev_final: bool,
                                curr_final: bool,
                                curr_raw: bool,
                                prob_today: float,
                                thr_entry: float,
                                thr_exit: float,
                                trend_ok_today: bool,
                                vol_ok_today: bool,
                                topn_kept_today: bool,
                                override_used: bool,
                                live_close: float) -> str:
    """Return a single-line instruction tagged with the latest close date and close price."""
    date_str = pd.Timestamp(live_date).date()
    px_txt = f" | close={live_close:,.2f}"

    # Transition flat -> long: enter at the NEXT session's open
    if (not prev_final) and curr_final:
        return (f"Enter Next Opening ({ticker}) ({date_str})  | prob_up={prob_today:.2%} | "
                f"band≥{thr_entry:.3f} | gates trend={'OK' if trend_ok_today else 'X'}, "
                f"vol={'OK' if vol_ok_today else 'X'}{px_txt}")

    # Stay long
    if prev_final and curr_final:
        return f"Hold Long Position ({ticker}) ({date_str})  | prob_up={prob_today:.2%}{px_txt}"

    # Transition long -> flat: exit at the NEXT session's open
    if prev_final and (not curr_final):
        reason = "band exit" if (not curr_raw) else "gate exit"
        return (f"Exit Next Opening ({ticker}) ({date_str})  | {reason}, "
                f"prob_up={prob_today:.2%}, exit<{thr_exit:.3f}{px_txt}")

    # Band passed but entry-only gates blocked the new position
    if curr_raw and (not curr_final):
        cause = []
        if not trend_ok_today: cause.append("trend gate")
        if not vol_ok_today:   cause.append("vol gate")
        if not topn_kept_today:cause.append("top-N cap")
        if override_used:      cause = []  # override bypasses gates
        cause_txt = " & ".join(cause) if cause else "policy filter"
        return (f"Flat: Band passed but {cause_txt} blocked ({ticker}) ({date_str})  | "
                f"prob_up={prob_today:.2%}{px_txt}")

    # Nothing to do
    return (f"Flat: No long (below entry band) ({ticker}) ({date_str})  | "
            f"prob_up={prob_today:.2%}, entry≥{thr_entry:.3f}{px_txt}")


# --- HOTFIX: ensure triple-barrier target truly mutates the passed DataFrame in place ---
def add_targets_triple_barrier_inplace(df: pd.DataFrame, H=5, atr_mult=1.0):
    """
    Robust daily triple-barrier labeling (in-place).
    - Upper/lower barriers: Close ± atr_mult * ATR(14)
    - Horizon H (in bars) ahead.
    - If both barriers are touched on the same bar, label is set to NaN (tie / unknown order).
    """
    atr = _atr(df, n=14)
    up = df['Close'] + atr_mult * atr
    dn = df['Close'] - atr_mult * atr

    highs = df['High'].to_numpy()
    lows  = df['Low'].to_numpy()
    up_v  = up.to_numpy()
    dn_v  = dn.to_numpy()

    n = len(df)
    tgt = np.full(n, np.nan, dtype=float)

    for i in range(n - H):
        hi_seq = highs[i+1:i+H+1]
        lo_seq = lows[i+1:i+H+1]
        up_hit_idx = np.where(hi_seq >= up_v[i])[0]
        dn_hit_idx = np.where(lo_seq <= dn_v[i])[0]

        if up_hit_idx.size == 0 and dn_hit_idx.size == 0:
            continue
        if up_hit_idx.size > 0 and (dn_hit_idx.size == 0 or up_hit_idx[0] < dn_hit_idx[0]):
            tgt[i] = 1.0
        elif dn_hit_idx.size > 0 and (up_hit_idx.size == 0 or dn_hit_idx[0] < up_hit_idx[0]):
            tgt[i] = 0.0
        else:
            pass

    df['Target'] = tgt
    df.dropna(subset=['Target'], inplace=True)
    df['Target'] = df['Target'].astype(int)
    df['FwdRet_H'] = np.log(df['Close'].shift(-H) / df['Close'])

# ---------- Market context download (SPY & VIX) ----------
# Downloads market wide context series used for regime features,
# currenlty SPY for broad market direction and VIX for volatility context.
def _download_market(period="15y"):
    # Use ADJUSTED OHLC for SPY to avoid split/dividend artifacts
    spy = yf.download(
        "SPY", period=period, interval="1d",
        auto_adjust=True, prepost=False, actions=False,
        threads=False, progress=False
    )
    if isinstance(spy.columns, pd.MultiIndex):
        spy.columns = [c[0] for c in spy.columns]
    spy = _normalize_daily_index_us(spy)
    spy_close = spy["Close"].astype(float).rename("SPY_Close").ffill().bfill()

    # VIX doesn't need adjustment; keep as-is
    vix = yf.download(
        "^VIX", period=period, interval="1d",
        auto_adjust=False, prepost=False, actions=False,
        threads=False, progress=False
    )
    if isinstance(vix.columns, pd.MultiIndex):
        vix.columns = [c[0] for c in vix.columns]
    vix = _normalize_daily_index_us(vix)
    vix_close = vix["Close"].astype(float).rename("VIX_Close").ffill().bfill()

    return spy_close, vix_close


# ---------- Regime features ----------
# Builds market regime features shared across the universe,
# including broad trend, volatility z-score and recent market return.
def compute_regime_features_by_date(per_ticker_df: dict, spy_close: pd.Series, vix_close: pd.Series):
    """
    Build regime features on the union of dates across all tickers (needs Close, MA50 in per_ticker_df).
    """
    all_idx = None
    for df in per_ticker_df.values():
        all_idx = df.index if all_idx is None else all_idx.union(df.index)
    all_idx = pd.DatetimeIndex(sorted(all_idx))

    spy = spy_close.reindex(all_idx).ffill()
    vix = vix_close.reindex(all_idx).ffill()

    spy_ma200 = spy.rolling(200).mean()
    trend = ((spy > spy_ma200) & (spy_ma200.pct_change(20) > 0)).astype(float)

    vix_z = (vix - vix.rolling(60).mean()) / (vix.rolling(60).std() + 1e-9)

    # Breadth: % of tickers above their MA50
    above_ma50 = []
    for t, df in per_ticker_df.items():
        ma50 = df['MA50'].reindex(all_idx)
        close = df['Close'].reindex(all_idx)
        above_ma50.append((close > ma50).astype(float))
    breadth = (pd.concat(above_ma50, axis=1).mean(axis=1)
               if len(above_ma50) > 0 else pd.Series(0.5, index=all_idx))

    mkt_ret20 = np.log(spy / spy.shift(20))

    reg = pd.DataFrame({
        "REG_trend_up": trend,
        "REG_vix_z": vix_z.clip(-4, 4),
        "REG_breadth": breadth,
        "REG_mkt_ret20": mkt_ret20.fillna(0.0)
    }, index=all_idx).ffill().fillna(0.0)
    return reg

# ---------- Cross-sectional quantile calibration & top-N ----------
# Cross-sectional calibration:
# rescales probabilities within each date so signals are interpreted
# relative to the rest of the universe on that day.
def cs_quantile_calibrate(dates: pd.Index, tickers: list, probs: np.ndarray, blend=0.5):
    df = pd.DataFrame({"date": dates, "ticker": tickers, "p": probs})
    out = np.zeros_like(probs, dtype=float)
    for d, grp in df.groupby("date"):
        n = len(grp)
        if n == 0:
            continue
        ranks = grp['p'].rank(method='average')  # 1..n
        q = (ranks - 0.5) / n
        raw = grp['p'].values
        cal = blend * q.values + (1.0 - blend) * raw
        out[grp.index.values] = cal
    return out

# Keeps only the top-N highest probability candidates on each date.
# This can be used to cap the number of simultaneous positions across the universe.
def top_n_mask_by_date(dates: pd.Index, probs: np.ndarray, N: int):
    df = pd.DataFrame({"date": dates, "p": probs})
    keep = np.zeros_like(probs, dtype=bool)
    for d, grp in df.groupby('date'):
        if N <= 0:
            continue
        idx_sorted = grp['p'].sort_values(ascending=False).index[:N]
        keep[idx_sorted.values] = True
    return keep

# ---------- Universe training & calibration ----------
# Main cross-sectional workflow.
# builds per ticker features and labels, construsts shared regime inputs,
# trains one pooled model, calibrates probabilities, applies bands and gates,
# and evaluates the strategy per ticker.
def run_universe_pipeline(tickers: list,
                          seq_len=30,
                          epochs=120,
                          patience=8,
                          data_period="10y",
                          split_mode="by_date",
                          val_years=1, test_years=2,
                          target_mode="riskadj_sign",
                          H=10, eps=0.0,
                          H_min=5, H_max=10,
                          use_band=True,
                          thr_entry=0.65,
                          thr_exit=0.40,
                          auto_band_from_val=True,
                          band_objective="riskaware",   # 'riskaware', 'precision', 'winrate', 'return'
                          min_trades_val=20,
                          lam=0.3, gamma=0.2,
                          use_trend_gate=True,
                          use_vol_gate=True,
                          atrp_min=0.004, atrp_max=0.10,
                          use_focal_loss=True,
                          weight_by_future_abs=True,
                          evaluate_baseline=True,
                          override_gate_prob=None,
                          top_n_per_date: int = 0,
                          init_cash=10_000, fees=0.001, slippage=0.001):
    print("\n=== Cross-Sectional Pipeline ===")
    mkt_period = "max" if str(data_period).lower() == "max" else (
        data_period if (isinstance(data_period, str) and data_period.endswith("y")
                        and int(re.sub(r'[^0-9]', '', data_period) or 0) >= 15) else "15y")
    spy_close, vix_close = _download_market(period=mkt_period)

    # 1) Build per-ticker **full features** and **labeled** frames
    per_full = {}   # features only (for regime/gates & latest day)
    per = {}        # labeled frames (for training)
    for t in tickers:
        try:
            df_raw = yf.download(t, period=data_period, interval="1d",
                                 auto_adjust=True, prepost=False, actions=False,
                                 threads=False, progress=False)
            if isinstance(df_raw.columns, pd.MultiIndex):
                df_raw.columns = [c[0] for c in df_raw.columns]
            df_raw = df_raw[['Open','High','Low','Close','Volume']].ffill().bfill()
            df_raw = _normalize_daily_index_us(df_raw)
            if len(df_raw) < 300:
                print(f"{t}: not enough history"); continue
            df_feat = compute_features(df_raw)
            per_full[t] = df_feat.copy()

            # Label targets on a copy
            if target_mode == "riskadj_sign":
                df_lab = add_targets_risk_adjusted_sign_inplace(df_feat, H_min=H_min, H_max=H_max, eps=eps)
            elif target_mode == "excess_sign":
                df_lab = add_targets_excess_sign_inplace(df_feat, market_close=spy_close, H=H, eps=eps)
            elif target_mode == "horizon":
                df_lab = df_feat.copy()
                add_targets_horizon_inplace(df_lab, H=max(H_min,5), eps=eps)
            else:
                df_lab = df_feat.copy()
                add_targets_triple_barrier_inplace(df_lab, H=max(H_min,5), atr_mult=1.0)

            per[t] = df_lab
        except Exception as e:
            print(f"{t}: download/feature error -> {e}")

    if len(per) == 0:
        raise RuntimeError("No usable tickers for cross-sectional training.")

    # 2) Regime features on union of dates
    regime_df = compute_regime_features_by_date(per_full, spy_close, vix_close)

    # 3) Split each name
    splits = {}
    for t, df in per.items():
        try:
            if split_mode == "by_date":
                tr, va, te = time_splits_by_date(df, test_years=test_years, val_years=val_years, purge_days=max(2, H_max))
            else:
                tr, va, te = time_splits(df, purge_days=max(2, H_max))
            splits[t] = (tr, va, te)
        except Exception as e:
            print(f"{t}: split error -> {e}")

    # 4) Features list
    features = [
        'LogReturn','RSI','MACD','SignalLine','BB_upper','BB_lower',
        'r1','r5','r10','ATRp','pctB','distMA50'
    ]

    # 5) Fit scaler on stacked TRAIN across names, then transform each split
    train_stack = pd.concat([splits[t][0][features] for t in splits if len(splits[t][0])>0], axis=0)
    scaler = RobustScaler()
    try:
        scaler.set_output(transform="pandas")
    except Exception:
        pass
    scaler.fit(train_stack)

    per_scaled = {}
    for t, (tr, va, te) in splits.items():
        tr_scaled = tr.copy(); va_scaled = va.copy(); te_scaled = te.copy()
        tr_scaled[features] = scaler.transform(tr_scaled[features])
        va_scaled[features] = scaler.transform(va_scaled[features])
        te_scaled[features] = scaler.transform(te_scaled[features])
        per_scaled[t] = (tr_scaled, va_scaled, te_scaled)

    # 6) Build per-name datasets with regime inputs
    per_ds = {}
    for t, (tr, va, te) in per_scaled.items():
        dtr = TimeSeriesDatasetWithRegime(tr, features, regime_df, seq_len)
        dva = TimeSeriesDatasetWithRegime(va, features, regime_df, seq_len)
        dte = TimeSeriesDatasetWithRegime(te, features, regime_df, seq_len)
        if len(dtr)==0 or len(dva)==0 or len(dte)==0:
            continue
        per_ds[t] = {"train": dtr, "val": dva, "test": dte}

    if len(per_ds) == 0:
        raise RuntimeError("Datasets empty after seq_len.")

    # 7) Assemble cross-sectional datasets
    dtr_xs = CrossSectionalDataset({t: per_ds[t]["train"] for t in per_ds})
    dva_xs = CrossSectionalDataset({t: per_ds[t]["val"] for t in per_ds})
    dte_xs = CrossSectionalDataset({t: per_ds[t]["test"] for t in per_ds})

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = RegimeMoEClassifier(in_features=len(features), seq_len=seq_len,
                                r_features=4, n_experts=3).to(device)

    # 8) Train
    def _train_loop(model, dtr, dva, epochs, patience, use_focal):
        opt = optim.Adam(model.parameters(), lr=1e-4)
        base_loss = FocalLoss() if use_focal else nn.CrossEntropyLoss()
        g = torch.Generator().manual_seed(SEED)
        tr_loader = DataLoader(TensorDataset(dtr.X, dtr.R, dtr.y), batch_size=64, shuffle=True, generator=g)
        best_w, best_val = None, float('inf'); bad = 0
        for ep in range(1, epochs+1):
            model.train(); tot=0.0; batches=0
            for xb, rb, yb in tr_loader:
                xb, rb, yb = xb.to(device), rb.to(device), yb.to(device)
                opt.zero_grad()
                logits = model(xb, rb)
                loss = base_loss(logits, yb)
                loss.backward(); opt.step()
                tot += loss.item(); batches += 1
            model.eval()
            with torch.no_grad():
                v_logits = model(dva.X.to(device), dva.R.to(device))
                vloss = base_loss(v_logits, dva.y.to(device))
            print(f"[XS] Epoch {ep:03d} | train_loss={tot/max(batches,1):.4f} | val_loss={vloss:.4f}")
            if vloss < best_val:
                best_val, bad = float(vloss), 0
                best_w = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad > patience:
                    print(f"[XS] Early stopping at epoch {ep}")
                    break
        if best_w is not None:
            model.load_state_dict(best_w)
        return model

    model = _train_loop(model, dtr_xs, dva_xs, epochs, patience, use_focal_loss)

    # 9) Infer probs and per-date calibration
    def _infer_probs(model, ds):
        model.eval()
        with torch.no_grad():
            p = torch.softmax(model(ds.X.to(device), ds.R.to(device)), dim=1)[:,1].cpu().numpy()
        return p

    p_tr = _infer_probs(model, dtr_xs)
    p_va = _infer_probs(model, dva_xs)
    p_te = _infer_probs(model, dte_xs)

    cal_va = cs_quantile_calibrate(dva_xs.index, dva_xs.tickers, p_va, blend=0.5)
    cal_te = cs_quantile_calibrate(dte_xs.index, dte_xs.tickers, p_te, blend=0.5)

    keep_mask_te = np.ones_like(cal_te, dtype=bool)
    if top_n_per_date and top_n_per_date > 0:
        keep_mask_te = top_n_mask_by_date(dte_xs.index, cal_te, top_n_per_date)

    # --- Collect after-close messages here (per-ticker latest close) ---
    after_close_msgs = []

    # 10) Per-name evaluation, band selection, baseline comparison + reporting
    per_results = {}
    report_date = latest_us_trading_date()
    for t, d in per_ds.items():
        dva = d["val"]; dte = d["test"]

        mask_va = (np.array(dva_xs.tickers) == t)
        mask_te = (np.array(dte_xs.tickers) == t)
        pva = cal_va[mask_va]
        pte_t = cal_te[mask_te]
        keep_ticker = keep_mask_te[mask_te]

        df_full = per_full[t]     # FULL features (includes yesterday)
        df_labeled = per[t]       # labeled (ends earlier)

        # gates computed on FULL frame so they exist on yesterday too
        trend_ok_full = (df_full['Close'] > df_full['MA200']) & (df_full['MA200'].pct_change(20) > 0)

        # Quantile-based realized vol gate: fit on TRAIN, apply everywhere
        rv_full  = realized_vol(df_full['Close'], lookback=60)
        rv_train = rv_full.reindex(splits[t][0].index).dropna()  # TRAIN index for this ticker
        vol_thr_t = fit_vol_gate_threshold(rv_train, q_high=0.80)
        vol_ok_full = (rv_full <= vol_thr_t)

        trend_gate_va = trend_ok_full.reindex(dva.index).fillna(False)
        vol_gate_va   = vol_ok_full.reindex(dva.index).fillna(False)

        # Band search
        band_dir = os.path.join("artifacts", "bands")
        thr_entry_t, thr_exit_t = thr_entry, thr_exit  # defaults

        # Try to load frozen band first
        try:
            loaded_entry, loaded_exit, _meta = load_band(t, band_dir)
            thr_entry_t, thr_exit_t = loaded_entry, loaded_exit
            print(f"[{t}] Using FROZEN band from disk: entry={thr_entry_t:.3f} exit={thr_exit_t:.3f}")
            auto_band_from_val = False  # prevent retuning on TEST
        except Exception:
            pass

        if auto_band_from_val:
            if band_objective == "riskaware":
                picked = pick_band_on_val_riskaware(
                    p_va=pva, idx_va=dva.index, close_series=df_full['Open'],
                    df_feat_val=df_full, min_trades=min_trades_val,
                    enforce_entry_min=max(0.60, thr_entry),
                    gap=max(0.20, thr_entry - thr_exit),
                    lam=lam, gamma=gamma,
                    use_trend_gate=use_trend_gate, use_vol_gate=use_vol_gate,
                    trend_gate_s=trend_gate_va, vol_gate_s=vol_gate_va,
                    gate_relax_penalty=10.0,
                    init_cash=init_cash, fees=fees, slippage=slippage
                )
                if picked is not None:
                    score, thr_entry_t, thr_exit_t, trades, _ = picked
                    print(f"[{t}] picked band (riskaware): entry={thr_entry_t:.3f} exit={thr_exit_t:.3f} trades={int(trades)} score={score:.2f}")
            else:
                picked = pick_band_on_val(
                    p_va=pva, idx_va=dva.index, close_series=df_full['Open'],
                    y_va=dva.y.numpy(), min_trades=min_trades_val,
                    entry_grid=np.linspace(max(0.52, thr_entry-0.10), 0.90, 41),
                    gap=max(0.20, thr_entry - thr_exit),
                    init_cash=10_000, fees=fees, slippage=slippage,
                    objective=band_objective
                )
                if picked is not None:
                    score, thr_entry_t, thr_exit_t, trades, _ = picked
                    print(f"[{t}] picked band ({band_objective}): entry={thr_entry_t:.3f} exit={thr_exit_t:.3f} trades={int(trades)} score={score:.2f}")

        # Persist the chosen band (frozen from VAL) for paper/live
        try:
            save_band(t, {"entry": float(thr_entry_t), "exit": float(thr_exit_t)}, band_dir)
        except Exception as e:
            print(f"[{t}] Warning: couldn't save band -> {e}")

        # Build TEST state with gates (TEST dates live on labeled frame indices)
        s_test = state_from_band(pte_t, dte.index, thr_entry=thr_entry_t, thr_exit=thr_exit_t)

        override = pd.Series(False, index=s_test.index)
        if override_gate_prob is not None:
            p_series = pd.Series(pte_t, index=dte.index)
            override = p_series >= override_gate_prob

        trend_gate = trend_ok_full.reindex(s_test.index).fillna(False) if use_trend_gate else pd.Series(True, index=s_test.index)
        vol_gate   = vol_ok_full.reindex(s_test.index).fillna(False)   if use_vol_gate   else pd.Series(True, index=s_test.index)

        # Entry-only gating on TEST
        gated_state_test = apply_entry_only_gates(s_test, trend_gate, vol_gate, override)
        if top_n_per_date and top_n_per_date > 0:
            mask_series = pd.Series(keep_ticker, index=dte.index)
            gated_state_test &= mask_series
        gated_state_test = gated_state_test.rename('state')

        price_test = df_full['Open'].reindex(gated_state_test.index).astype(float)
        pf_m, st_m = backtest_state(price_test, gated_state_test,
                                    init_cash=init_cash, fees=fees, slippage=slippage,
                                    execution="next_open")

        # Baseline
        pf_b, st_b = None, None
        if evaluate_baseline:
            base_state = baseline_ma_atr_signals(df_full)
            base_state = base_state.reindex(gated_state_test.index).fillna(False)
            pf_b, st_b = backtest_state(price_test, base_state,
                                        init_cash=init_cash, fees=fees, slippage=slippage,
                                        execution="next_open")

        # Reporting
        _print_prob_stats(t, pte_t)
        print(f"[{t}] band used: entry={thr_entry_t:.3f} | exit={thr_exit_t:.3f}")
        print(f"[{t}] gates pass rates -> trend={trend_gate.mean():.3f} | vol={vol_gate.mean():.3f}")
        _print_signal_summary(t, gated_state_test)
        _print_bt_subset(t, st_m)
        if evaluate_baseline and st_b is not None:
            _print_bt_subset(f"{t} (baseline MA+ATR)", st_b)

        per_results[t] = {
            "band": (thr_entry_t, thr_exit_t),
            "model_pf": pf_m, "model_stats": st_m,
            "baseline_pf": pf_b, "baseline_stats": st_b
        }

        # --- AFTER-CLOSE: evaluate strictly at yesterday's US/Eastern trading day ---
        eval_date = report_date
        if eval_date not in df_full.index:
            try:
                eval_date = df_full.index[df_full.index.get_loc(report_date, method='pad')]
            except Exception:
                eval_date = df_full.index.max()

        live_close = float(df_full['Close'].loc[eval_date])

        if eval_date in dte.index:
            loc = dte.index.get_loc(eval_date)
        
            # --- LIVE recompute even inside TEST ---
            p_live = infer_live_probs_cs(
                df_full, features, scaler, model, regime_df, seq_len,
                pd.DatetimeIndex([eval_date])
            )
            prob_today = float(p_live.iloc[0])
            # --- END LIVE recompute ---
        
            raw_curr   = bool(s_test.iloc[loc])
            curr_final = bool(gated_state_test.iloc[loc])
            prev_final = bool(gated_state_test.shift(1, fill_value=False).iloc[loc])
            trend_ok_today = bool(trend_ok_full.reindex([eval_date]).fillna(False).iloc[0])
            vol_ok_today   = bool(vol_ok_full.reindex([eval_date]).fillna(False).iloc[0])
            override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
            topn_today     = True
            if top_n_per_date and top_n_per_date > 0:
                topn_today = bool(pd.Series(keep_ticker, index=dte.index).iloc[loc])

        else:
            # live inference for yesterday (beyond TEST end)
            p_tail = infer_live_probs_cs(df_full, features, scaler, model, regime_df, seq_len,
                                         pd.DatetimeIndex([eval_date]))
            prob_today = float(p_tail.iloc[0])
            raw_prev = bool(s_test.iloc[-1]) if len(s_test) else False
            raw_curr = bool(prob_today >= (thr_exit_t if raw_prev else thr_entry_t))
            trend_ok_today = bool(trend_ok_full.reindex([eval_date]).fillna(False).iloc[0])
            vol_ok_today   = bool(vol_ok_full.reindex([eval_date]).fillna(False).iloc[0])
            override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
            prev_final = bool(gated_state_test.iloc[-1]) if len(gated_state_test) else False
            curr_final = entry_only_gate(prev_final, raw_curr, trend_ok_today, vol_ok_today, override_used)
            topn_today = True  # can't compute cross-sec cap for tail

        msg = _format_after_close_message(
            ticker=t,
            live_date=eval_date,        # this is yesterday (or last <= yesterday)
            prev_final=prev_final,
            curr_final=curr_final,
            curr_raw=raw_curr,
            prob_today=prob_today,
            thr_entry=thr_entry_t,
            thr_exit=thr_exit_t,
            trend_ok_today=trend_ok_today,
            vol_ok_today=vol_ok_today,
            topn_kept_today=bool(topn_today),
            override_used=override_used,
            live_close=live_close
        )
        after_close_msgs.append(msg)

    # 11) Aggregate & print + consolidated latest-close sheet
    total_start, total_end, ok_count = 0.0, 0.0, 0
    beats = 0
    for t, res in per_results.items():
        st_m = res["model_stats"]
        sv = float(st_m.get("Start Value", np.nan))
        ev = float(st_m.get("End Value", np.nan))
        if not (np.isnan(sv) or np.isnan(ev)):
            total_start += sv; total_end += ev; ok_count += 1
        if evaluate_baseline and res["baseline_stats"] is not None:
            mret = float(st_m.get("Total Return [%]", 0.0))
            bret = float(res["baseline_stats"].get("Total Return [%]", 0.0))
            if mret > max(bret, 0.0):
                beats += 1
        print(f"[{t}] model_ret={st_m.get('Total Return [%]', np.nan):.2f}% | "
              f"vol={st_m.get('Volatility [%]', np.nan):.2f}% | "
              f"mdd={st_m.get('Max Drawdown [%]', np.nan):.2f}% | "
              f"baseline_ret={(res['baseline_stats'].get('Total Return [%]', np.nan) if res['baseline_stats'] is not None else np.nan):.2f}%")

    if ok_count > 0:
        pnl = total_end - total_start
        ret_pct = (total_end / total_start - 1.0) * 100.0 if total_start > 0 else float("nan")
        print("\n=== Cross-Sectional Aggregate summary ===")
        print(f"Tickers processed: {ok_count} | Beat baseline on {beats}")
        print(f"Total Start Value: ${total_start:,.2f}")
        print(f"Total End Value:   ${total_end:,.2f}")
        print(f"Total PnL:         ${pnl:,.2f}  ({ret_pct:,.2f}%)")

    if len(after_close_msgs) > 0:
        print(f"\n=== After-Close Signals — {report_date.date()} (US/Eastern) ===")
        for line in after_close_msgs:
            print(line)

    return {"model": model, "scaler": scaler, "per_results": per_results, "regime_df": regime_df}

# ---------- HOTFIX: redefine compute_features to fix a minor type() check typo ----------
# Final active version used later in script.
# This redefinition was kept to preserve working behaviour after debugging.
def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().ffill().bfill()
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].astype(float)

    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / loss.where(loss != 0, np.nan)
    df['RSI'] = 100 - (100 / (1 + rs))

    ema12 = df['Close'].ewm(span=12, adjust=False).mean()
    ema26 = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26
    df['SignalLine'] = df['MACD'].ewm(span=9, adjust=False).mean()

    ma20 = df['Close'].rolling(20).mean()
    std20 = df['Close'].rolling(20).std()
    df['BB_upper'] = ma20 + 2 * std20
    df['BB_lower'] = ma20 - 2 * std20

    df['LogReturn'] = np.log(df['Close'] / df['Close'].shift(1))
    df['r1'] = df['Close'].pct_change(1)
    df['r5'] = df['Close'].pct_change(5)
    df['r10'] = df['Close'].pct_change(10)

    atr = _atr(df, n=14)
    df['ATRp'] = atr / df['Close']

    rng = df['BB_upper'] - df['BB_lower']
    numerator = df['Close'] - df['BB_lower']
    df['pctB'] = numerator / rng.where(rng != 0, np.nan)

    df['MA50'] = df['Close'].rolling(50).mean()
    df['distMA50'] = df['Close'] / df['MA50'] - 1.0
    df['MA200'] = df['Close'].rolling(200).mean()

    for col in ['RSI', 'MACD', 'SignalLine', 'BB_upper', 'BB_lower', 'LogReturn',
                'r1', 'r5', 'r10', 'ATRp', 'pctB', 'MA50', 'distMA50', 'MA200']:
        if isinstance(df[col], pd.DataFrame):
            raise ValueError(f"Column {col} is a DataFrame with shape {df[col].shape}, expected Series")
        elif not isinstance(df[col], pd.Series):
            raise ValueError(f"Column {col} is not a Series, got {type(df[col])}")

    df.dropna(inplace=True)
    return df


# ---------- Main pipeline (single ticker) ----------
# Main single ticker workflow.
# runs the full pipeline for one asset, including feature generation, 
# chronological splitting, target creation, training, calibration,
# band selection, gating, backtesting and latest signal reporting.
def run_ticker_pipeline(ticker: str,
                        seq_len=30,
                        epochs=150,
                        patience=10,
                        purge_days=2,
                        init_cash=10_000,
                        fees=0.001,
                        slippage=0.001,
                        data_period="10y",
                        split_mode="by_date",
                        val_years=1,
                        test_years=2,
                        target_mode="horizon",
                        H=3, eps=0.003,
                        tb_H=5, tb_atr_mult=1.0,
                        use_band=True,
                        thr_entry=0.65,
                        thr_exit=0.40,
                        auto_band_from_val=False,
                        band_objective="winrate",
                        min_trades_val=5,
                        use_trend_gate=True,
                        use_vol_gate=True,
                        atrp_min=0.004, atrp_max=0.10,
                        use_focal_loss=True,
                        weight_by_future_abs=True,
                        device_str=None,
                        # --- NEW OPTIONS ---
                        risk_H_min=5, risk_H_max=10,
                        evaluate_baseline=False,
                        lam=0.3, gamma=0.2,
                        calibrate_platt=False,
                        override_gate_prob=None):
    print(f"\n=== {ticker} ===")

    purge_days = max(purge_days, H, risk_H_max)

    # 1) Download + features (normalized to US/Eastern calendar dates)
    df_raw = yf.download(
        ticker, period=data_period, interval="1d",
        auto_adjust=True, prepost=False, actions=False,
        threads=False, progress=False
    )
    print(f"Raw data shape: {df_raw.shape}, columns: {df_raw.columns}")
    print(f"Raw data date range: {df_raw.index.min()} to {df_raw.index.max()}")
    if isinstance(df_raw.columns, pd.MultiIndex):
        df_raw.columns = [col[0] for col in df_raw.columns]
    df_raw = df_raw[['Open', 'High', 'Low', 'Close', 'Volume']].ffill().bfill()
    df_raw = _normalize_daily_index_us(df_raw)  # normalize to US/Eastern dates
    if df_raw.empty or len(df_raw) < 300:
        raise ValueError(f"Not enough data for {ticker}")
    df_feat = compute_features(df_raw)

    # 2) Splits
    if split_mode == "by_date":
        train, val, test = time_splits_by_date(df_feat, test_years=test_years, val_years=val_years, purge_days=purge_days)
    else:
        train, val, test = time_splits(df_feat, val_frac=0.15, test_frac=0.15, purge_days=purge_days)

    # 3) Targets
    if target_mode == "horizon":
        for part in (train, val, test):
            add_targets_horizon_inplace(part, H=H, eps=eps)
    elif target_mode == "triple_barrier":
        for part in (train, val, test):
            add_targets_triple_barrier_inplace(part, H=tb_H, atr_mult=tb_atr_mult)
    elif target_mode == "riskadj_sign":
        for name, part in zip(['train','val','test'], (train, val, test)):
            df_tmp = add_targets_risk_adjusted_sign_inplace(part, H_min=risk_H_min, H_max=risk_H_max, eps=eps)
            if name == 'train':   train = df_tmp
            elif name == 'val':   val = df_tmp
            else:                 test = df_tmp
    elif target_mode == "excess_sign":
        spy_close, _vix = _download_market(period=data_period)
        for name, part in zip(['train','val','test'], (train, val, test)):
            df_tmp = add_targets_excess_sign_inplace(part, market_close=spy_close, H=max(H, risk_H_max), eps=eps)
            if name == 'train':   train = df_tmp
            elif name == 'val':   val = df_tmp
            else:                 test = df_tmp
    else:
        raise ValueError(f"Unknown target_mode: {target_mode}")

    # 4) Features
    features = [
        'LogReturn','RSI','MACD','SignalLine','BB_upper','BB_lower',
        'r1','r5','r10','ATRp','pctB','distMA50'
    ]

    # 5) Scale on TRAIN only
    scaler, train, val, test = scale_by_train(train, val, test, features)

    # 6) Datasets
    dtr = TimeSeriesDataset(train, features, seq_len)
    dva = TimeSeriesDataset(val, features, seq_len)
    dte = TimeSeriesDataset(test, features, seq_len)
    if len(dtr) == 0 or len(dva) == 0 or len(dte) == 0:
        raise ValueError("One of the splits is too short after seq_len; adjust years/fractions/seq_len.")

    Xtr, ytr = dtr.X, dtr.y
    Xva, yva = dva.X, dva.y
    Xte, yte = dte.X, dte.y

    # 6b) Sample weights
    wtr = None
    if weight_by_future_abs and 'FwdRet_H' in train.columns:
        wtr = torch.tensor(
            np.clip(train.loc[dtr.index, 'FwdRet_H'].abs().values, 1e-6, None),
            dtype=torch.float32
        )
        wtr = wtr / (wtr.mean() + 1e-9)

    # 7) Train
    device = torch.device(device_str) if device_str else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TransformerClassifier(in_features=len(features), seq_len=seq_len).to(device)
    model = train_with_early_stopping(
        model, Xtr, ytr, Xva, yva,
        epochs=epochs, patience=patience,
        weights_tr=wtr, use_focal=use_focal_loss
    )

    # 8) Probs for each split
    with torch.no_grad():
        p_tr = torch.softmax(model(dtr.X.to(device)), dim=1)[:, 1].cpu().numpy()
        p_va = torch.softmax(model(dva.X.to(device)), dim=1)[:, 1].cpu().numpy()
        p_te_raw = torch.softmax(model(dte.X.to(device)), dim=1)[:, 1].cpu().numpy()
        print(f"Test set prob_up range (raw): min={p_te_raw.min():.3f}, max={p_te_raw.max():.3f}, mean={p_te_raw.mean():.3f}")

    # C1) Optional Platt calibration (keep calibrator for live)
    p_te = p_te_raw.copy()
    platt_cal = None
    if calibrate_platt:
        p_te, platt_cal = platt_fit_apply(p_va, yva.numpy(), p_te_raw)
        print("Applied Platt calibration to TEST probabilities (fit on VAL).")

    # 9) Metrics & decision rule
    if use_band:
        # C2) Use frozen band if present, save if newly picked
        band_dir = os.path.join("artifacts", "bands")
        try:
            thr_entry, thr_exit, _meta = load_band(ticker, band_dir)
            print(f"[{ticker}] Using FROZEN band from disk: entry={thr_entry:.3f} exit={thr_exit:.3f}")
            auto_band_from_val = False
        except Exception:
            pass

        if auto_band_from_val:
            trend_ok_val = (df_feat['Close'] > df_feat['MA200']) & (df_feat['MA200'].pct_change(20) > 0)
            # keep ATRp-based gate here just for VAL band-pick (as before)
            vol_ok_val   = df_feat['ATRp'].between(atrp_min, atrp_max)
            if band_objective == "riskaware":
                picked = pick_band_on_val_riskaware(
                    p_va, dva.index, df_feat['Open'],
                    df_feat_val=df_feat,
                    min_trades=max(min_trades_val, 20),
                    enforce_entry_min=thr_entry,
                    gap=max(0.12, thr_entry - thr_exit),
                    lam=lam, gamma=gamma,
                    use_trend_gate=use_trend_gate, use_vol_gate=use_vol_gate,
                    trend_gate_s=trend_ok_val.reindex(dva.index),
                    vol_gate_s=vol_ok_val.reindex(dva.index),
                    gate_relax_penalty=10.0,
                    init_cash=init_cash, fees=fees, slippage=slippage
                )
            else:
                picked = pick_band_on_val(
                    p_va, dva.index, df_feat['Open'],
                    y_va=yva.numpy(), min_trades=min_trades_val,
                    entry_grid=np.linspace(max(0.52, thr_entry-0.10), 0.90, 41),
                    gap=max(0.20, thr_entry - thr_exit),
                    init_cash=init_cash, fees=fees, slippage=slippage,
                    objective=band_objective
                )
            if picked is not None:
                score, thr_entry, thr_exit, trades, _stats = picked
                print(f"[Auto-band:{band_objective}] score={score:.2f} | thr_entry={thr_entry:.3f} thr_exit={thr_exit:.3f} | trades={int(trades)}")
            else:
                print("[Auto-band] No band met min trade count on validation; using provided band.")

        # Persist the chosen band
        try:
            save_band(ticker, {"entry": float(thr_entry), "exit": float(thr_exit)}, band_dir)
        except Exception as e:
            print(f"[{ticker}] Warning: couldn't save band -> {e}")

        print(f"Using band: entry >= {thr_entry:.2f}, exit < {thr_exit:.2f}")
        m_tr = eval_classification_band(p_tr, ytr, dtr.index, thr_entry, thr_exit)
        m_va = eval_classification_band(p_va, yva, dva.index, thr_entry, thr_exit)
        m_te = eval_classification_band(p_te, yte, dte.index, thr_entry, thr_exit)
        raw_state_test = m_te["state"]
        state_test = raw_state_test.copy()
        thr = None
    else:
        prec, thr, cnt = pick_threshold_for_precision(p_va, yva.numpy(), min_signals=12)
        if prec == 0.0:
            thr = pick_threshold_from_val(model, Xva, yva)
        thr = max(thr, 0.60)
        print(f"Chosen threshold for precision: {thr:.3f}")
        m_tr = eval_classification(model, Xtr, ytr, thr)
        m_va = eval_classification(model, Xva, yva, thr)
        m_te = eval_classification(model, Xte, yte, thr)
        raw_state_test = pd.Series(m_te['probs'] >= thr, index=dte.index, name='state')
        state_test = raw_state_test.copy()

    print(f"Train  acc={m_tr['acc']:.3f} prec={m_tr['prec']:.3f} rec={m_tr['rec']:.3f} f1={m_tr['f1']:.3f}")
    print(f"Val    acc={m_va['acc']:.3f} prec={m_va['prec']:.3f} rec={m_va['rec']:.3f} f1={m_va['f1']:.3f}")
    print(f"Test   acc={m_te['acc']:.3f} prec={m_te['prec']:.3f} rec={m_te['rec']:.3f} f1={m_te['f1']:.3f}")

    # 10) Regime & volatility gating on TEST + optional override
    trend_ok = (df_feat['Close'] > df_feat['MA200']) & (df_feat['MA200'].pct_change(20) > 0)
    trend_gate = trend_ok.reindex(state_test.index).fillna(False) if use_trend_gate else pd.Series(True, index=state_test.index)

    # C3) Quantile-based realized vol threshold fitted on TRAIN
    rv_full  = realized_vol(df_feat['Close'], lookback=60)
    rv_train = rv_full.reindex(train.index).dropna()
    vol_thr  = fit_vol_gate_threshold(rv_train, q_high=0.80)
    vol_ok   = (rv_full <= vol_thr)
    vol_gate = vol_ok.reindex(state_test.index).fillna(False) if use_vol_gate else pd.Series(True, index=state_test.index)

    override = pd.Series(False, index=state_test.index)
    if override_gate_prob is not None:
        active_probs = m_te['probs']
        override = pd.Series(active_probs, index=state_test.index) >= float(override_gate_prob)

    # C4) Entry-only gating on TEST
    gated_state_test = apply_entry_only_gates(state_test, trend_gate, vol_gate, override).rename('state')
    print(f"Trend gate pass rate: {trend_gate.mean():.3f}")
    print(f"Vol gate  pass rate: {vol_gate.mean():.3f}")
    _print_signal_summary(ticker, gated_state_test)

    # 11) Backtest on TEST (next-bar execution at next OPEN)
    price_test = df_feat['Open'].reindex(gated_state_test.index).astype(float)
    pf, stats = backtest_state(
        price_test, gated_state_test,
        init_cash=init_cash, fees=fees, slippage=slippage,
        execution="next_open"
    )
    print("\n--- Backtest (TEST split, next-bar OPEN, after costs; gated) ---")
    print(stats)
    _print_bt_subset(ticker, stats)

    # Baseline comparison (optional)
    if evaluate_baseline:
        base_state = baseline_ma_atr_signals(df_feat).reindex(gated_state_test.index).fillna(False)
        pf_b, st_b = backtest_state(
            price_test, base_state,
            init_cash=init_cash, fees=fees, slippage=slippage,
            execution="next_open"
        )
        print("\n--- Baseline (MA+ATR) backtest ---")
        print(st_b)
        try:
            mret = float(stats.get('Total Return [%]', 0.0))
            bret = float(st_b.get('Total Return [%]', 0.0))
            print(f"Model beats baseline? {mret:.2f}% vs {bret:.2f}% -> {mret > max(bret, 0.0)}")
        except Exception:
            pass

    # 12) After-close single-line instruction — ALWAYS yesterday's US/Eastern close (or nearest <=)
    msg = None

    report_date = latest_us_trading_date()
    if report_date in df_feat.index:
        eval_date = report_date
    else:
        try:
            eval_date = df_feat.index[df_feat.index.get_loc(report_date, method='pad')]
        except Exception:
            eval_date = df_feat.index.max()

    live_close = float(df_feat['Close'].loc[eval_date])

    if use_band:
        if eval_date in dte.index:
            # Recompute prob live for that exact date (still inside TEST)
            loc = dte.index.get_loc(eval_date)
            p_live = infer_live_probs_basic(
                df_full_feats=df_feat, features=features, scaler=scaler, model=model,
                seq_len=seq_len, dates=pd.DatetimeIndex([eval_date])
            )
            prob_today = float(p_live.iloc[0])

            # C5) Apply Platt calibration to live prob if enabled
            if calibrate_platt and (platt_cal is not None):
                prob_today = float(platt_cal.transform(np.array([prob_today]))[0])

            # Keep states from precomputed TEST artifacts
            raw_curr      = bool(raw_state_test.iloc[loc])
            curr_final    = bool(gated_state_test.iloc[loc])
            prev_final    = bool(gated_state_test.shift(1, fill_value=False).iloc[loc])
            # gate checks at eval_date
            trend_ok_today = bool(trend_ok.reindex([eval_date]).fillna(False).iloc[0]) if use_trend_gate else True
            vol_ok_today   = bool(vol_ok.reindex([eval_date]).fillna(False).iloc[0])   if use_vol_gate   else True
            override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
        else:
            # Tail beyond TEST: infer prob, roll band by one, then entry-only gate
            p_tail = infer_live_probs_basic(
                df_full_feats=df_feat, features=features, scaler=scaler, model=model,
                seq_len=seq_len, dates=pd.DatetimeIndex([eval_date])
            )
            prob_today = float(p_tail.iloc[0])

            # C5) Apply Platt to tail prob if enabled
            if calibrate_platt and (platt_cal is not None):
                prob_today = float(platt_cal.transform(np.array([prob_today]))[0])

            prev_raw       = bool(raw_state_test.iloc[-1]) if len(raw_state_test) else False
            raw_curr       = bool(prob_today >= (thr_exit if prev_raw else thr_entry))
            prev_final     = bool(gated_state_test.iloc[-1]) if len(gated_state_test) else False
            trend_ok_today = bool(trend_ok.reindex([eval_date]).fillna(False).iloc[0]) if use_trend_gate else True
            vol_ok_today   = bool(vol_ok.reindex([eval_date]).fillna(False).iloc[0])   if use_vol_gate   else True
            override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
            # C6) Entry-only decision for tail
            curr_final     = entry_only_gate(prev_final, raw_curr, trend_ok_today, vol_ok_today, override_used)
    else:
        # Threshold mode
        if eval_date in dte.index:
            p_live = infer_live_probs_basic(
                df_full_feats=df_feat, features=features, scaler=scaler, model=model,
                seq_len=seq_len, dates=pd.DatetimeIndex([eval_date])
            )
            prob_today = float(p_live.iloc[0])
            if calibrate_platt and (platt_cal is not None):
                prob_today = float(platt_cal.transform(np.array([prob_today]))[0])
        else:
            p_tail = infer_live_probs_basic(
                df_full_feats=df_feat, features=features, scaler=scaler, model=model,
                seq_len=seq_len, dates=pd.DatetimeIndex([eval_date])
            )
            prob_today = float(p_tail.iloc[0])
            if calibrate_platt and (platt_cal is not None):
                prob_today = float(platt_cal.transform(np.array([prob_today]))[0])

        raw_curr       = bool(prob_today >= (thr if thr is not None else 0.5))
        prev_final     = bool(gated_state_test.iloc[-1]) if len(gated_state_test) else False
        trend_ok_today = bool(trend_ok.reindex([eval_date]).fillna(False).iloc[0]) if use_trend_gate else True
        vol_ok_today   = bool(vol_ok.reindex([eval_date]).fillna(False).iloc[0])   if use_vol_gate   else True
        override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
        curr_final     = bool(raw_curr and ((trend_ok_today and vol_ok_today) or override_used))

    msg = _format_after_close_message(
        ticker=ticker,
        live_date=eval_date,
        prev_final=prev_final,
        curr_final=curr_final,
        curr_raw=raw_curr,
        prob_today=float(prob_today),
        thr_entry=float(thr_entry),
        thr_exit=float(thr_exit),
        trend_ok_today=bool(trend_ok_today),
        vol_ok_today=bool(vol_ok_today),
        topn_kept_today=True,
        override_used=bool(override_used),
        live_close=live_close
    )

    print("\n=== After-Close Signal (yesterday close) ===")
    print(msg)
    try:
        if msg.startswith("Enter Next Opening"):
            notify("Transformer Signal: ENTRY", msg)
        elif msg.startswith("Exit Next Opening"):
            notify("Transformer Signal: EXIT", msg)
    except Exception:
        pass


    # 13) "Today" inference (convenience)
    live_date_inf, live_prob = infer_today_prob(df_feat, features, scaler, model, seq_len=seq_len)
    if live_date_inf is not None:
        # C7) Apply Platt to today's prob
        if calibrate_platt and (platt_cal is not None) and (live_prob is not None):
            live_prob = float(platt_cal.transform(np.array([live_prob]))[0])

        next_ix = df_feat.index[df_feat.index > live_date_inf]
        next_date = next_ix[0] if len(next_ix) else None

        prev_idx = gated_state_test.index[gated_state_test.index < live_date_inf]
        st_yday = bool(gated_state_test.loc[prev_idx[-1]]) if len(prev_idx) else False

        if use_band:
            st_today_raw = (live_prob >= (thr_exit if st_yday else thr_entry))
        else:
            st_today_raw = (live_prob >= (thr if thr is not None else 0.5))

        trend_ok_today = bool(trend_ok.reindex([live_date_inf]).fillna(False).iloc[0]) if use_trend_gate else True
        vol_ok_today   = bool(vol_ok.reindex([live_date_inf]).fillna(False).iloc[0])   if use_vol_gate   else True
        override_used  = (override_gate_prob is not None) and (live_prob >= float(override_gate_prob))

        st_today = entry_only_gate(prev_final=st_yday, raw_curr=st_today_raw,
                                   trend_ok_today=trend_ok_today, vol_ok_today=vol_ok_today,
                                   override_used=override_used)

        entry_today = (not st_yday) and st_today
        exit_today  = st_yday and (not st_today)
        close_today = float(df_feat['Close'].reindex([live_date_inf]).iloc[0])

        when = f"next session ({next_date.date()})" if next_date is not None else "next session (no further bar available)"
        if entry_today:
            msg2 = f"{ticker}: ENTRY signal for {when} | prob_up={live_prob:.2%} | today_close=${close_today:,.2f}"
            notify("Transformer Signal: ENTRY", msg2); print(msg2)
        elif exit_today:
            msg2 = f"{ticker}: EXIT signal for {when} | prob_up={live_prob:.2%} | today_close=${close_today:,.2f}"
            notify("Transformer Signal: EXIT", msg2); print(msg2)
        else:
            state = "LONG" if st_today else "FLAT"
            print(f"{ticker}: No new signal ({when}). State {state}. prob_up={live_prob:.2%} | today_close=${close_today:,.2f}")

    return {
        "model": model,
        "scaler": scaler,
        "threshold": thr if not use_band else None,
        "band": (thr_entry, thr_exit) if use_band else None,
        "datasets": {"train": dtr, "val": dva, "test": dte},
        "metrics": {"train": m_tr, "val": m_va, "test": m_te},
        "backtest": {"portfolio": pf, "stats": stats},
        "features_df": df_feat
    }


# ---------- Run ----------
# Entry point.
# runs either the cross sectional universe or the single ticker loop, 
# depending on the selected execution mode below.
if __name__ == "__main__":

    total_start = 0.0
    total_end = 0.0
    total_beats = 0
    ok_count = 0

    tickers = [
        "MSFT", "NVDA", "META",
        "LRCX", "ICE",
        "ORLY", "GS", "BLK", "PGR", "LLY",
        "MRK", "ISRG", "HCA", "ETN",
        "AAPL", "AMZN", "GOOGL", "CRM",
        "CME", "TSLA", "COST", "V", "MCK",
        "CAT", "ROK", "CVX",
    ]


    USE_CROSS_SECTIONAL = True  # set True to run the cross-sectional pipeline with calibration

    if USE_CROSS_SECTIONAL:
        # Cross-sectional training + calibrated bands + baseline comparison + richer per-ticker reports
        run_universe_pipeline(
            tickers=tickers,
            seq_len=30,
            epochs=120,
            patience=8,
            data_period="max",
            split_mode="by_date",
            val_years=3, test_years=5,
            target_mode="riskadj_sign",  # or "excess_sign"
            H=10, eps=0.0,
            H_min=5, H_max=10,
            use_band=True,
            thr_entry=0.60,
            thr_exit=0.45,
            auto_band_from_val=True,
            band_objective="precision",   # also supports: "riskaware", "winrate", "return"
            min_trades_val=10,
            lam=0.3, gamma=0.2,
            use_trend_gate=True,
            use_vol_gate=True,
            atrp_min=0.004, atrp_max=0.10,
            evaluate_baseline=True,
            override_gate_prob=None,  # e.g., 0.92 to bypass gates on very high probs
            top_n_per_date=0,         # set >0 to cap positions per day across names
            init_cash=10_000, fees=0.001, slippage=0.001
        )
        print("\n[MAIN] Cross-sectional run complete.")
    else:
        # Single-name pipeline — prints the detailed report and an after-close single-line signal
        for t in tickers:
            try:
                res = run_ticker_pipeline(
                    t,
                    seq_len=30,
                    epochs=120,
                    patience=8,
                    purge_days=3,
                    init_cash=10_000,
                    fees=0.001,
                    slippage=0.001,
                    data_period="10y",
                    split_mode="by_date",
                    val_years=1,
                    test_years=2,
                    target_mode="riskadj_sign",   # try: "riskadj_sign" / "excess_sign" / "horizon"
                    H=3, eps=0.0,
                    use_band=True,
                    thr_entry=0.55,               # auto-band may raise this
                    thr_exit=0.42,
                    auto_band_from_val=True,      # enable to use band search
                    band_objective="precision",   # "riskaware" / "winrate" / "precision" / "return"
                    min_trades_val=10,
                    use_trend_gate=True,
                    use_vol_gate=True,
                    atrp_min=0.004,
                    atrp_max=0.10,
                    use_focal_loss=False,
                    weight_by_future_abs=True,
                    evaluate_baseline=True,
                    lam=0.3, gamma=0.2,
                    calibrate_platt=True,
                    override_gate_prob=None
                )

                # Aggregate Start/End values
                stats = res["backtest"]["stats"]
                sv = float(stats.get("Start Value", np.nan))
                ev = float(stats.get("End Value", np.nan))
                if not (np.isnan(sv) or np.isnan(ev)):
                    total_start += sv
                    total_end += ev
                    ok_count += 1

            except Exception as e:
                print(f"{t}: error -> {e}")

        # Final aggregate summary
        if ok_count > 0:
            pnl = total_end - total_start
            ret_pct = (total_end / total_start - 1.0) * 100.0 if total_start > 0 else float("nan")
            print("\n=== Aggregate summary ===")
            print(f"Tickers processed: {ok_count}")
            print(f"Total Start Value: ${total_start:,.2f}")
            print(f"Total End Value:   ${total_end:,.2f}")
            print(f"Total PnL:         ${pnl:,.2f}  ({ret_pct:,.2f}%)")
        else:
            print("\nNo successful backtests to summarize.")

    # Final confirmation print so the script always ends with output
    print("\n=== Script finished ===")


/Users/jurchiks/Desktop/Final code/Work In Progeress/ZOE ML/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(



=== Cross-Sectional Pipeline ===
Train range: 1987-01-22 00:00:00 to 2018-03-12 00:00:00
Val range: 2018-03-27 00:00:00 to 2021-03-12 00:00:00
Test range: 2021-03-29 00:00:00 to 2026-03-13 00:00:00
Train range: 1999-12-02 00:00:00 to 2018-03-12 00:00:00
Val range: 2018-03-27 00:00:00 to 2021-03-12 00:00:00
Test range: 2021-03-29 00:00:00 to 2026-03-13 00:00:00
Train range: 2013-04-04 00:00:00 to 2018-03-12 00:00:00
Val range: 2018-03-27 00:00:00 to 2021-03-12 00:00:00
Test range: 2021-03-29 00:00:00 to 2026-03-13 00:00:00
Train range: 1985-03-15 00:00:00 to 2018-03-12 00:00:00
Val range: 2018-03-27 00:00:00 to 2021-03-12 00:00:00
Test range: 2021-03-29 00:00:00 to 2026-03-13 00:00:00
Train range: 2006-09-29 00:00:00 to 2018-03-12 00:00:00
Val range: 2018-03-27 00:00:00 to 2021-03-12 00:00:00
Test range: 2021-03-29 00:00:00 to 2026-03-13 00:00:00
Train range: 1994-03-03 00:00:00 to 2018-03-12 00:00:00
Val range: 2018-03-27 00:00:00 to 2021-03-12 00:00:00
Test range: 2021-03-29 00:00:00

# BELOW IS A BACKUP COPY

In [ ]:

# # --- Imports & setup ---
# import warnings, sys, os, re, random, platform, math, pickle
# import numpy as np, pandas as pd, torch
# import json


# warnings.filterwarnings("ignore", message="Metric '.*' requires frequency to be set")
# warnings.filterwarnings("ignore", message="Object has multiple columns.*")

# import yfinance as yf
# import vectorbt as vbt

# from sklearn.preprocessing import RobustScaler
# from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
# from sklearn.linear_model import LogisticRegression

# # Ensure sklearn keeps feature names / returns DataFrames where possible (silences warnings)
# try:
#     from sklearn import set_config
#     set_config(transform_output="pandas")
# except Exception:
#     pass

# from torch import nn, optim
# from torch.utils.data import Dataset, DataLoader, TensorDataset
# from pandas.tseries.offsets import BDay  # for next-business-day timestamps

# SEED = 42
# os.environ["PYTHONHASHSEED"] = str(SEED)
# random.seed(SEED); np.random.seed(SEED)
# torch.manual_seed(SEED)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(SEED)
# try:
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False
#     torch.use_deterministic_algorithms(True)
# except Exception:
#     pass

# # ---------- Notification helper ----------
# def notify(title: str, message: str):
#     sent = False
#     try:
#         from player import notification
#         notification.notify(title=title, message=message, app_name="TF Signal", timeout=8)
#         sent = True
#     except Exception:
#         pass
#     if not sent:
#         try:
#             if platform.system() == "Windows":
#                 import winsound; winsound.MessageBeep()
#         except Exception:
#             pass
#         print(f"[NOTIFY] {title}: {message}")

# def _safe_ticker_for_path(t: str) -> str:
#     return re.sub(r"[^A-Za-z0-9_.-]+", "_", t)

# # ---------- Yahoo daily normalization ----------
# def _normalize_daily_index_us(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Normalize yfinance daily index to US/Eastern calendar days (naive midnight) and drop dups.
#     Prevents mixed T vs T-1 dates caused by timezone offsets.
#     """
#     df = df.copy()
#     idx = pd.DatetimeIndex(df.index)
#     if idx.tz is not None:
#         idx = idx.tz_convert('America/New_York').tz_localize(None)
#     idx = pd.to_datetime(idx.date)   # force to calendar date (midnight)
#     df.index = idx
#     df = df[~df.index.duplicated(keep='last')]  # keep the latest row if duplicates
#     return df

# def latest_us_trading_date(buffer_min: int = 5) -> pd.Timestamp:
#     """
#     Return yesterday's US trading date (US/Eastern) unless today's close + buffer has passed.
#     """
#     now_et = pd.Timestamp.now(tz='America/New_York')
#     d = now_et.normalize()
#     # roll back weekends
#     while d.weekday() >= 5:
#         d = d - BDay(1)
#     close_ts = d + pd.Timedelta(hours=16)
#     if now_et < close_ts + pd.Timedelta(minutes=buffer_min):
#         d = d - BDay(1)
#     return d.tz_localize(None)

# # ---------- Technical helpers ----------
# def _atr(df: pd.DataFrame, n=14):
#     high, low, close = df['High'], df['Low'], df['Close']
#     prev_close = close.shift(1)
#     tr1 = (high - low).abs()
#     tr2 = (high - prev_close).abs()
#     tr3 = (low - prev_close).abs()
#     tr = np.maximum(tr1, np.maximum(tr2, tr3))
#     atr = tr.rolling(n).mean()
#     return atr

# # ---------- Features (precision-oriented & lean) ----------
# def compute_features(df: pd.DataFrame) -> pd.DataFrame:
#     df = df.copy().ffill().bfill()
#     df = df[['Open', 'High', 'Low', 'Close', 'Volume']].astype(float)

#     delta = df['Close'].diff()
#     gain = delta.where(delta > 0, 0).rolling(14).mean()
#     loss = -delta.where(delta < 0, 0).rolling(14).mean()
#     rs = gain / loss.where(loss != 0, np.nan)
#     df['RSI'] = 100 - (100 / (1 + rs))

#     ema12 = df['Close'].ewm(span=12, adjust=False).mean()
#     ema26 = df['Close'].ewm(span=26, adjust=False).mean()
#     df['MACD'] = ema12 - ema26
#     df['SignalLine'] = df['MACD'].ewm(span=9, adjust=False).mean()

#     ma20 = df['Close'].rolling(20).mean()
#     std20 = df['Close'].rolling(20).std()
#     df['BB_upper'] = ma20 + 2 * std20
#     df['BB_lower'] = ma20 - 2 * std20

#     df['LogReturn'] = np.log(df['Close'] / df['Close'].shift(1))
#     df['r1'] = df['Close'].pct_change(1)
#     df['r5'] = df['Close'].pct_change(5)
#     df['r10'] = df['Close'].pct_change(10)

#     atr = _atr(df, n=14)
#     df['ATRp'] = atr / df['Close']

#     rng = df['BB_upper'] - df['BB_lower']
#     numerator = df['Close'] - df['BB_lower']
#     df['pctB'] = numerator / rng.where(rng != 0, np.nan)

#     df['MA50'] = df['Close'].rolling(50).mean()
#     df['distMA50'] = df['Close'] / df['MA50'] - 1.0
#     df['MA200'] = df['Close'].rolling(200).mean()

#     for col in ['RSI', 'MACD', 'SignalLine', 'BB_upper', 'BB_lower', 'LogReturn',
#                 'r1', 'r5', 'r10', 'ATRp', 'pctB', 'MA50', 'distMA50', 'MA200']:
#         if isinstance(df[col], pd.DataFrame):
#             raise ValueError(f"Column {col} is a DataFrame with shape {df[col].shape}, expected Series")
#         elif not isinstance(df[col], pd.Series):
#             raise ValueError(f"Column {col} is not a Series, got {type[df[col]]}")

#     df.dropna(inplace=True)
#     return df

# # ---------- Splits ----------
# def time_splits(df: pd.DataFrame, val_frac=0.15, test_frac=0.15, purge_days=2):
#     n = len(df)
#     n_test = int(round(n * test_frac))
#     n_val = int(round(n * val_frac))
#     n_train = max(0, n - n_val - n_test)
#     if n_train < 60:
#         raise ValueError("Not enough data after splits; reduce seq_len or fractions.")
#     train = df.iloc[:n_train].copy()
#     val = df.iloc[n_train:n_train+n_val].copy()
#     test = df.iloc[n_train+n_val:].copy()
#     if purge_days > 0:
#         if len(val) > purge_days: val = val.iloc[purge_days:]
#         if len(test) > purge_days: test = test.iloc[purge_days:]
#     return train, val, test

# def time_splits_by_date(df: pd.DataFrame, test_years=2, val_years=1, purge_days=2):
#     last = df.index.max()
#     test_start = last - pd.DateOffset(years=test_years)
#     val_start = test_start - pd.DateOffset(years=val_years)

#     train = df.loc[:val_start - pd.Timedelta(days=1)].copy()
#     val = df.loc[val_start:test_start - pd.Timedelta(days=1)].copy()
#     test = df.loc[test_start:].copy()

#     if purge_days > 0:
#         if len(val) > purge_days: val = val.iloc[purge_days:]
#         if len(test) > purge_days: test = test.iloc[purge_days:]

#     # Ensure test includes the latest date in df
#     if test.index.max() < df.index.max():
#         test = df.loc[test_start:df.index.max()].copy()
#         if purge_days > 0 and len(test) > purge_days:
#             test = test.iloc[purge_days:]

#     if min(map(len, [train, val, test])) < 60:
#         raise ValueError("One of the splits is too short; reduce seq_len or adjust years.")
    
#     print(f"Train range: {train.index.min()} to {train.index.max()}")
#     print(f"Val range: {val.index.min()} to {val.index.max()}")
#     print(f"Test range: {test.index.min()} to {test.index.max()}")
#     return train, val, test

# # ---------- Targets ----------
# def add_targets_horizon_inplace(df: pd.DataFrame, H=3, eps=0.003):
#     df['FwdRet_H'] = np.log(df['Close'].shift(-H) / df['Close'])
#     y = np.where(df['FwdRet_H'] > eps, 1,
#                  np.where(df['FwdRet_H'] < -eps, 0, np.nan))
#     df['Target'] = y
#     df.dropna(subset=['Target'], inplace=True)
#     df['Target'] = df['Target'].astype(int)

# def add_targets_triple_barrier_inplace(df: pd.DataFrame, H=5, atr_mult=1.0):
#     """
#     Robust daily triple-barrier labeling.
#     - Upper/lower barriers: Close ± atr_mult * ATR(14)
#     - Horizon H (in bars) ahead.
#     - If both barriers are touched on the same bar, label is set to NaN (tie / unknown order).
#     """
#     df = df.copy()
#     atr = _atr(df, n=14)
#     up = df['Close'] + atr_mult * atr
#     dn = df['Close'] - atr_mult * atr

#     highs = df['High'].values
#     lows  = df['Low'].values
#     up_v  = up.values
#     dn_v  = dn.values

#     n = len(df)
#     tgt = np.full(n, np.nan, dtype=float)

#     for i in range(n - H):
#         hi_seq = highs[i+1:i+H+1]
#         lo_seq = lows[i+1:i+H+1]
#         up_hit_idx = np.where(hi_seq >= up_v[i])[0]
#         dn_hit_idx = np.where(lo_seq <= dn_v[i])[0]

#         if up_hit_idx.size == 0 and dn_hit_idx.size == 0:
#             continue
#         if up_hit_idx.size > 0 and (dn_hit_idx.size == 0 or up_hit_idx[0] < dn_hit_idx[0]):
#             tgt[i] = 1.0
#         elif dn_hit_idx.size > 0 and (up_hit_idx.size == 0 or dn_hit_idx[0] < up_hit_idx[0]):
#             tgt[i] = 0.0
#         else:
#             # same-bar hit: ambiguous on daily data -> skip (remain NaN)
#             pass

#     df['Target'] = tgt
#     df.dropna(subset=['Target'], inplace=True)
#     df['Target'] = df['Target'].astype(int)
#     df['FwdRet_H'] = np.log(df['Close'].shift(-H) / df['Close'])
#     # write columns back into the original frame
#     for col in ['Target', 'FwdRet_H']:
#         df_col = df[col]
#         df_col = df_col.reindex(df.index)
#         try:
#             pass
#         except Exception:
#             pass

# # ---------- NEW TARGETS (extras) ----------
# def _forward_log_return(df: pd.DataFrame, H: int) -> pd.Series:
#     return np.log(df['Close'].shift(-H) / df['Close'])

# def add_targets_risk_adjusted_sign_inplace(df: pd.DataFrame,
#                                            H_min=5, H_max=10,
#                                            risk_window=20,
#                                            eps=0.0):
#     """
#     Label by the sign of average forward Sharpe-like return over horizons H_min..H_max.
#     FwdRiskAdj = mean_H [ log(C_{t+H}/C_t) / (roll_std(LogReturn, risk_window) * sqrt(H)) ]

#     Returns a new DataFrame with ['FwdRiskAdj','Target'] added.
#     """
#     df = df.copy()
#     r = df['LogReturn'] if 'LogReturn' in df.columns else np.log(df['Close']/df['Close'].shift(1))
#     vol = r.rolling(risk_window).std()

#     sharps = []
#     for H in range(H_min, H_max + 1):
#         f = _forward_log_return(df, H)
#         sharps.append(f / (vol * np.sqrt(H)))
#     F = pd.concat(sharps, axis=1).mean(axis=1)
#     df['FwdRiskAdj'] = F
#     y = np.where(F > eps, 1, np.where(F < -eps, 0, np.nan))
#     df['Target'] = y
#     df.dropna(subset=['Target'], inplace=True)
#     df['Target'] = df['Target'].astype(int)
#     for col in ['FwdRiskAdj', 'Target']:
#         df[col] = df[col].astype(float) if col == 'FwdRiskAdj' else df[col]
#     return df

# def add_targets_excess_sign_inplace(df: pd.DataFrame, market_close: pd.Series,
#                                     H=10, eps=0.0):
#     """
#     Label by sign of forward EXCESS return vs market (e.g., SPY) over H days.
#     Returns a new DataFrame (copy) with ['FwdExcessRet','Target'].
#     """
#     df = df.copy()
#     mrk = market_close.reindex(df.index).ffill()
#     f_stock = _forward_log_return(df, H)
#     f_mkt = np.log(mrk.shift(-H) / mrk)
#     excess = f_stock - f_mkt
#     df['FwdExcessRet'] = excess
#     y = np.where(excess > eps, 1, np.where(excess < -eps, 0, np.nan))
#     df['Target'] = y
#     df.dropna(subset=['Target'], inplace=True)
#     df['Target'] = df['Target'].astype(int)
#     return df

# # ---------- Scaling ----------
# def scale_by_train(train, val, test, features):
#     scaler = RobustScaler()
#     try:
#         scaler.set_output(transform="pandas")
#     except Exception:
#         pass
#     train[features] = scaler.fit_transform(train[features])
#     val[features] = scaler.transform(val[features])
#     test[features] = scaler.transform(test[features])
#     return scaler, train, val, test

# # ---------- Dataset ----------
# class TimeSeriesDataset(Dataset):
#     def __init__(self, df, features, seq_len=30):
#         self.seq_len = seq_len
#         self.features = features
#         X_list, y_list, idx = [], [], []
#         if len(df) <= seq_len:
#             self.X = torch.empty(0, seq_len, len(features))
#             self.y = torch.empty(0, dtype=torch.long)
#             self.index = pd.DatetimeIndex([])
#             return
#         for i in range(len(df) - seq_len):
#             end = i + seq_len - 1            # last feature row (decision time t)
#             X_list.append(df[features].iloc[i:i+seq_len].values)
#             y_list.append(int(df['Target'].iloc[end]))   # label anchored at t
#             idx.append(df.index[end])                    # index = decision time t

#         self.X = torch.tensor(np.stack(X_list), dtype=torch.float32)
#         self.y = torch.tensor(np.array(y_list), dtype=torch.long)
#         self.index = pd.Index(idx)

#     def __len__(self): return len(self.y)
#     def __getitem__(self, idx): return self.X[idx], self.y[idx]

# # ---------- Datasets with regime features & cross-sectional ----------
# class TimeSeriesDatasetWithRegime(Dataset):
#     def __init__(self, df, features, regime_df, seq_len=30):
#         self.seq_len = seq_len
#         self.features = features
#         self.regime_df = regime_df
#         X_list, y_list, r_list, idx = [], [], [], []
#         if len(df) <= seq_len:
#             self.X = torch.empty(0, seq_len, len(features))
#             self.R = torch.empty(0, regime_df.shape[1])
#             self.y = torch.empty(0, dtype=torch.long)
#             self.index = pd.DatetimeIndex([])
#             return
#         for i in range(len(df) - seq_len):
#             end = i + seq_len - 1            # decision time t
#             tgt_ts = df.index[end]
#             X_list.append(df[features].iloc[i:i+seq_len].values)
#             y_list.append(int(df['Target'].iloc[end]))
#             r_list.append(regime_df.reindex([tgt_ts]).iloc[0].values)
#             idx.append(tgt_ts)
#         self.X = torch.tensor(np.stack(X_list), dtype=torch.float32)
#         self.R = torch.tensor(np.stack(r_list), dtype=torch.float32)
#         self.y = torch.tensor(np.array(y_list), dtype=torch.long)
#         self.index = pd.Index(idx)

#     def __len__(self): return len(self.y)
#     def __getitem__(self, idx): return self.X[idx], self.R[idx], self.y[idx]

# class CrossSectionalDataset(Dataset):
#     def __init__(self, per_ticker_ds: dict):
#         self.samples = []
#         for t, ds in per_ticker_ds.items():
#             for i in range(len(ds)):
#                 self.samples.append((t, ds.X[i], ds.R[i], ds.y[i], ds.index[i]))
#         if len(self.samples) == 0:
#             self.X = torch.empty(0,1,1); self.R = torch.empty(0,1); self.y = torch.empty(0, dtype=torch.long)
#             self.tickers = []; self.index = pd.DatetimeIndex([]); return
#         self.X = torch.stack([s[1] for s in self.samples])
#         self.R = torch.stack([s[2] for s in self.samples])
#         self.y = torch.stack([s[3] for s in self.samples])
#         self.tickers = [s[0] for s in self.samples]
#         self.index = pd.Index([s[4] for s in self.samples])
#     def __len__(self): return len(self.y)
#     def __getitem__(self, idx): return self.X[idx], self.R[idx], self.y[idx]

# # ---------- Mixture-of-Experts model (regime-aware head) ----------
# class PositionalEncoding(nn.Module):
#     def __init__(self, d_model, max_len=1024):
#         super().__init__()
#         pe = torch.zeros(max_len, d_model)
#         pos = torch.arange(0, max_len).unsqueeze(1).float()
#         div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
#         pe[:, 0::2] = torch.sin(pos * div)
#         pe[:, 1::2] = torch.cos(pos * div)
#         self.register_buffer('pe', pe.unsqueeze(0))
#     def forward(self, x):
#         S = x.size(1)
#         return x + self.pe[:, :S, :]

# class RegimeMoEClassifier(nn.Module):
#     def __init__(self, in_features, seq_len=30, d_model=64, nhead=4, num_layers=2,
#                  r_features=4, n_experts=3):
#         super().__init__()
#         self.seq_len = seq_len
#         self.input_proj = nn.Linear(in_features, d_model)
#         self.pos = PositionalEncoding(d_model, max_len=seq_len)
#         enc = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=128, batch_first=True)
#         self.transformer = nn.TransformerEncoder(enc, num_layers)

#         rep_dim = seq_len * d_model
#         self.backbone_head = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(rep_dim, 128),
#             nn.ReLU(),
#             nn.Dropout(0.2)
#         )
#         self.experts = nn.ModuleList([nn.Linear(128, 2) for _ in range(n_experts)])
#         self.gate = nn.Sequential(
#             nn.Linear(r_features, 16),
#             nn.ReLU(),
#             nn.Linear(16, n_experts)
#         )

#     def forward(self, x, r):
#         x = self.input_proj(x)
#         x = self.pos(x)
#         x = self.transformer(x)
#         h = self.backbone_head(x)
#         weights = torch.softmax(self.gate(r), dim=1)
#         logits_stack = torch.stack([exp(h) for exp in self.experts], dim=1)  # (B, n_exp, 2)
#         logits = (logits_stack * weights.unsqueeze(-1)).sum(dim=1)           # (B, 2)
#         return logits

# class TransformerClassifier(nn.Module):
#     def __init__(self, in_features, seq_len=30, d_model=64, nhead=4, num_layers=2):
#         super().__init__()
#         self.seq_len = seq_len
#         self.input_proj = nn.Linear(in_features, d_model)
#         self.pos = PositionalEncoding(d_model, max_len=seq_len)
#         enc = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=128, batch_first=True)
#         self.transformer = nn.TransformerEncoder(enc, num_layers)
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(seq_len*d_model, 128),
#             nn.ReLU(),
#             nn.Dropout(0.2),
#             nn.Linear(128, 2)
#         )
#     def forward(self, x):
#         x = self.input_proj(x)
#         x = self.pos(x)
#         x = self.transformer(x)
#         return self.classifier(x)

# # ---------- Loss: Focal ----------
# class FocalLoss(nn.Module):
#     def __init__(self, alpha=0.5, gamma=2.0):
#         super().__init__(); self.alpha, self.gamma = alpha, gamma
#     def forward(self, logits, targets):
#         ce = nn.functional.cross_entropy(logits, targets, reduction='none')
#         p = torch.softmax(logits, dim=1).gather(1, targets.view(-1,1)).squeeze()
#         loss = self.alpha * (1 - p).pow(self.gamma) * ce
#         return loss.mean()

# # ---------- Train / Early stopping ----------
# def train_with_early_stopping(model, Xtr, ytr, Xva, yva, epochs=150, lr=1e-4, patience=10,
#                               seed=SEED, batch_size=64, weights_tr: torch.Tensor=None,
#                               use_focal=True):
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     model = model.to(device)
#     opt = optim.Adam(model.parameters(), lr=lr)
#     base_loss = FocalLoss() if use_focal else nn.CrossEntropyLoss(reduction='none')
#     g = torch.Generator().manual_seed(SEED)

#     if weights_tr is None:
#         weights_tr = torch.ones(len(Xtr), dtype=torch.float32)
#     ds = TensorDataset(Xtr, ytr, weights_tr)
#     tr_loader = DataLoader(ds, batch_size=batch_size, shuffle=True, generator=g)

#     best_w, best_val = None, float('inf'); bad = 0
#     for ep in range(1, epochs+1):
#         model.train(); tot = 0.0; batches = 0
#         for xb, yb, wb in tr_loader:
#             xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
#             opt.zero_grad()
#             logits = model(xb)
#             if isinstance(base_loss, FocalLoss):
#                 loss = base_loss(logits, yb)
#             else:
#                 loss_vec = base_loss(logits, yb)
#                 loss = (loss_vec * wb).mean()
#             loss.backward(); opt.step()
#             tot += loss.item(); batches += 1
#         model.eval()
#         with torch.no_grad():
#             v_logits = model(Xva.to(device))
#             vloss = (FocalLoss().to(device)(v_logits, yva.to(device))
#                      if isinstance(base_loss, FocalLoss)
#                      else nn.CrossEntropyLoss()(v_logits, yva.to(device)))
#         print(f"Epoch {ep:03d} | train_loss={tot/max(batches,1):.4f} | val_loss={vloss:.4f}")
#         if vloss < best_val:
#             best_val, bad = vloss.item() if torch.is_tensor(vloss) else float(vloss), 0
#             best_w = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
#         else:
#             bad += 1
#             if bad > patience:
#                 print(f"Early stopping at epoch {ep}")
#                 break
#     if best_w is not None:
#         model.load_state_dict(best_w)
#     return model

# # ---------- Threshold selection (precision) ----------
# def pick_threshold_for_precision(p, y, min_signals=12):
#     thrs = np.linspace(0.60, 0.90, 61)
#     best = (0.0, 0.50, 0)
#     for t in thrs:
#         yhat = (p >= t).astype(int)
#         n = int(yhat.sum())
#         if n < min_signals:
#             continue
#         prec = precision_score(y, yhat, zero_division=0)
#         if prec > best[0]:
#             best = (prec, float(t), n)
#     return best

# def pick_threshold_from_val(model, Xva, yva):
#     device = next(model.parameters()).device
#     model.eval()
#     with torch.no_grad():
#         p = torch.softmax(model(Xva.to(device)), dim=1)[:, 1].cpu().numpy()
#     thr_grid = np.linspace(0.45, 0.65, 41)
#     scores = [f1_score(yva.numpy(), (p >= t).astype(int)) for t in thr_grid]
#     t_best = float(thr_grid[int(np.argmax(scores))])
#     return t_best

# # ---------- Platt calibration ----------
# class PlattCalibrator:
#     def __init__(self):
#         self.lr = None
#     def fit(self, p_val: np.ndarray, y_val: np.ndarray):
#         y_unique = np.unique(y_val)
#         if len(y_unique) < 2:
#             self.lr = None
#             return self
#         x = np.clip(p_val.reshape(-1,1), 1e-6, 1-1e-6)
#         self.lr = LogisticRegression(solver='lbfgs')
#         self.lr.fit(x, y_val)
#         return self
#     def transform(self, p: np.ndarray) -> np.ndarray:
#         if self.lr is None:
#             return p
#         x = np.clip(p.reshape(-1,1), 1e-6, 1-1e-6)
#         return self.lr.predict_proba(x)[:,1]

# def platt_fit_apply(p_val: np.ndarray, y_val: np.ndarray, p_test: np.ndarray):
#     cal = PlattCalibrator().fit(p_val, y_val)
#     return cal.transform(p_test), cal

# # ---------- Helper: robust label -> numpy ----------
# def _as_numpy_1d(y):
#     if isinstance(y, np.ndarray):
#         arr = y
#     elif hasattr(y, "detach"):
#         arr = y.detach().cpu().numpy()
#     elif hasattr(y, "numpy"):
#         arr = y.numpy()
#     else:
#         arr = np.asarray(y)
#     return arr.astype(int).reshape(-1)

# # ---------- Eval helpers ----------
# def eval_classification(model, X, y, thr):
#     y_np = _as_numpy_1d(y)
#     device = next(model.parameters()).device
#     model.eval()
#     with torch.no_grad():
#         p = torch.softmax(model(X.to(device)), dim=1)[:, 1].cpu().numpy()
#     yhat = (p >= thr).astype(int)
#     return {
#         "acc": accuracy_score(y_np, yhat),
#         "prec": precision_score(y_np, yhat, zero_division=0),
#         "rec": recall_score(y_np, yhat, zero_division=0),
#         "f1": f1_score(y_np, yhat, zero_division=0),
#         "probs": p,
#         "yhat": yhat
#     }

# # ---------- "Today" inference ----------
# def infer_today_prob(df_full_feats: pd.DataFrame, features, scaler: RobustScaler, model, seq_len=30):
#     window = df_full_feats[features].iloc[-seq_len:].copy()
#     if len(window) < seq_len:
#         return None, None
#     window_scaled = scaler.transform(window)
#     if isinstance(window_scaled, pd.DataFrame):
#         window_scaled = window_scaled.to_numpy()
#     X_live = torch.tensor(window_scaled, dtype=torch.float32).unsqueeze(0)
#     device = next(model.parameters()).device
#     model.eval()
#     with torch.no_grad():
#         prob_up = torch.softmax(model(X_live.to(device)), dim=1)[:, 1].item()
#     return df_full_feats.index[-1], float(prob_up)

# # --- Live inference on arbitrary dates (single-ticker model) ---
# def infer_live_probs_basic(df_full_feats: pd.DataFrame, features, scaler: RobustScaler,
#                            model, seq_len: int, dates: pd.DatetimeIndex) -> pd.Series:
#     dates = pd.DatetimeIndex(dates)
#     device = next(model.parameters()).device
#     out = []
#     for d in dates:
#         if d not in df_full_feats.index:
#             out.append(np.nan); continue
#         end_loc = df_full_feats.index.get_loc(d)
#         if isinstance(end_loc, slice):
#             end_loc = end_loc.stop - 1
#         start = end_loc - (seq_len - 1)
#         if start < 0:
#             out.append(np.nan); continue
#         window = df_full_feats[features].iloc[start:end_loc+1]
#         X = scaler.transform(window)
#         if isinstance(X, pd.DataFrame):
#             X = X.to_numpy()
#         X = torch.tensor(X, dtype=torch.float32).unsqueeze(0).to(device)
#         with torch.no_grad():
#             p = torch.softmax(model(X), dim=1)[:, 1].item()
#         out.append(p)
#     return pd.Series(out, index=dates, name="prob_up").dropna()

# # --- Live inference on arbitrary dates (cross-sectional MoE model + regime) ---
# def infer_live_probs_cs(df_full_feats: pd.DataFrame, features, scaler: RobustScaler,
#                         model, regime_df: pd.DataFrame, seq_len: int,
#                         dates: pd.DatetimeIndex) -> pd.Series:
#     dates = pd.DatetimeIndex(dates)
#     device = next(model.parameters()).device
#     out = []
#     for d in dates:
#         if d not in df_full_feats.index:
#             out.append(np.nan); continue
#         end_loc = df_full_feats.index.get_loc(d)
#         if isinstance(end_loc, slice):
#             end_loc = end_loc.stop - 1
#         start = end_loc - (seq_len - 1)
#         if start < 0:
#             out.append(np.nan); continue
#         window = df_full_feats[features].iloc[start:end_loc+1]
#         X = scaler.transform(window)
#         if isinstance(X, pd.DataFrame):
#             X = X.to_numpy()
#         X = torch.tensor(X, dtype=torch.float32).unsqueeze(0).to(device)
#         r = regime_df.reindex([d]).iloc[0].values.reshape(1, -1)
#         R = torch.tensor(r, dtype=torch.float32).to(device)
#         with torch.no_grad():
#             p = torch.softmax(model(X, R), dim=1)[:, 1].item()
#         out.append(p)
#     return pd.Series(out, index=dates, name="prob_up").dropna()

# # ---------- Band (hysteresis) ----------
# def state_from_band(probs: np.ndarray, idx: pd.Index, thr_entry=0.78, thr_exit=0.48):
#     state = np.zeros_like(probs, dtype=bool)
#     for i, p in enumerate(probs):
#         if i == 0:
#             state[i] = (p >= thr_entry)
#         else:
#             state[i] = (p >= (thr_exit if state[i-1] else thr_entry))
#     return pd.Series(state, index=idx, name='state')

# def eval_classification_band(probs, y, idx, thr_entry, thr_exit):
#     y_np = _as_numpy_1d(y)
#     s = state_from_band(probs, idx, thr_entry, thr_exit)
#     yhat = s.astype(int).values
#     return {
#         "acc": accuracy_score(y_np, yhat),
#         "prec": precision_score(y_np, yhat, zero_division=0),
#         "rec": recall_score(y_np, yhat, zero_division=0),
#         "f1": f1_score(y_np, yhat, zero_division=0),
#         "probs": probs,
#         "yhat": yhat,
#         "state": s
#     }

# # ---------- Band artifacts + entry-only gating + vol gate helpers ----------
# def _safe_band_path(ticker: str, out_dir: str) -> str:
#     os.makedirs(out_dir, exist_ok=True)
#     return os.path.join(out_dir, f"{_safe_ticker_for_path(ticker)}_band.json")

# def save_band(ticker: str, best: dict, out_dir: str):
#     """
#     best = {"entry": float, "exit": float, ...}
#     """
#     path = _safe_band_path(ticker, out_dir)
#     with open(path, "w") as f:
#         json.dump(best, f, indent=2)

# def load_band(ticker: str, out_dir: str):
#     path = _safe_band_path(ticker, out_dir)
#     with open(path, "r") as f:
#         j = json.load(f)
#     return float(j["entry"]), float(j["exit"]), j

# def entry_only_gate(prev_final: bool, raw_curr: bool,
#                     trend_ok_today: bool, vol_ok_today: bool,
#                     override_used: bool) -> bool:
#     """
#     Gates checked only on entries (flat -> long).
#     """
#     is_entry = (not prev_final) and raw_curr
#     if is_entry:
#         return bool(raw_curr and ((trend_ok_today and vol_ok_today) or override_used))
#     else:
#         return bool(raw_curr)  # once in, gates don't force exit

# def apply_entry_only_gates(raw_state: pd.Series,
#                            trend_gate: pd.Series,
#                            vol_gate: pd.Series,
#                            override: pd.Series) -> pd.Series:
#     """
#     Build final state where gates can only *block new entries*.
#     """
#     idx = raw_state.index
#     trend_gate = trend_gate.reindex(idx).fillna(False)
#     vol_gate   = vol_gate.reindex(idx).fillna(False)
#     override   = override.reindex(idx).fillna(False)

#     out = []
#     prev_final = False
#     for i, d in enumerate(idx):
#         raw_curr = bool(raw_state.iloc[i])
#         if (not prev_final) and raw_curr:
#             passed = bool((trend_gate.iloc[i] and vol_gate.iloc[i]) or override.iloc[i])
#             final = raw_curr and passed
#         else:
#             final = raw_curr
#         out.append(final)
#         prev_final = final
#     return pd.Series(out, index=idx, name="state")

# def realized_vol(close: pd.Series, lookback: int = 60) -> pd.Series:
#     """Annualized rolling realized volatility from daily pct returns."""
#     ret = close.pct_change()
#     return (ret.rolling(lookback, min_periods=lookback//2).std() * np.sqrt(252.0)).rename("rv")

# def fit_vol_gate_threshold(vol_train: pd.Series, q_high: float = 0.80) -> float:
#     vol_train = vol_train.dropna()
#     return float(vol_train.quantile(q_high)) if len(vol_train) else np.inf


# # ---------- Backtest ----------
# def backtest_state(price_s: pd.Series, state: pd.Series,
#                    init_cash=10_000, fees=0.001, slippage=0.001,
#                    execution: str = "next_open"):  # "next_open" or "same_close"
#     state = state.astype(bool)
#     prev = state.shift(1, fill_value=False)
#     entries = (state & ~prev)
#     exits   = (~state & prev)

#     if execution == "next_open":
#         entries = entries.shift(1, fill_value=False)
#         exits   = exits.shift(1, fill_value=False)

#     idx = price_s.index.union(entries.index).union(exits.index)
#     price_s = price_s.reindex(idx).astype(float)
#     entries = entries.reindex(idx, fill_value=False).astype(bool)
#     exits   = exits.reindex(idx, fill_value=False).astype(bool)

#     if entries.any():
#         last_true_idx = entries.index[entries].max()
#         if not exits.any() or exits.index[exits].max() < last_true_idx:
#             exits.iloc[-1] = True

#     pf = vbt.Portfolio.from_signals(price_s, entries, exits,
#                                     init_cash=init_cash, fees=fees, slippage=slippage)

#     def _get_returns_and_curve(_pf):
#         ret_s, curve_s = None, None
#         for cand in ('returns',):
#             obj = getattr(_pf, cand, None)
#             if obj is None: continue
#             try:
#                 ret_s = obj() if callable(obj) else obj
#                 if isinstance(ret_s, (pd.Series, pd.DataFrame)):
#                     ret_s = ret_s.squeeze()
#                     break
#             except Exception:
#                 pass
#         for cand in ('value', 'asset_value', 'portfolio_value'):
#             obj = getattr(_pf, cand, None)
#             if obj is None: continue
#             try:
#                 curve_s = obj() if callable(obj) else obj
#                 if isinstance(curve_s, (pd.Series, pd.DataFrame)):
#                     curve_s = curve_s.squeeze()
#                     break
#             except Exception:
#                 pass
#         if (ret_s is None) and (curve_s is not None):
#             try:
#                 ret_s = curve_s.pct_change().dropna()
#             except Exception:
#                 pass
#         return ret_s, curve_s

#     try:
#         stats = pf.stats(freq='1D')
#     except TypeError:
#         stats = pf.stats()
#         rets, curve = _get_returns_and_curve(pf)
#         if ('Volatility [%]' not in stats.index) or pd.isna(stats.get('Volatility [%]', np.nan)):
#             if isinstance(rets, pd.Series) and len(rets) > 0:
#                 vol_pct = float(rets.std(ddof=0) * math.sqrt(252.0) * 100.0)
#                 stats.loc['Volatility [%]'] = vol_pct
#         if ('Max Drawdown [%]' not in stats.index) or pd.isna(stats.get('Max Drawdown [%]', np.nan)):
#             try:
#                 if isinstance(curve, pd.Series) and len(curve) > 1:
#                     dd = (curve / curve.cummax() - 1.0).min() * 100.0
#                     stats.loc['Max Drawdown [%]'] = float(dd)
#             except Exception:
#                 pass
#         wr = stats.get('Win Rate [%]', np.nan)
#         if pd.isna(wr):
#             try:
#                 rec = pf.trades.records_readable
#                 wins = int((rec['PnL'] > 0).sum()); total = int(len(rec))
#                 stats.loc['Win Rate [%]'] = (100.0 * wins / max(total, 1)) if total > 0 else np.nan
#             except Exception:
#                 pass

#     return pf, stats

# # ---------- Auto-tune band on validation ----------
# def pick_band_on_val(p_va, idx_va, close_series, y_va=None, min_trades=5,
#                      entry_grid=np.linspace(0.62, 0.90, 29), gap=0.25,
#                      init_cash=10_000, fees=0.001, slippage=0.001,
#                      objective="winrate"):
#     best = None
#     close_va = close_series.reindex(idx_va).dropna()
#     for e in entry_grid:
#         x = max(0.40, e - gap)
#         state = state_from_band(p_va, idx_va, thr_entry=e, thr_exit=x)
#         pf, stats = backtest_state(close_va, state, init_cash=init_cash, fees=fees, slippage=slippage)
#         trades = float(stats.get('Total Trades', 0.0))
#         print(f"Testing band: entry={e:.3f}, exit={x:.3f}, trades={trades}")
#         if trades < min_trades:
#             continue

#         if objective == "winrate":
#             wr = stats.get('Win Rate [%]', np.nan)
#             if pd.isna(wr):
#                 try:
#                     rec = pf.trades.records_readable
#                     wins = int((rec['PnL'] > 0).sum()); total = int(len(rec))
#                     wr = 100.0 * wins / max(total, 1)
#                 except Exception:
#                     wr = -np.inf
#             score = wr
#         elif objective == "precision":
#             if y_va is None:
#                 continue
#             m_va = eval_classification_band(p_va, y_va, idx_va, e, x)
#             score = 100.0 * m_va['prec']
#         else:  # return
#             score = float(stats['Total Return [%]'])
#         cand = (score, e, x, trades, stats)
#         if (best is None) or (cand[0] > best[0]):
#             best = cand
#     return best

# # ---------- Risk-aware band search with constraints ----------
# def _risk_score_from_stats(stats, lam=0.3, gamma=0.2):
#     ret = float(stats.get('Total Return [%]', 0.0))
#     vol = float(stats.get('Volatility [%]', stats.get('Volatility', 0.0)))
#     mdd = float(stats.get('Max Drawdown [%]', stats.get('Max Drawdown', 0.0)))
#     if np.isnan(ret): ret = 0.0
#     if np.isnan(vol): vol = 0.0
#     if np.isnan(mdd): mdd = 0.0
#     return ret - lam * vol - gamma * abs(mdd)

# def _rr_precheck_by_atr(df_feat: pd.DataFrame, entries_state: pd.Series,
#                         target_mult=1.5, stop_mult=1.0, atr_col='ATRp',
#                         max_fail_rate=0.5):
#     entries = entries_state[entries_state].index
#     if len(entries) < 3:
#         return False
#     atrp = df_feat[atr_col].reindex(entries).dropna()
#     if len(atrp) == 0:
#         return False
#     ok1 = (target_mult > stop_mult * 1.1)
#     ok2 = (atrp.median() < 0.06)
#     ok3 = (atrp.quantile(0.9) < 0.12)
#     fail_rate = 1.0 - float((atrp < 0.08).mean())
#     return bool(ok1 and ok2 and ok3 and (fail_rate <= max_fail_rate))

# def pick_band_on_val_riskaware(p_va, idx_va, close_series, df_feat_val,
#                                min_trades=20, enforce_entry_min=0.60,
#                                gap=0.25, lam=0.3, gamma=0.2,
#                                use_trend_gate=True, use_vol_gate=True,
#                                trend_gate_s=None, vol_gate_s=None,
#                                gate_relax_penalty=10.0,
#                                init_cash=10_000, fees=0.001, slippage=0.001):
#     best = None
#     close_va = close_series.reindex(idx_va).dropna()
#     if len(close_va) == 0:
#         return None
#     gate_penalty = 0.0
#     if not use_trend_gate or not use_vol_gate:
#         gate_penalty += gate_relax_penalty

#     for e in np.linspace(max(0.50, enforce_entry_min-0.05), 0.90, 36):
#         x = max(0.40, e - gap)
#         state = state_from_band(p_va, idx_va, thr_entry=e, thr_exit=x)

#         if e < enforce_entry_min:
#             entries = (state & ~state.shift(1, fill_value=False))
#             if not _rr_precheck_by_atr(df_feat_val, entries, target_mult=1.6, stop_mult=1.0):
#                 continue

#         gated_state = state.copy()
#         if use_trend_gate and trend_gate_s is not None:
#             gated_state &= trend_gate_s.reindex(gated_state.index).fillna(False)
#         if use_vol_gate and vol_gate_s is not None:
#             gated_state &= vol_gate_s.reindex(gated_state.index).fillna(False)

#         pf, stats = backtest_state(close_va, gated_state,
#                                    init_cash=init_cash, fees=fees, slippage=slippage,
#                                    execution="next_open")
#         trades = float(stats.get('Total Trades', 0.0))
#         if trades < min_trades:
#             continue
#         score = _risk_score_from_stats(stats, lam=lam, gamma=gamma) - gate_penalty
#         cand = (score, e, x, trades, stats)
#         print(f"[riskaware] entry={e:.3f} exit={x:.3f} trades={trades:.0f} score={score:.2f}")
#         if (best is None) or (cand[0] > best[0]):
#             best = cand
#     return best

# # ---------- Baseline: MA momentum + ATR-style exit ----------
# def baseline_ma_atr_signals(df_feat: pd.DataFrame,
#                             fast=50, slow=200,
#                             atr_mult_exit=1.5):
#     ma_fast = df_feat['MA50']
#     ma_slow = df_feat['MA200']
#     atr = df_feat['ATRp'] * df_feat['Close']
#     momentum_state = (ma_fast > ma_slow) & (ma_slow.pct_change(20) > 0)

#     exit_line = ma_fast - atr_mult_exit * (atr.fillna(0.0))
#     early_exit = df_feat['Close'] < exit_line

#     st = pd.Series(False, index=df_feat.index)
#     prev = False
#     for i in range(len(df_feat)):
#         if not prev and momentum_state.iloc[i]:
#             prev = True
#             st.iloc[i] = True
#         elif prev:
#             if (not momentum_state.iloc[i]) or early_exit.iloc[i]:
#                 prev = False
#                 st.iloc[i] = False
#             else:
#                 st.iloc[i] = True
#     return st.rename('baseline_state')

# # ---------- Pretty helpers for per-ticker reporting ----------
# def _print_prob_stats(name, probs):
#     if probs is None or len(probs) == 0:
#         print(f"[{name}] No probabilities to summarize.")
#         return
#     print(f"[{name}] prob_up range: min={np.min(probs):.3f} | max={np.max(probs):.3f} | mean={np.mean(probs):.3f}")

# def _print_signal_summary(ticker, state_series: pd.Series):
#     if state_series is None or len(state_series) == 0:
#         print(f"[{ticker}] No state to summarize.")
#         return
#     entries = (state_series & ~state_series.shift(1, fill_value=False))
#     exits   = (~state_series & state_series.shift(1, fill_value=False))
#     long_ratio = float(state_series.mean())
#     print(f"[{ticker}] signals: entries={int(entries.sum())}, exits={int(exits.sum())}, long-day%={100*long_ratio:.1f}%")

# def _print_bt_subset(ticker, stats):
#     fields = [
#         'Start', 'End', 'Period', 'Start Value', 'End Value',
#         'Total Return [%]', 'Max Drawdown [%]', 'Volatility [%]',
#         'Total Trades', 'Win Rate [%]', 'Profit Factor', 'Expectancy'
#     ]
#     print(f"\n--- {ticker} backtest summary (freq=1D) ---")
#     for f in fields:
#         if f in stats.index:
#             print(f"{f:>20}: {stats[f]}")

# # ---------- NEW: After-close decision helpers ----------
# def _next_business_day(d: pd.Timestamp) -> pd.Timestamp:
#     try:
#         return (pd.Timestamp(d) + BDay(1)).normalize()
#     except Exception:
#         return pd.Timestamp(d) + pd.Timedelta(days=1)

# def _format_after_close_message(ticker: str,
#                                 live_date: pd.Timestamp,
#                                 prev_final: bool,
#                                 curr_final: bool,
#                                 curr_raw: bool,
#                                 prob_today: float,
#                                 thr_entry: float,
#                                 thr_exit: float,
#                                 trend_ok_today: bool,
#                                 vol_ok_today: bool,
#                                 topn_kept_today: bool,
#                                 override_used: bool,
#                                 live_close: float) -> str:
#     """Return a single-line instruction tagged with the latest close date and close price."""
#     date_str = pd.Timestamp(live_date).date()
#     px_txt = f" | close={live_close:,.2f}"

#     # Transition flat -> long: enter at the NEXT session's open
#     if (not prev_final) and curr_final:
#         return (f"Enter Next Opening ({ticker}) ({date_str})  | prob_up={prob_today:.2%} | "
#                 f"band≥{thr_entry:.3f} | gates trend={'OK' if trend_ok_today else 'X'}, "
#                 f"vol={'OK' if vol_ok_today else 'X'}{px_txt}")

#     # Stay long
#     if prev_final and curr_final:
#         return f"Hold Long Position ({ticker}) ({date_str})  | prob_up={prob_today:.2%}{px_txt}"

#     # Transition long -> flat: exit at the NEXT session's open
#     if prev_final and (not curr_final):
#         reason = "band exit" if (not curr_raw) else "gate exit"
#         return (f"Exit Next Opening ({ticker}) ({date_str})  | {reason}, "
#                 f"prob_up={prob_today:.2%}, exit<{thr_exit:.3f}{px_txt}")

#     # Band passed but entry-only gates blocked the new position
#     if curr_raw and (not curr_final):
#         cause = []
#         if not trend_ok_today: cause.append("trend gate")
#         if not vol_ok_today:   cause.append("vol gate")
#         if not topn_kept_today:cause.append("top-N cap")
#         if override_used:      cause = []  # override bypasses gates
#         cause_txt = " & ".join(cause) if cause else "policy filter"
#         return (f"Flat: Band passed but {cause_txt} blocked ({ticker}) ({date_str})  | "
#                 f"prob_up={prob_today:.2%}{px_txt}")

#     # Nothing to do
#     return (f"Flat: No long (below entry band) ({ticker}) ({date_str})  | "
#             f"prob_up={prob_today:.2%}, entry≥{thr_entry:.3f}{px_txt}")


# # --- HOTFIX: ensure triple-barrier target truly mutates the passed DataFrame in place ---
# def add_targets_triple_barrier_inplace(df: pd.DataFrame, H=5, atr_mult=1.0):
#     """
#     Robust daily triple-barrier labeling (in-place).
#     - Upper/lower barriers: Close ± atr_mult * ATR(14)
#     - Horizon H (in bars) ahead.
#     - If both barriers are touched on the same bar, label is set to NaN (tie / unknown order).
#     """
#     atr = _atr(df, n=14)
#     up = df['Close'] + atr_mult * atr
#     dn = df['Close'] - atr_mult * atr

#     highs = df['High'].to_numpy()
#     lows  = df['Low'].to_numpy()
#     up_v  = up.to_numpy()
#     dn_v  = dn.to_numpy()

#     n = len(df)
#     tgt = np.full(n, np.nan, dtype=float)

#     for i in range(n - H):
#         hi_seq = highs[i+1:i+H+1]
#         lo_seq = lows[i+1:i+H+1]
#         up_hit_idx = np.where(hi_seq >= up_v[i])[0]
#         dn_hit_idx = np.where(lo_seq <= dn_v[i])[0]

#         if up_hit_idx.size == 0 and dn_hit_idx.size == 0:
#             continue
#         if up_hit_idx.size > 0 and (dn_hit_idx.size == 0 or up_hit_idx[0] < dn_hit_idx[0]):
#             tgt[i] = 1.0
#         elif dn_hit_idx.size > 0 and (up_hit_idx.size == 0 or dn_hit_idx[0] < up_hit_idx[0]):
#             tgt[i] = 0.0
#         else:
#             pass

#     df['Target'] = tgt
#     df.dropna(subset=['Target'], inplace=True)
#     df['Target'] = df['Target'].astype(int)
#     df['FwdRet_H'] = np.log(df['Close'].shift(-H) / df['Close'])

# # ---------- Market context download (SPY & VIX) ----------
# def _download_market(period="15y"):
#     # Use ADJUSTED OHLC for SPY to avoid split/dividend artifacts
#     spy = yf.download(
#         "SPY", period=period, interval="1d",
#         auto_adjust=True, prepost=False, actions=False,
#         threads=False, progress=False
#     )
#     if isinstance(spy.columns, pd.MultiIndex):
#         spy.columns = [c[0] for c in spy.columns]
#     spy = _normalize_daily_index_us(spy)
#     spy_close = spy["Close"].astype(float).rename("SPY_Close").ffill().bfill()

#     # VIX doesn't need adjustment; keep as-is
#     vix = yf.download(
#         "^VIX", period=period, interval="1d",
#         auto_adjust=False, prepost=False, actions=False,
#         threads=False, progress=False
#     )
#     if isinstance(vix.columns, pd.MultiIndex):
#         vix.columns = [c[0] for c in vix.columns]
#     vix = _normalize_daily_index_us(vix)
#     vix_close = vix["Close"].astype(float).rename("VIX_Close").ffill().bfill()

#     return spy_close, vix_close


# # ---------- Regime features ----------
# def compute_regime_features_by_date(per_ticker_df: dict, spy_close: pd.Series, vix_close: pd.Series):
#     """
#     Build regime features on the union of dates across all tickers (needs Close, MA50 in per_ticker_df).
#     """
#     all_idx = None
#     for df in per_ticker_df.values():
#         all_idx = df.index if all_idx is None else all_idx.union(df.index)
#     all_idx = pd.DatetimeIndex(sorted(all_idx))

#     spy = spy_close.reindex(all_idx).ffill()
#     vix = vix_close.reindex(all_idx).ffill()

#     spy_ma200 = spy.rolling(200).mean()
#     trend = ((spy > spy_ma200) & (spy_ma200.pct_change(20) > 0)).astype(float)

#     vix_z = (vix - vix.rolling(60).mean()) / (vix.rolling(60).std() + 1e-9)

#     # Breadth: % of tickers above their MA50
#     above_ma50 = []
#     for t, df in per_ticker_df.items():
#         ma50 = df['MA50'].reindex(all_idx)
#         close = df['Close'].reindex(all_idx)
#         above_ma50.append((close > ma50).astype(float))
#     breadth = (pd.concat(above_ma50, axis=1).mean(axis=1)
#                if len(above_ma50) > 0 else pd.Series(0.5, index=all_idx))

#     mkt_ret20 = np.log(spy / spy.shift(20))

#     reg = pd.DataFrame({
#         "REG_trend_up": trend,
#         "REG_vix_z": vix_z.clip(-4, 4),
#         "REG_breadth": breadth,
#         "REG_mkt_ret20": mkt_ret20.fillna(0.0)
#     }, index=all_idx).ffill().fillna(0.0)
#     return reg

# # ---------- Cross-sectional quantile calibration & top-N ----------
# def cs_quantile_calibrate(dates: pd.Index, tickers: list, probs: np.ndarray, blend=0.5):
#     df = pd.DataFrame({"date": dates, "ticker": tickers, "p": probs})
#     out = np.zeros_like(probs, dtype=float)
#     for d, grp in df.groupby("date"):
#         n = len(grp)
#         if n == 0:
#             continue
#         ranks = grp['p'].rank(method='average')  # 1..n
#         q = (ranks - 0.5) / n
#         raw = grp['p'].values
#         cal = blend * q.values + (1.0 - blend) * raw
#         out[grp.index.values] = cal
#     return out

# def top_n_mask_by_date(dates: pd.Index, probs: np.ndarray, N: int):
#     df = pd.DataFrame({"date": dates, "p": probs})
#     keep = np.zeros_like(probs, dtype=bool)
#     for d, grp in df.groupby('date'):
#         if N <= 0:
#             continue
#         idx_sorted = grp['p'].sort_values(ascending=False).index[:N]
#         keep[idx_sorted.values] = True
#     return keep

# # ---------- Universe training & calibration ----------
# def run_universe_pipeline(tickers: list,
#                           seq_len=30,
#                           epochs=120,
#                           patience=8,
#                           data_period="10y",
#                           split_mode="by_date",
#                           val_years=1, test_years=2,
#                           target_mode="riskadj_sign",
#                           H=10, eps=0.0,
#                           H_min=5, H_max=10,
#                           use_band=True,
#                           thr_entry=0.65,
#                           thr_exit=0.40,
#                           auto_band_from_val=True,
#                           band_objective="riskaware",   # 'riskaware', 'precision', 'winrate', 'return'
#                           min_trades_val=20,
#                           lam=0.3, gamma=0.2,
#                           use_trend_gate=True,
#                           use_vol_gate=True,
#                           atrp_min=0.004, atrp_max=0.10,
#                           use_focal_loss=True,
#                           weight_by_future_abs=True,
#                           evaluate_baseline=True,
#                           override_gate_prob=None,
#                           top_n_per_date: int = 0,
#                           init_cash=10_000, fees=0.001, slippage=0.001):
#     print("\n=== Cross-Sectional Pipeline ===")
#     mkt_period = "max" if str(data_period).lower() == "max" else (
#         data_period if (isinstance(data_period, str) and data_period.endswith("y")
#                         and int(re.sub(r'[^0-9]', '', data_period) or 0) >= 15) else "15y")
#     spy_close, vix_close = _download_market(period=mkt_period)

#     # 1) Build per-ticker **full features** and **labeled** frames
#     per_full = {}   # features only (for regime/gates & latest day)
#     per = {}        # labeled frames (for training)
#     for t in tickers:
#         try:
#             df_raw = yf.download(t, period=data_period, interval="1d",
#                                  auto_adjust=True, prepost=False, actions=False,
#                                  threads=False, progress=False)
#             if isinstance(df_raw.columns, pd.MultiIndex):
#                 df_raw.columns = [c[0] for c in df_raw.columns]
#             df_raw = df_raw[['Open','High','Low','Close','Volume']].ffill().bfill()
#             df_raw = _normalize_daily_index_us(df_raw)
#             if len(df_raw) < 300:
#                 print(f"{t}: not enough history"); continue
#             df_feat = compute_features(df_raw)
#             per_full[t] = df_feat.copy()

#             # Label targets on a copy
#             if target_mode == "riskadj_sign":
#                 df_lab = add_targets_risk_adjusted_sign_inplace(df_feat, H_min=H_min, H_max=H_max, eps=eps)
#             elif target_mode == "excess_sign":
#                 df_lab = add_targets_excess_sign_inplace(df_feat, market_close=spy_close, H=H, eps=eps)
#             elif target_mode == "horizon":
#                 df_lab = df_feat.copy()
#                 add_targets_horizon_inplace(df_lab, H=max(H_min,5), eps=eps)
#             else:
#                 df_lab = df_feat.copy()
#                 add_targets_triple_barrier_inplace(df_lab, H=max(H_min,5), atr_mult=1.0)

#             per[t] = df_lab
#         except Exception as e:
#             print(f"{t}: download/feature error -> {e}")

#     if len(per) == 0:
#         raise RuntimeError("No usable tickers for cross-sectional training.")

#     # 2) Regime features on union of dates
#     regime_df = compute_regime_features_by_date(per_full, spy_close, vix_close)

#     # 3) Split each name
#     splits = {}
#     for t, df in per.items():
#         try:
#             if split_mode == "by_date":
#                 tr, va, te = time_splits_by_date(df, test_years=test_years, val_years=val_years, purge_days=max(2, H_max))
#             else:
#                 tr, va, te = time_splits(df, purge_days=max(2, H_max))
#             splits[t] = (tr, va, te)
#         except Exception as e:
#             print(f"{t}: split error -> {e}")

#     # 4) Features list
#     features = [
#         'LogReturn','RSI','MACD','SignalLine','BB_upper','BB_lower',
#         'r1','r5','r10','ATRp','pctB','distMA50'
#     ]

#     # 5) Fit scaler on stacked TRAIN across names, then transform each split
#     train_stack = pd.concat([splits[t][0][features] for t in splits if len(splits[t][0])>0], axis=0)
#     scaler = RobustScaler()
#     try:
#         scaler.set_output(transform="pandas")
#     except Exception:
#         pass
#     scaler.fit(train_stack)

#     per_scaled = {}
#     for t, (tr, va, te) in splits.items():
#         tr_scaled = tr.copy(); va_scaled = va.copy(); te_scaled = te.copy()
#         tr_scaled[features] = scaler.transform(tr_scaled[features])
#         va_scaled[features] = scaler.transform(va_scaled[features])
#         te_scaled[features] = scaler.transform(te_scaled[features])
#         per_scaled[t] = (tr_scaled, va_scaled, te_scaled)

#     # 6) Build per-name datasets with regime inputs
#     per_ds = {}
#     for t, (tr, va, te) in per_scaled.items():
#         dtr = TimeSeriesDatasetWithRegime(tr, features, regime_df, seq_len)
#         dva = TimeSeriesDatasetWithRegime(va, features, regime_df, seq_len)
#         dte = TimeSeriesDatasetWithRegime(te, features, regime_df, seq_len)
#         if len(dtr)==0 or len(dva)==0 or len(dte)==0:
#             continue
#         per_ds[t] = {"train": dtr, "val": dva, "test": dte}

#     if len(per_ds) == 0:
#         raise RuntimeError("Datasets empty after seq_len.")

#     # 7) Assemble cross-sectional datasets
#     dtr_xs = CrossSectionalDataset({t: per_ds[t]["train"] for t in per_ds})
#     dva_xs = CrossSectionalDataset({t: per_ds[t]["val"] for t in per_ds})
#     dte_xs = CrossSectionalDataset({t: per_ds[t]["test"] for t in per_ds})

#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     model = RegimeMoEClassifier(in_features=len(features), seq_len=seq_len,
#                                 r_features=4, n_experts=3).to(device)

#     # 8) Train
#     def _train_loop(model, dtr, dva, epochs, patience, use_focal):
#         opt = optim.Adam(model.parameters(), lr=1e-4)
#         base_loss = FocalLoss() if use_focal else nn.CrossEntropyLoss()
#         g = torch.Generator().manual_seed(SEED)
#         tr_loader = DataLoader(TensorDataset(dtr.X, dtr.R, dtr.y), batch_size=64, shuffle=True, generator=g)
#         best_w, best_val = None, float('inf'); bad = 0
#         for ep in range(1, epochs+1):
#             model.train(); tot=0.0; batches=0
#             for xb, rb, yb in tr_loader:
#                 xb, rb, yb = xb.to(device), rb.to(device), yb.to(device)
#                 opt.zero_grad()
#                 logits = model(xb, rb)
#                 loss = base_loss(logits, yb)
#                 loss.backward(); opt.step()
#                 tot += loss.item(); batches += 1
#             model.eval()
#             with torch.no_grad():
#                 v_logits = model(dva.X.to(device), dva.R.to(device))
#                 vloss = base_loss(v_logits, dva.y.to(device))
#             print(f"[XS] Epoch {ep:03d} | train_loss={tot/max(batches,1):.4f} | val_loss={vloss:.4f}")
#             if vloss < best_val:
#                 best_val, bad = float(vloss), 0
#                 best_w = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
#             else:
#                 bad += 1
#                 if bad > patience:
#                     print(f"[XS] Early stopping at epoch {ep}")
#                     break
#         if best_w is not None:
#             model.load_state_dict(best_w)
#         return model

#     model = _train_loop(model, dtr_xs, dva_xs, epochs, patience, use_focal_loss)

#     # 9) Infer probs and per-date calibration
#     def _infer_probs(model, ds):
#         model.eval()
#         with torch.no_grad():
#             p = torch.softmax(model(ds.X.to(device), ds.R.to(device)), dim=1)[:,1].cpu().numpy()
#         return p

#     p_tr = _infer_probs(model, dtr_xs)
#     p_va = _infer_probs(model, dva_xs)
#     p_te = _infer_probs(model, dte_xs)

#     cal_va = cs_quantile_calibrate(dva_xs.index, dva_xs.tickers, p_va, blend=0.5)
#     cal_te = cs_quantile_calibrate(dte_xs.index, dte_xs.tickers, p_te, blend=0.5)

#     keep_mask_te = np.ones_like(cal_te, dtype=bool)
#     if top_n_per_date and top_n_per_date > 0:
#         keep_mask_te = top_n_mask_by_date(dte_xs.index, cal_te, top_n_per_date)

#     # --- Collect after-close messages here (per-ticker latest close) ---
#     after_close_msgs = []

#     # 10) Per-name evaluation, band selection, baseline comparison + reporting
#     per_results = {}
#     report_date = latest_us_trading_date()
#     for t, d in per_ds.items():
#         dva = d["val"]; dte = d["test"]

#         mask_va = (np.array(dva_xs.tickers) == t)
#         mask_te = (np.array(dte_xs.tickers) == t)
#         pva = cal_va[mask_va]
#         pte_t = cal_te[mask_te]
#         keep_ticker = keep_mask_te[mask_te]

#         df_full = per_full[t]     # FULL features (includes yesterday)
#         df_labeled = per[t]       # labeled (ends earlier)

#         # gates computed on FULL frame so they exist on yesterday too
#         trend_ok_full = (df_full['Close'] > df_full['MA200']) & (df_full['MA200'].pct_change(20) > 0)

#         # Quantile-based realized vol gate: fit on TRAIN, apply everywhere
#         rv_full  = realized_vol(df_full['Close'], lookback=60)
#         rv_train = rv_full.reindex(splits[t][0].index).dropna()  # TRAIN index for this ticker
#         vol_thr_t = fit_vol_gate_threshold(rv_train, q_high=0.80)
#         vol_ok_full = (rv_full <= vol_thr_t)

#         trend_gate_va = trend_ok_full.reindex(dva.index).fillna(False)
#         vol_gate_va   = vol_ok_full.reindex(dva.index).fillna(False)

#         # Band search
#         band_dir = os.path.join("artifacts", "bands")
#         thr_entry_t, thr_exit_t = thr_entry, thr_exit  # defaults

#         # Try to load frozen band first
#         try:
#             loaded_entry, loaded_exit, _meta = load_band(t, band_dir)
#             thr_entry_t, thr_exit_t = loaded_entry, loaded_exit
#             print(f"[{t}] Using FROZEN band from disk: entry={thr_entry_t:.3f} exit={thr_exit_t:.3f}")
#             auto_band_from_val = False  # prevent retuning on TEST
#         except Exception:
#             pass

#         if auto_band_from_val:
#             if band_objective == "riskaware":
#                 picked = pick_band_on_val_riskaware(
#                     p_va=pva, idx_va=dva.index, close_series=df_full['Open'],
#                     df_feat_val=df_full, min_trades=min_trades_val,
#                     enforce_entry_min=max(0.60, thr_entry),
#                     gap=max(0.20, thr_entry - thr_exit),
#                     lam=lam, gamma=gamma,
#                     use_trend_gate=use_trend_gate, use_vol_gate=use_vol_gate,
#                     trend_gate_s=trend_gate_va, vol_gate_s=vol_gate_va,
#                     gate_relax_penalty=10.0,
#                     init_cash=init_cash, fees=fees, slippage=slippage
#                 )
#                 if picked is not None:
#                     score, thr_entry_t, thr_exit_t, trades, _ = picked
#                     print(f"[{t}] picked band (riskaware): entry={thr_entry_t:.3f} exit={thr_exit_t:.3f} trades={int(trades)} score={score:.2f}")
#             else:
#                 picked = pick_band_on_val(
#                     p_va=pva, idx_va=dva.index, close_series=df_full['Open'],
#                     y_va=dva.y.numpy(), min_trades=min_trades_val,
#                     entry_grid=np.linspace(max(0.52, thr_entry-0.10), 0.90, 41),
#                     gap=max(0.20, thr_entry - thr_exit),
#                     init_cash=10_000, fees=fees, slippage=slippage,
#                     objective=band_objective
#                 )
#                 if picked is not None:
#                     score, thr_entry_t, thr_exit_t, trades, _ = picked
#                     print(f"[{t}] picked band ({band_objective}): entry={thr_entry_t:.3f} exit={thr_exit_t:.3f} trades={int(trades)} score={score:.2f}")

#         # Persist the chosen band (frozen from VAL) for paper/live
#         try:
#             save_band(t, {"entry": float(thr_entry_t), "exit": float(thr_exit_t)}, band_dir)
#         except Exception as e:
#             print(f"[{t}] Warning: couldn't save band -> {e}")

#         # Build TEST state with gates (TEST dates live on labeled frame indices)
#         s_test = state_from_band(pte_t, dte.index, thr_entry=thr_entry_t, thr_exit=thr_exit_t)

#         override = pd.Series(False, index=s_test.index)
#         if override_gate_prob is not None:
#             p_series = pd.Series(pte_t, index=dte.index)
#             override = p_series >= override_gate_prob

#         trend_gate = trend_ok_full.reindex(s_test.index).fillna(False) if use_trend_gate else pd.Series(True, index=s_test.index)
#         vol_gate   = vol_ok_full.reindex(s_test.index).fillna(False)   if use_vol_gate   else pd.Series(True, index=s_test.index)

#         # Entry-only gating on TEST
#         gated_state_test = apply_entry_only_gates(s_test, trend_gate, vol_gate, override)
#         if top_n_per_date and top_n_per_date > 0:
#             mask_series = pd.Series(keep_ticker, index=dte.index)
#             gated_state_test &= mask_series
#         gated_state_test = gated_state_test.rename('state')

#         price_test = df_full['Open'].reindex(gated_state_test.index).astype(float)
#         pf_m, st_m = backtest_state(price_test, gated_state_test,
#                                     init_cash=init_cash, fees=fees, slippage=slippage,
#                                     execution="next_open")

#         # Baseline
#         pf_b, st_b = None, None
#         if evaluate_baseline:
#             base_state = baseline_ma_atr_signals(df_full)
#             base_state = base_state.reindex(gated_state_test.index).fillna(False)
#             pf_b, st_b = backtest_state(price_test, base_state,
#                                         init_cash=init_cash, fees=fees, slippage=slippage,
#                                         execution="next_open")

#         # Reporting
#         _print_prob_stats(t, pte_t)
#         print(f"[{t}] band used: entry={thr_entry_t:.3f} | exit={thr_exit_t:.3f}")
#         print(f"[{t}] gates pass rates -> trend={trend_gate.mean():.3f} | vol={vol_gate.mean():.3f}")
#         _print_signal_summary(t, gated_state_test)
#         _print_bt_subset(t, st_m)
#         if evaluate_baseline and st_b is not None:
#             _print_bt_subset(f"{t} (baseline MA+ATR)", st_b)

#         per_results[t] = {
#             "band": (thr_entry_t, thr_exit_t),
#             "model_pf": pf_m, "model_stats": st_m,
#             "baseline_pf": pf_b, "baseline_stats": st_b
#         }

#         # --- AFTER-CLOSE: evaluate strictly at yesterday's US/Eastern trading day ---
#         eval_date = report_date
#         if eval_date not in df_full.index:
#             try:
#                 eval_date = df_full.index[df_full.index.get_loc(report_date, method='pad')]
#             except Exception:
#                 eval_date = df_full.index.max()

#         live_close = float(df_full['Close'].loc[eval_date])

#         if eval_date in dte.index:
#             loc = dte.index.get_loc(eval_date)
        
#             # --- LIVE recompute even inside TEST ---
#             p_live = infer_live_probs_cs(
#                 df_full, features, scaler, model, regime_df, seq_len,
#                 pd.DatetimeIndex([eval_date])
#             )
#             prob_today = float(p_live.iloc[0])
#             # --- END LIVE recompute ---
        
#             raw_curr   = bool(s_test.iloc[loc])
#             curr_final = bool(gated_state_test.iloc[loc])
#             prev_final = bool(gated_state_test.shift(1, fill_value=False).iloc[loc])
#             trend_ok_today = bool(trend_ok_full.reindex([eval_date]).fillna(False).iloc[0])
#             vol_ok_today   = bool(vol_ok_full.reindex([eval_date]).fillna(False).iloc[0])
#             override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
#             topn_today     = True
#             if top_n_per_date and top_n_per_date > 0:
#                 topn_today = bool(pd.Series(keep_ticker, index=dte.index).iloc[loc])

#         else:
#             # live inference for yesterday (beyond TEST end)
#             p_tail = infer_live_probs_cs(df_full, features, scaler, model, regime_df, seq_len,
#                                          pd.DatetimeIndex([eval_date]))
#             prob_today = float(p_tail.iloc[0])
#             raw_prev = bool(s_test.iloc[-1]) if len(s_test) else False
#             raw_curr = bool(prob_today >= (thr_exit_t if raw_prev else thr_entry_t))
#             trend_ok_today = bool(trend_ok_full.reindex([eval_date]).fillna(False).iloc[0])
#             vol_ok_today   = bool(vol_ok_full.reindex([eval_date]).fillna(False).iloc[0])
#             override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
#             prev_final = bool(gated_state_test.iloc[-1]) if len(gated_state_test) else False
#             curr_final = entry_only_gate(prev_final, raw_curr, trend_ok_today, vol_ok_today, override_used)
#             topn_today = True  # can't compute cross-sec cap for tail

#         msg = _format_after_close_message(
#             ticker=t,
#             live_date=eval_date,        # this is yesterday (or last <= yesterday)
#             prev_final=prev_final,
#             curr_final=curr_final,
#             curr_raw=raw_curr,
#             prob_today=prob_today,
#             thr_entry=thr_entry_t,
#             thr_exit=thr_exit_t,
#             trend_ok_today=trend_ok_today,
#             vol_ok_today=vol_ok_today,
#             topn_kept_today=bool(topn_today),
#             override_used=override_used,
#             live_close=live_close
#         )
#         after_close_msgs.append(msg)

#     # 11) Aggregate & print + consolidated latest-close sheet
#     total_start, total_end, ok_count = 0.0, 0.0, 0
#     beats = 0
#     for t, res in per_results.items():
#         st_m = res["model_stats"]
#         sv = float(st_m.get("Start Value", np.nan))
#         ev = float(st_m.get("End Value", np.nan))
#         if not (np.isnan(sv) or np.isnan(ev)):
#             total_start += sv; total_end += ev; ok_count += 1
#         if evaluate_baseline and res["baseline_stats"] is not None:
#             mret = float(st_m.get("Total Return [%]", 0.0))
#             bret = float(res["baseline_stats"].get("Total Return [%]", 0.0))
#             if mret > max(bret, 0.0):
#                 beats += 1
#         print(f"[{t}] model_ret={st_m.get('Total Return [%]', np.nan):.2f}% | "
#               f"vol={st_m.get('Volatility [%]', np.nan):.2f}% | "
#               f"mdd={st_m.get('Max Drawdown [%]', np.nan):.2f}% | "
#               f"baseline_ret={(res['baseline_stats'].get('Total Return [%]', np.nan) if res['baseline_stats'] is not None else np.nan):.2f}%")

#     if ok_count > 0:
#         pnl = total_end - total_start
#         ret_pct = (total_end / total_start - 1.0) * 100.0 if total_start > 0 else float("nan")
#         print("\n=== Cross-Sectional Aggregate summary ===")
#         print(f"Tickers processed: {ok_count} | Beat baseline on {beats}")
#         print(f"Total Start Value: ${total_start:,.2f}")
#         print(f"Total End Value:   ${total_end:,.2f}")
#         print(f"Total PnL:         ${pnl:,.2f}  ({ret_pct:,.2f}%)")

#     if len(after_close_msgs) > 0:
#         print(f"\n=== After-Close Signals — {report_date.date()} (US/Eastern) ===")
#         for line in after_close_msgs:
#             print(line)

#     return {"model": model, "scaler": scaler, "per_results": per_results, "regime_df": regime_df}

# # ---------- HOTFIX: redefine compute_features to fix a minor type() check typo ----------
# def compute_features(df: pd.DataFrame) -> pd.DataFrame:
#     df = df.copy().ffill().bfill()
#     df = df[['Open', 'High', 'Low', 'Close', 'Volume']].astype(float)

#     delta = df['Close'].diff()
#     gain = delta.where(delta > 0, 0).rolling(14).mean()
#     loss = -delta.where(delta < 0, 0).rolling(14).mean()
#     rs = gain / loss.where(loss != 0, np.nan)
#     df['RSI'] = 100 - (100 / (1 + rs))

#     ema12 = df['Close'].ewm(span=12, adjust=False).mean()
#     ema26 = df['Close'].ewm(span=26, adjust=False).mean()
#     df['MACD'] = ema12 - ema26
#     df['SignalLine'] = df['MACD'].ewm(span=9, adjust=False).mean()

#     ma20 = df['Close'].rolling(20).mean()
#     std20 = df['Close'].rolling(20).std()
#     df['BB_upper'] = ma20 + 2 * std20
#     df['BB_lower'] = ma20 - 2 * std20

#     df['LogReturn'] = np.log(df['Close'] / df['Close'].shift(1))
#     df['r1'] = df['Close'].pct_change(1)
#     df['r5'] = df['Close'].pct_change(5)
#     df['r10'] = df['Close'].pct_change(10)

#     atr = _atr(df, n=14)
#     df['ATRp'] = atr / df['Close']

#     rng = df['BB_upper'] - df['BB_lower']
#     numerator = df['Close'] - df['BB_lower']
#     df['pctB'] = numerator / rng.where(rng != 0, np.nan)

#     df['MA50'] = df['Close'].rolling(50).mean()
#     df['distMA50'] = df['Close'] / df['MA50'] - 1.0
#     df['MA200'] = df['Close'].rolling(200).mean()

#     for col in ['RSI', 'MACD', 'SignalLine', 'BB_upper', 'BB_lower', 'LogReturn',
#                 'r1', 'r5', 'r10', 'ATRp', 'pctB', 'MA50', 'distMA50', 'MA200']:
#         if isinstance(df[col], pd.DataFrame):
#             raise ValueError(f"Column {col} is a DataFrame with shape {df[col].shape}, expected Series")
#         elif not isinstance(df[col], pd.Series):
#             raise ValueError(f"Column {col} is not a Series, got {type(df[col])}")

#     df.dropna(inplace=True)
#     return df


# # ---------- Main pipeline (single ticker) ----------
# def run_ticker_pipeline(ticker: str,
#                         seq_len=30,
#                         epochs=150,
#                         patience=10,
#                         purge_days=2,
#                         init_cash=10_000,
#                         fees=0.001,
#                         slippage=0.001,
#                         data_period="10y",
#                         split_mode="by_date",
#                         val_years=1,
#                         test_years=2,
#                         target_mode="horizon",
#                         H=3, eps=0.003,
#                         tb_H=5, tb_atr_mult=1.0,
#                         use_band=True,
#                         thr_entry=0.65,
#                         thr_exit=0.40,
#                         auto_band_from_val=False,
#                         band_objective="winrate",
#                         min_trades_val=5,
#                         use_trend_gate=True,
#                         use_vol_gate=True,
#                         atrp_min=0.004, atrp_max=0.10,
#                         use_focal_loss=True,
#                         weight_by_future_abs=True,
#                         device_str=None,
#                         # --- NEW OPTIONS ---
#                         risk_H_min=5, risk_H_max=10,
#                         evaluate_baseline=False,
#                         lam=0.3, gamma=0.2,
#                         calibrate_platt=False,
#                         override_gate_prob=None):
#     print(f"\n=== {ticker} ===")

#     purge_days = max(purge_days, H, risk_H_max)

#     # 1) Download + features (normalized to US/Eastern calendar dates)
#     df_raw = yf.download(
#         ticker, period=data_period, interval="1d",
#         auto_adjust=True, prepost=False, actions=False,
#         threads=False, progress=False
#     )
#     print(f"Raw data shape: {df_raw.shape}, columns: {df_raw.columns}")
#     print(f"Raw data date range: {df_raw.index.min()} to {df_raw.index.max()}")
#     if isinstance(df_raw.columns, pd.MultiIndex):
#         df_raw.columns = [col[0] for col in df_raw.columns]
#     df_raw = df_raw[['Open', 'High', 'Low', 'Close', 'Volume']].ffill().bfill()
#     df_raw = _normalize_daily_index_us(df_raw)  # normalize to US/Eastern dates
#     if df_raw.empty or len(df_raw) < 300:
#         raise ValueError(f"Not enough data for {ticker}")
#     df_feat = compute_features(df_raw)

#     # 2) Splits
#     if split_mode == "by_date":
#         train, val, test = time_splits_by_date(df_feat, test_years=test_years, val_years=val_years, purge_days=purge_days)
#     else:
#         train, val, test = time_splits(df_feat, val_frac=0.15, test_frac=0.15, purge_days=purge_days)

#     # 3) Targets
#     if target_mode == "horizon":
#         for part in (train, val, test):
#             add_targets_horizon_inplace(part, H=H, eps=eps)
#     elif target_mode == "triple_barrier":
#         for part in (train, val, test):
#             add_targets_triple_barrier_inplace(part, H=tb_H, atr_mult=tb_atr_mult)
#     elif target_mode == "riskadj_sign":
#         for name, part in zip(['train','val','test'], (train, val, test)):
#             df_tmp = add_targets_risk_adjusted_sign_inplace(part, H_min=risk_H_min, H_max=risk_H_max, eps=eps)
#             if name == 'train':   train = df_tmp
#             elif name == 'val':   val = df_tmp
#             else:                 test = df_tmp
#     elif target_mode == "excess_sign":
#         spy_close, _vix = _download_market(period=data_period)
#         for name, part in zip(['train','val','test'], (train, val, test)):
#             df_tmp = add_targets_excess_sign_inplace(part, market_close=spy_close, H=max(H, risk_H_max), eps=eps)
#             if name == 'train':   train = df_tmp
#             elif name == 'val':   val = df_tmp
#             else:                 test = df_tmp
#     else:
#         raise ValueError(f"Unknown target_mode: {target_mode}")

#     # 4) Features
#     features = [
#         'LogReturn','RSI','MACD','SignalLine','BB_upper','BB_lower',
#         'r1','r5','r10','ATRp','pctB','distMA50'
#     ]

#     # 5) Scale on TRAIN only
#     scaler, train, val, test = scale_by_train(train, val, test, features)

#     # 6) Datasets
#     dtr = TimeSeriesDataset(train, features, seq_len)
#     dva = TimeSeriesDataset(val, features, seq_len)
#     dte = TimeSeriesDataset(test, features, seq_len)
#     if len(dtr) == 0 or len(dva) == 0 or len(dte) == 0:
#         raise ValueError("One of the splits is too short after seq_len; adjust years/fractions/seq_len.")

#     Xtr, ytr = dtr.X, dtr.y
#     Xva, yva = dva.X, dva.y
#     Xte, yte = dte.X, dte.y

#     # 6b) Sample weights
#     wtr = None
#     if weight_by_future_abs and 'FwdRet_H' in train.columns:
#         wtr = torch.tensor(
#             np.clip(train.loc[dtr.index, 'FwdRet_H'].abs().values, 1e-6, None),
#             dtype=torch.float32
#         )
#         wtr = wtr / (wtr.mean() + 1e-9)

#     # 7) Train
#     device = torch.device(device_str) if device_str else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     model = TransformerClassifier(in_features=len(features), seq_len=seq_len).to(device)
#     model = train_with_early_stopping(
#         model, Xtr, ytr, Xva, yva,
#         epochs=epochs, patience=patience,
#         weights_tr=wtr, use_focal=use_focal_loss
#     )

#     # 8) Probs for each split
#     with torch.no_grad():
#         p_tr = torch.softmax(model(dtr.X.to(device)), dim=1)[:, 1].cpu().numpy()
#         p_va = torch.softmax(model(dva.X.to(device)), dim=1)[:, 1].cpu().numpy()
#         p_te_raw = torch.softmax(model(dte.X.to(device)), dim=1)[:, 1].cpu().numpy()
#         print(f"Test set prob_up range (raw): min={p_te_raw.min():.3f}, max={p_te_raw.max():.3f}, mean={p_te_raw.mean():.3f}")

#     # C1) Optional Platt calibration (keep calibrator for live)
#     p_te = p_te_raw.copy()
#     platt_cal = None
#     if calibrate_platt:
#         p_te, platt_cal = platt_fit_apply(p_va, yva.numpy(), p_te_raw)
#         print("Applied Platt calibration to TEST probabilities (fit on VAL).")

#     # 9) Metrics & decision rule
#     if use_band:
#         # C2) Use frozen band if present, save if newly picked
#         band_dir = os.path.join("artifacts", "bands")
#         try:
#             thr_entry, thr_exit, _meta = load_band(ticker, band_dir)
#             print(f"[{ticker}] Using FROZEN band from disk: entry={thr_entry:.3f} exit={thr_exit:.3f}")
#             auto_band_from_val = False
#         except Exception:
#             pass

#         if auto_band_from_val:
#             trend_ok_val = (df_feat['Close'] > df_feat['MA200']) & (df_feat['MA200'].pct_change(20) > 0)
#             # keep ATRp-based gate here just for VAL band-pick (as before)
#             vol_ok_val   = df_feat['ATRp'].between(atrp_min, atrp_max)
#             if band_objective == "riskaware":
#                 picked = pick_band_on_val_riskaware(
#                     p_va, dva.index, df_feat['Open'],
#                     df_feat_val=df_feat,
#                     min_trades=max(min_trades_val, 20),
#                     enforce_entry_min=thr_entry,
#                     gap=max(0.12, thr_entry - thr_exit),
#                     lam=lam, gamma=gamma,
#                     use_trend_gate=use_trend_gate, use_vol_gate=use_vol_gate,
#                     trend_gate_s=trend_ok_val.reindex(dva.index),
#                     vol_gate_s=vol_ok_val.reindex(dva.index),
#                     gate_relax_penalty=10.0,
#                     init_cash=init_cash, fees=fees, slippage=slippage
#                 )
#             else:
#                 picked = pick_band_on_val(
#                     p_va, dva.index, df_feat['Open'],
#                     y_va=yva.numpy(), min_trades=min_trades_val,
#                     entry_grid=np.linspace(max(0.52, thr_entry-0.10), 0.90, 41),
#                     gap=max(0.20, thr_entry - thr_exit),
#                     init_cash=init_cash, fees=fees, slippage=slippage,
#                     objective=band_objective
#                 )
#             if picked is not None:
#                 score, thr_entry, thr_exit, trades, _stats = picked
#                 print(f"[Auto-band:{band_objective}] score={score:.2f} | thr_entry={thr_entry:.3f} thr_exit={thr_exit:.3f} | trades={int(trades)}")
#             else:
#                 print("[Auto-band] No band met min trade count on validation; using provided band.")

#         # Persist the chosen band
#         try:
#             save_band(ticker, {"entry": float(thr_entry), "exit": float(thr_exit)}, band_dir)
#         except Exception as e:
#             print(f"[{ticker}] Warning: couldn't save band -> {e}")

#         print(f"Using band: entry >= {thr_entry:.2f}, exit < {thr_exit:.2f}")
#         m_tr = eval_classification_band(p_tr, ytr, dtr.index, thr_entry, thr_exit)
#         m_va = eval_classification_band(p_va, yva, dva.index, thr_entry, thr_exit)
#         m_te = eval_classification_band(p_te, yte, dte.index, thr_entry, thr_exit)
#         raw_state_test = m_te["state"]
#         state_test = raw_state_test.copy()
#         thr = None
#     else:
#         prec, thr, cnt = pick_threshold_for_precision(p_va, yva.numpy(), min_signals=12)
#         if prec == 0.0:
#             thr = pick_threshold_from_val(model, Xva, yva)
#         thr = max(thr, 0.60)
#         print(f"Chosen threshold for precision: {thr:.3f}")
#         m_tr = eval_classification(model, Xtr, ytr, thr)
#         m_va = eval_classification(model, Xva, yva, thr)
#         m_te = eval_classification(model, Xte, yte, thr)
#         raw_state_test = pd.Series(m_te['probs'] >= thr, index=dte.index, name='state')
#         state_test = raw_state_test.copy()

#     print(f"Train  acc={m_tr['acc']:.3f} prec={m_tr['prec']:.3f} rec={m_tr['rec']:.3f} f1={m_tr['f1']:.3f}")
#     print(f"Val    acc={m_va['acc']:.3f} prec={m_va['prec']:.3f} rec={m_va['rec']:.3f} f1={m_va['f1']:.3f}")
#     print(f"Test   acc={m_te['acc']:.3f} prec={m_te['prec']:.3f} rec={m_te['rec']:.3f} f1={m_te['f1']:.3f}")

#     # 10) Regime & volatility gating on TEST + optional override
#     trend_ok = (df_feat['Close'] > df_feat['MA200']) & (df_feat['MA200'].pct_change(20) > 0)
#     trend_gate = trend_ok.reindex(state_test.index).fillna(False) if use_trend_gate else pd.Series(True, index=state_test.index)

#     # C3) Quantile-based realized vol threshold fitted on TRAIN
#     rv_full  = realized_vol(df_feat['Close'], lookback=60)
#     rv_train = rv_full.reindex(train.index).dropna()
#     vol_thr  = fit_vol_gate_threshold(rv_train, q_high=0.80)
#     vol_ok   = (rv_full <= vol_thr)
#     vol_gate = vol_ok.reindex(state_test.index).fillna(False) if use_vol_gate else pd.Series(True, index=state_test.index)

#     override = pd.Series(False, index=state_test.index)
#     if override_gate_prob is not None:
#         active_probs = m_te['probs']
#         override = pd.Series(active_probs, index=state_test.index) >= float(override_gate_prob)

#     # C4) Entry-only gating on TEST
#     gated_state_test = apply_entry_only_gates(state_test, trend_gate, vol_gate, override).rename('state')
#     print(f"Trend gate pass rate: {trend_gate.mean():.3f}")
#     print(f"Vol gate  pass rate: {vol_gate.mean():.3f}")
#     _print_signal_summary(ticker, gated_state_test)

#     # 11) Backtest on TEST (next-bar execution at next OPEN)
#     price_test = df_feat['Open'].reindex(gated_state_test.index).astype(float)
#     pf, stats = backtest_state(
#         price_test, gated_state_test,
#         init_cash=init_cash, fees=fees, slippage=slippage,
#         execution="next_open"
#     )
#     print("\n--- Backtest (TEST split, next-bar OPEN, after costs; gated) ---")
#     print(stats)
#     _print_bt_subset(ticker, stats)

#     # Baseline comparison (optional)
#     if evaluate_baseline:
#         base_state = baseline_ma_atr_signals(df_feat).reindex(gated_state_test.index).fillna(False)
#         pf_b, st_b = backtest_state(
#             price_test, base_state,
#             init_cash=init_cash, fees=fees, slippage=slippage,
#             execution="next_open"
#         )
#         print("\n--- Baseline (MA+ATR) backtest ---")
#         print(st_b)
#         try:
#             mret = float(stats.get('Total Return [%]', 0.0))
#             bret = float(st_b.get('Total Return [%]', 0.0))
#             print(f"Model beats baseline? {mret:.2f}% vs {bret:.2f}% -> {mret > max(bret, 0.0)}")
#         except Exception:
#             pass

#     # 12) After-close single-line instruction — ALWAYS yesterday's US/Eastern close (or nearest <=)
#     msg = None

#     report_date = latest_us_trading_date()
#     if report_date in df_feat.index:
#         eval_date = report_date
#     else:
#         try:
#             eval_date = df_feat.index[df_feat.index.get_loc(report_date, method='pad')]
#         except Exception:
#             eval_date = df_feat.index.max()

#     live_close = float(df_feat['Close'].loc[eval_date])

#     if use_band:
#         if eval_date in dte.index:
#             # Recompute prob live for that exact date (still inside TEST)
#             loc = dte.index.get_loc(eval_date)
#             p_live = infer_live_probs_basic(
#                 df_full_feats=df_feat, features=features, scaler=scaler, model=model,
#                 seq_len=seq_len, dates=pd.DatetimeIndex([eval_date])
#             )
#             prob_today = float(p_live.iloc[0])

#             # C5) Apply Platt calibration to live prob if enabled
#             if calibrate_platt and (platt_cal is not None):
#                 prob_today = float(platt_cal.transform(np.array([prob_today]))[0])

#             # Keep states from precomputed TEST artifacts
#             raw_curr      = bool(raw_state_test.iloc[loc])
#             curr_final    = bool(gated_state_test.iloc[loc])
#             prev_final    = bool(gated_state_test.shift(1, fill_value=False).iloc[loc])
#             # gate checks at eval_date
#             trend_ok_today = bool(trend_ok.reindex([eval_date]).fillna(False).iloc[0]) if use_trend_gate else True
#             vol_ok_today   = bool(vol_ok.reindex([eval_date]).fillna(False).iloc[0])   if use_vol_gate   else True
#             override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
#         else:
#             # Tail beyond TEST: infer prob, roll band by one, then entry-only gate
#             p_tail = infer_live_probs_basic(
#                 df_full_feats=df_feat, features=features, scaler=scaler, model=model,
#                 seq_len=seq_len, dates=pd.DatetimeIndex([eval_date])
#             )
#             prob_today = float(p_tail.iloc[0])

#             # C5) Apply Platt to tail prob if enabled
#             if calibrate_platt and (platt_cal is not None):
#                 prob_today = float(platt_cal.transform(np.array([prob_today]))[0])

#             prev_raw       = bool(raw_state_test.iloc[-1]) if len(raw_state_test) else False
#             raw_curr       = bool(prob_today >= (thr_exit if prev_raw else thr_entry))
#             prev_final     = bool(gated_state_test.iloc[-1]) if len(gated_state_test) else False
#             trend_ok_today = bool(trend_ok.reindex([eval_date]).fillna(False).iloc[0]) if use_trend_gate else True
#             vol_ok_today   = bool(vol_ok.reindex([eval_date]).fillna(False).iloc[0])   if use_vol_gate   else True
#             override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
#             # C6) Entry-only decision for tail
#             curr_final     = entry_only_gate(prev_final, raw_curr, trend_ok_today, vol_ok_today, override_used)
#     else:
#         # Threshold mode
#         if eval_date in dte.index:
#             p_live = infer_live_probs_basic(
#                 df_full_feats=df_feat, features=features, scaler=scaler, model=model,
#                 seq_len=seq_len, dates=pd.DatetimeIndex([eval_date])
#             )
#             prob_today = float(p_live.iloc[0])
#             if calibrate_platt and (platt_cal is not None):
#                 prob_today = float(platt_cal.transform(np.array([prob_today]))[0])
#         else:
#             p_tail = infer_live_probs_basic(
#                 df_full_feats=df_feat, features=features, scaler=scaler, model=model,
#                 seq_len=seq_len, dates=pd.DatetimeIndex([eval_date])
#             )
#             prob_today = float(p_tail.iloc[0])
#             if calibrate_platt and (platt_cal is not None):
#                 prob_today = float(platt_cal.transform(np.array([prob_today]))[0])

#         raw_curr       = bool(prob_today >= (thr if thr is not None else 0.5))
#         prev_final     = bool(gated_state_test.iloc[-1]) if len(gated_state_test) else False
#         trend_ok_today = bool(trend_ok.reindex([eval_date]).fillna(False).iloc[0]) if use_trend_gate else True
#         vol_ok_today   = bool(vol_ok.reindex([eval_date]).fillna(False).iloc[0])   if use_vol_gate   else True
#         override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
#         curr_final     = bool(raw_curr and ((trend_ok_today and vol_ok_today) or override_used))

#     msg = _format_after_close_message(
#         ticker=ticker,
#         live_date=eval_date,
#         prev_final=prev_final,
#         curr_final=curr_final,
#         curr_raw=raw_curr,
#         prob_today=float(prob_today),
#         thr_entry=float(thr_entry),
#         thr_exit=float(thr_exit),
#         trend_ok_today=bool(trend_ok_today),
#         vol_ok_today=bool(vol_ok_today),
#         topn_kept_today=True,
#         override_used=bool(override_used),
#         live_close=live_close
#     )

#     print("\n=== After-Close Signal (yesterday close) ===")
#     print(msg)
#     try:
#         if msg.startswith("Enter Next Opening"):
#             notify("Transformer Signal: ENTRY", msg)
#         elif msg.startswith("Exit Next Opening"):
#             notify("Transformer Signal: EXIT", msg)
#     except Exception:
#         pass


#     # 13) "Today" inference (convenience)
#     live_date_inf, live_prob = infer_today_prob(df_feat, features, scaler, model, seq_len=seq_len)
#     if live_date_inf is not None:
#         # C7) Apply Platt to today's prob
#         if calibrate_platt and (platt_cal is not None) and (live_prob is not None):
#             live_prob = float(platt_cal.transform(np.array([live_prob]))[0])

#         next_ix = df_feat.index[df_feat.index > live_date_inf]
#         next_date = next_ix[0] if len(next_ix) else None

#         prev_idx = gated_state_test.index[gated_state_test.index < live_date_inf]
#         st_yday = bool(gated_state_test.loc[prev_idx[-1]]) if len(prev_idx) else False

#         if use_band:
#             st_today_raw = (live_prob >= (thr_exit if st_yday else thr_entry))
#         else:
#             st_today_raw = (live_prob >= (thr if thr is not None else 0.5))

#         trend_ok_today = bool(trend_ok.reindex([live_date_inf]).fillna(False).iloc[0]) if use_trend_gate else True
#         vol_ok_today   = bool(vol_ok.reindex([live_date_inf]).fillna(False).iloc[0])   if use_vol_gate   else True
#         override_used  = (override_gate_prob is not None) and (live_prob >= float(override_gate_prob))

#         st_today = entry_only_gate(prev_final=st_yday, raw_curr=st_today_raw,
#                                    trend_ok_today=trend_ok_today, vol_ok_today=vol_ok_today,
#                                    override_used=override_used)

#         entry_today = (not st_yday) and st_today
#         exit_today  = st_yday and (not st_today)
#         close_today = float(df_feat['Close'].reindex([live_date_inf]).iloc[0])

#         when = f"next session ({next_date.date()})" if next_date is not None else "next session (no further bar available)"
#         if entry_today:
#             msg2 = f"{ticker}: ENTRY signal for {when} | prob_up={live_prob:.2%} | today_close=${close_today:,.2f}"
#             notify("Transformer Signal: ENTRY", msg2); print(msg2)
#         elif exit_today:
#             msg2 = f"{ticker}: EXIT signal for {when} | prob_up={live_prob:.2%} | today_close=${close_today:,.2f}"
#             notify("Transformer Signal: EXIT", msg2); print(msg2)
#         else:
#             state = "LONG" if st_today else "FLAT"
#             print(f"{ticker}: No new signal ({when}). State {state}. prob_up={live_prob:.2%} | today_close=${close_today:,.2f}")

#     return {
#         "model": model,
#         "scaler": scaler,
#         "threshold": thr if not use_band else None,
#         "band": (thr_entry, thr_exit) if use_band else None,
#         "datasets": {"train": dtr, "val": dva, "test": dte},
#         "metrics": {"train": m_tr, "val": m_va, "test": m_te},
#         "backtest": {"portfolio": pf, "stats": stats},
#         "features_df": df_feat
#     }


# # ---------- Run ----------
# if __name__ == "__main__":

#     total_start = 0.0
#     total_end = 0.0
#     total_beats = 0
#     ok_count = 0

#     tickers = [
#         "MSFT", "NVDA", "META",
#         "LRCX", "ICE",
#         "ORLY", "GS", "BLK", "PGR", "LLY",
#         "MRK", "ISRG", "HCA", "ETN",
#         "AAPL", "AMZN", "GOOGL", "CRM",
#         "CME", "TSLA", "COST", "V", "MCK",
#         "CAT", "ROK", "CVX",
#     ]


#     USE_CROSS_SECTIONAL = True  # set True to run the cross-sectional pipeline with calibration

#     if USE_CROSS_SECTIONAL:
#         # Cross-sectional training + calibrated bands + baseline comparison + richer per-ticker reports
#         run_universe_pipeline(
#             tickers=tickers,
#             seq_len=30,
#             epochs=120,
#             patience=8,
#             data_period="max",
#             split_mode="by_date",
#             val_years=3, test_years=5,
#             target_mode="riskadj_sign",  # or "excess_sign"
#             H=10, eps=0.0,
#             H_min=5, H_max=10,
#             use_band=True,
#             thr_entry=0.60,
#             thr_exit=0.45,
#             auto_band_from_val=True,
#             band_objective="precision",   # also supports: "riskaware", "winrate", "return"
#             min_trades_val=10,
#             lam=0.3, gamma=0.2,
#             use_trend_gate=True,
#             use_vol_gate=True,
#             atrp_min=0.004, atrp_max=0.10,
#             evaluate_baseline=True,
#             override_gate_prob=None,  # e.g., 0.92 to bypass gates on very high probs
#             top_n_per_date=0,         # set >0 to cap positions per day across names
#             init_cash=10_000, fees=0.001, slippage=0.001
#         )
#         print("\n[MAIN] Cross-sectional run complete.")
#     else:
#         # Single-name pipeline — prints the detailed report and an after-close single-line signal
#         for t in tickers:
#             try:
#                 res = run_ticker_pipeline(
#                     t,
#                     seq_len=30,
#                     epochs=120,
#                     patience=8,
#                     purge_days=3,
#                     init_cash=10_000,
#                     fees=0.001,
#                     slippage=0.001,
#                     data_period="10y",
#                     split_mode="by_date",
#                     val_years=1,
#                     test_years=2,
#                     target_mode="riskadj_sign",   # try: "riskadj_sign" / "excess_sign" / "horizon"
#                     H=3, eps=0.0,
#                     use_band=True,
#                     thr_entry=0.55,               # auto-band may raise this
#                     thr_exit=0.42,
#                     auto_band_from_val=True,      # enable to use band search
#                     band_objective="precision",   # "riskaware" / "winrate" / "precision" / "return"
#                     min_trades_val=10,
#                     use_trend_gate=True,
#                     use_vol_gate=True,
#                     atrp_min=0.004,
#                     atrp_max=0.10,
#                     use_focal_loss=False,
#                     weight_by_future_abs=True,
#                     evaluate_baseline=True,
#                     lam=0.3, gamma=0.2,
#                     calibrate_platt=True,
#                     override_gate_prob=None
#                 )

#                 # Aggregate Start/End values
#                 stats = res["backtest"]["stats"]
#                 sv = float(stats.get("Start Value", np.nan))
#                 ev = float(stats.get("End Value", np.nan))
#                 if not (np.isnan(sv) or np.isnan(ev)):
#                     total_start += sv
#                     total_end += ev
#                     ok_count += 1

#             except Exception as e:
#                 print(f"{t}: error -> {e}")

#         # Final aggregate summary
#         if ok_count > 0:
#             pnl = total_end - total_start
#             ret_pct = (total_end / total_start - 1.0) * 100.0 if total_start > 0 else float("nan")
#             print("\n=== Aggregate summary ===")
#             print(f"Tickers processed: {ok_count}")
#             print(f"Total Start Value: ${total_start:,.2f}")
#             print(f"Total End Value:   ${total_end:,.2f}")
#             print(f"Total PnL:         ${pnl:,.2f}  ({ret_pct:,.2f}%)")
#         else:
#             print("\nNo successful backtests to summarize.")

#     # Final confirmation print so the script always ends with output
#     print("\n=== Script finished ===")


In [ ]:
# # ===========================
# # PART 1 / 2  (FIXED & MERGED)
# # ===========================

# # --- Imports & setup ---
# import warnings, sys, os, re, random, platform, math, pickle
# import numpy as np, pandas as pd, torch
# import json


# warnings.filterwarnings("ignore", message="Metric '.*' requires frequency to be set")
# warnings.filterwarnings("ignore", message="Object has multiple columns.*")

# import yfinance as yf
# import vectorbt as vbt

# from sklearn.preprocessing import RobustScaler
# from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
# from sklearn.linear_model import LogisticRegression

# # Ensure sklearn keeps feature names / returns DataFrames where possible (silences warnings)
# try:
#     from sklearn import set_config
#     set_config(transform_output="pandas")
# except Exception:
#     pass

# from torch import nn, optim
# from torch.utils.data import Dataset, DataLoader, TensorDataset
# from pandas.tseries.offsets import BDay  # for next-business-day timestamps

# SEED = 42
# os.environ["PYTHONHASHSEED"] = str(SEED)
# random.seed(SEED); np.random.seed(SEED)
# torch.manual_seed(SEED)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(SEED)
# try:
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False
#     torch.use_deterministic_algorithms(True)
# except Exception:
#     pass

# # ---------- Notification helper ----------
# def notify(title: str, message: str):
#     sent = False
#     try:
#         from plyer import notification
#         notification.notify(title=title, message=message, app_name="TF Signal", timeout=8)
#         sent = True
#     except Exception:
#         pass
#     if not sent:
#         try:
#             if platform.system() == "Windows":
#                 import winsound; winsound.MessageBeep()
#         except Exception:
#             pass
#         print(f"[NOTIFY] {title}: {message}")

# def _safe_ticker_for_path(t: str) -> str:
#     return re.sub(r"[^A-Za-z0-9_.-]+", "_", t)

# # ---------- Yahoo daily normalization ----------
# def _normalize_daily_index_us(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Normalize yfinance daily index to US/Eastern calendar days (naive midnight) and drop dups.
#     Prevents mixed T vs T-1 dates caused by timezone offsets.
#     """
#     df = df.copy()
#     idx = pd.DatetimeIndex(df.index)
#     if idx.tz is not None:
#         idx = idx.tz_convert('America/New_York').tz_localize(None)
#     idx = pd.to_datetime(idx.date)   # force to calendar date (midnight)
#     df.index = idx
#     df = df[~df.index.duplicated(keep='last')]  # keep the latest row if duplicates
#     return df

# def latest_us_trading_date(buffer_min: int = 5) -> pd.Timestamp:
#     """
#     Return yesterday's US trading date (US/Eastern) unless today's close + buffer has passed.
#     """
#     now_et = pd.Timestamp.now(tz='America/New_York')
#     d = now_et.normalize()
#     # roll back weekends
#     while d.weekday() >= 5:
#         d = d - BDay(1)
#     close_ts = d + pd.Timedelta(hours=16)
#     if now_et < close_ts + pd.Timedelta(minutes=buffer_min):
#         d = d - BDay(1)
#     return d.tz_localize(None)

# # ---------- Technical helpers ----------
# def _atr(df: pd.DataFrame, n=14):
#     high, low, close = df['High'], df['Low'], df['Close']
#     prev_close = close.shift(1)
#     tr1 = (high - low).abs()
#     tr2 = (high - prev_close).abs()
#     tr3 = (low - prev_close).abs()
#     tr = np.maximum(tr1, np.maximum(tr2, tr3))
#     atr = tr.rolling(n).mean()
#     return atr

# # ---------- Features (precision-oriented & lean) ----------
# def compute_features(df: pd.DataFrame) -> pd.DataFrame:
#     df = df.copy().ffill().bfill()
#     df = df[['Open', 'High', 'Low', 'Close', 'Volume']].astype(float)

#     delta = df['Close'].diff()
#     gain = delta.where(delta > 0, 0).rolling(14).mean()
#     loss = -delta.where(delta < 0, 0).rolling(14).mean()
#     rs = gain / loss.where(loss != 0, np.nan)
#     df['RSI'] = 100 - (100 / (1 + rs))

#     ema12 = df['Close'].ewm(span=12, adjust=False).mean()
#     ema26 = df['Close'].ewm(span=26, adjust=False).mean()
#     df['MACD'] = ema12 - ema26
#     df['SignalLine'] = df['MACD'].ewm(span=9, adjust=False).mean()

#     ma20 = df['Close'].rolling(20).mean()
#     std20 = df['Close'].rolling(20).std()
#     df['BB_upper'] = ma20 + 2 * std20
#     df['BB_lower'] = ma20 - 2 * std20

#     df['LogReturn'] = np.log(df['Close'] / df['Close'].shift(1))
#     df['r1'] = df['Close'].pct_change(1)
#     df['r5'] = df['Close'].pct_change(5)
#     df['r10'] = df['Close'].pct_change(10)

#     atr = _atr(df, n=14)
#     df['ATRp'] = atr / df['Close']

#     rng = df['BB_upper'] - df['BB_lower']
#     numerator = df['Close'] - df['BB_lower']
#     df['pctB'] = numerator / rng.where(rng != 0, np.nan)

#     df['MA50'] = df['Close'].rolling(50).mean()
#     df['distMA50'] = df['Close'] / df['MA50'] - 1.0
#     df['MA200'] = df['Close'].rolling(200).mean()

#     for col in ['RSI', 'MACD', 'SignalLine', 'BB_upper', 'BB_lower', 'LogReturn',
#                 'r1', 'r5', 'r10', 'ATRp', 'pctB', 'MA50', 'distMA50', 'MA200']:
#         if isinstance(df[col], pd.DataFrame):
#             raise ValueError(f"Column {col} is a DataFrame with shape {df[col].shape}, expected Series")
#         elif not isinstance(df[col], pd.Series):
#             raise ValueError(f"Column {col} is not a Series, got {type[df[col]]}")

#     df.dropna(inplace=True)
#     return df

# # ---------- Splits ----------
# def time_splits(df: pd.DataFrame, val_frac=0.15, test_frac=0.15, purge_days=2):
#     n = len(df)
#     n_test = int(round(n * test_frac))
#     n_val = int(round(n * val_frac))
#     n_train = max(0, n - n_val - n_test)
#     if n_train < 60:
#         raise ValueError("Not enough data after splits; reduce seq_len or fractions.")
#     train = df.iloc[:n_train].copy()
#     val = df.iloc[n_train:n_train+n_val].copy()
#     test = df.iloc[n_train+n_val:].copy()
#     if purge_days > 0:
#         if len(val) > purge_days: val = val.iloc[purge_days:]
#         if len(test) > purge_days: test = test.iloc[purge_days:]
#     return train, val, test

# def time_splits_by_date(df: pd.DataFrame, test_years=2, val_years=1, purge_days=2):
#     last = df.index.max()
#     test_start = last - pd.DateOffset(years=test_years)
#     val_start = test_start - pd.DateOffset(years=val_years)

#     train = df.loc[:val_start - pd.Timedelta(days=1)].copy()
#     val = df.loc[val_start:test_start - pd.Timedelta(days=1)].copy()
#     test = df.loc[test_start:].copy()

#     if purge_days > 0:
#         if len(val) > purge_days: val = val.iloc[purge_days:]
#         if len(test) > purge_days: test = test.iloc[purge_days:]

#     # Ensure test includes the latest date in df
#     if test.index.max() < df.index.max():
#         test = df.loc[test_start:df.index.max()].copy()
#         if purge_days > 0 and len(test) > purge_days:
#             test = test.iloc[purge_days:]

#     if min(map(len, [train, val, test])) < 60:
#         raise ValueError("One of the splits is too short; reduce seq_len or adjust years.")
    
#     print(f"Train range: {train.index.min()} to {train.index.max()}")
#     print(f"Val range: {val.index.min()} to {val.index.max()}")
#     print(f"Test range: {test.index.min()} to {test.index.max()}")
#     return train, val, test

# # ---------- Targets ----------
# def add_targets_horizon_inplace(df: pd.DataFrame, H=3, eps=0.003):
#     df['FwdRet_H'] = np.log(df['Close'].shift(-H) / df['Close'])
#     y = np.where(df['FwdRet_H'] > eps, 1,
#                  np.where(df['FwdRet_H'] < -eps, 0, np.nan))
#     df['Target'] = y
#     df.dropna(subset=['Target'], inplace=True)
#     df['Target'] = df['Target'].astype(int)

# def add_targets_triple_barrier_inplace(df: pd.DataFrame, H=5, atr_mult=1.0):
#     """
#     Robust daily triple-barrier labeling.
#     - Upper/lower barriers: Close ± atr_mult * ATR(14)
#     - Horizon H (in bars) ahead.
#     - If both barriers are touched on the same bar, label is set to NaN (tie / unknown order).
#     """
#     df = df.copy()
#     atr = _atr(df, n=14)
#     up = df['Close'] + atr_mult * atr
#     dn = df['Close'] - atr_mult * atr

#     highs = df['High'].values
#     lows  = df['Low'].values
#     up_v  = up.values
#     dn_v  = dn.values

#     n = len(df)
#     tgt = np.full(n, np.nan, dtype=float)

#     for i in range(n - H):
#         hi_seq = highs[i+1:i+H+1]
#         lo_seq = lows[i+1:i+H+1]
#         up_hit_idx = np.where(hi_seq >= up_v[i])[0]
#         dn_hit_idx = np.where(lo_seq <= dn_v[i])[0]

#         if up_hit_idx.size == 0 and dn_hit_idx.size == 0:
#             continue
#         if up_hit_idx.size > 0 and (dn_hit_idx.size == 0 or up_hit_idx[0] < dn_hit_idx[0]):
#             tgt[i] = 1.0
#         elif dn_hit_idx.size > 0 and (up_hit_idx.size == 0 or dn_hit_idx[0] < up_hit_idx[0]):
#             tgt[i] = 0.0
#         else:
#             # same-bar hit: ambiguous on daily data -> skip (remain NaN)
#             pass

#     df['Target'] = tgt
#     df.dropna(subset=['Target'], inplace=True)
#     df['Target'] = df['Target'].astype(int)
#     df['FwdRet_H'] = np.log(df['Close'].shift(-H) / df['Close'])
#     # write columns back into the original frame
#     for col in ['Target', 'FwdRet_H']:
#         df_col = df[col]
#         df_col = df_col.reindex(df.index)
#         try:
#             pass
#         except Exception:
#             pass

# # ---------- NEW TARGETS (extras) ----------
# def _forward_log_return(df: pd.DataFrame, H: int) -> pd.Series:
#     return np.log(df['Close'].shift(-H) / df['Close'])

# def add_targets_risk_adjusted_sign_inplace(df: pd.DataFrame,
#                                            H_min=5, H_max=10,
#                                            risk_window=20,
#                                            eps=0.0):
#     """
#     Label by the sign of average forward Sharpe-like return over horizons H_min..H_max.
#     FwdRiskAdj = mean_H [ log(C_{t+H}/C_t) / (roll_std(LogReturn, risk_window) * sqrt(H)) ]

#     Returns a new DataFrame with ['FwdRiskAdj','Target'] added.
#     """
#     df = df.copy()
#     r = df['LogReturn'] if 'LogReturn' in df.columns else np.log(df['Close']/df['Close'].shift(1))
#     vol = r.rolling(risk_window).std()

#     sharps = []
#     for H in range(H_min, H_max + 1):
#         f = _forward_log_return(df, H)
#         sharps.append(f / (vol * np.sqrt(H)))
#     F = pd.concat(sharps, axis=1).mean(axis=1)
#     df['FwdRiskAdj'] = F
#     y = np.where(F > eps, 1, np.where(F < -eps, 0, np.nan))
#     df['Target'] = y
#     df.dropna(subset=['Target'], inplace=True)
#     df['Target'] = df['Target'].astype(int)
#     for col in ['FwdRiskAdj', 'Target']:
#         df[col] = df[col].astype(float) if col == 'FwdRiskAdj' else df[col]
#     return df

# def add_targets_excess_sign_inplace(df: pd.DataFrame, market_close: pd.Series,
#                                     H=10, eps=0.0):
#     """
#     Label by sign of forward EXCESS return vs market (e.g., SPY) over H days.
#     Returns a new DataFrame (copy) with ['FwdExcessRet','Target'].
#     """
#     df = df.copy()
#     mrk = market_close.reindex(df.index).ffill()
#     f_stock = _forward_log_return(df, H)
#     f_mkt = np.log(mrk.shift(-H) / mrk)
#     excess = f_stock - f_mkt
#     df['FwdExcessRet'] = excess
#     y = np.where(excess > eps, 1, np.where(excess < -eps, 0, np.nan))
#     df['Target'] = y
#     df.dropna(subset=['Target'], inplace=True)
#     df['Target'] = df['Target'].astype(int)
#     return df

# # ---------- Scaling ----------
# def scale_by_train(train, val, test, features):
#     scaler = RobustScaler()
#     try:
#         scaler.set_output(transform="pandas")
#     except Exception:
#         pass
#     train[features] = scaler.fit_transform(train[features])
#     val[features] = scaler.transform(val[features])
#     test[features] = scaler.transform(test[features])
#     return scaler, train, val, test

# # ---------- Dataset ----------
# class TimeSeriesDataset(Dataset):
#     def __init__(self, df, features, seq_len=30):
#         self.seq_len = seq_len
#         self.features = features
#         X_list, y_list, idx = [], [], []
#         if len(df) <= seq_len:
#             self.X = torch.empty(0, seq_len, len(features))
#             self.y = torch.empty(0, dtype=torch.long)
#             self.index = pd.DatetimeIndex([])
#             return
#         for i in range(len(df) - seq_len):
#             end = i + seq_len - 1            # last feature row (decision time t)
#             X_list.append(df[features].iloc[i:i+seq_len].values)
#             y_list.append(int(df['Target'].iloc[end]))   # label anchored at t
#             idx.append(df.index[end])                    # index = decision time t

#         self.X = torch.tensor(np.stack(X_list), dtype=torch.float32)
#         self.y = torch.tensor(np.array(y_list), dtype=torch.long)
#         self.index = pd.Index(idx)

#     def __len__(self): return len(self.y)
#     def __getitem__(self, idx): return self.X[idx], self.y[idx]

# # ---------- Datasets with regime features & cross-sectional ----------
# class TimeSeriesDatasetWithRegime(Dataset):
#     def __init__(self, df, features, regime_df, seq_len=30):
#         self.seq_len = seq_len
#         self.features = features
#         self.regime_df = regime_df
#         X_list, y_list, r_list, idx = [], [], [], []
#         if len(df) <= seq_len:
#             self.X = torch.empty(0, seq_len, len(features))
#             self.R = torch.empty(0, regime_df.shape[1])
#             self.y = torch.empty(0, dtype=torch.long)
#             self.index = pd.DatetimeIndex([])
#             return
#         for i in range(len(df) - seq_len):
#             end = i + seq_len - 1            # decision time t
#             tgt_ts = df.index[end]
#             X_list.append(df[features].iloc[i:i+seq_len].values)
#             y_list.append(int(df['Target'].iloc[end]))
#             r_list.append(regime_df.reindex([tgt_ts]).iloc[0].values)
#             idx.append(tgt_ts)
#         self.X = torch.tensor(np.stack(X_list), dtype=torch.float32)
#         self.R = torch.tensor(np.stack(r_list), dtype=torch.float32)
#         self.y = torch.tensor(np.array(y_list), dtype=torch.long)
#         self.index = pd.Index(idx)

#     def __len__(self): return len(self.y)
#     def __getitem__(self, idx): return self.X[idx], self.R[idx], self.y[idx]

# class CrossSectionalDataset(Dataset):
#     def __init__(self, per_ticker_ds: dict):
#         self.samples = []
#         for t, ds in per_ticker_ds.items():
#             for i in range(len(ds)):
#                 self.samples.append((t, ds.X[i], ds.R[i], ds.y[i], ds.index[i]))
#         if len(self.samples) == 0:
#             self.X = torch.empty(0,1,1); self.R = torch.empty(0,1); self.y = torch.empty(0, dtype=torch.long)
#             self.tickers = []; self.index = pd.DatetimeIndex([]); return
#         self.X = torch.stack([s[1] for s in self.samples])
#         self.R = torch.stack([s[2] for s in self.samples])
#         self.y = torch.stack([s[3] for s in self.samples])
#         self.tickers = [s[0] for s in self.samples]
#         self.index = pd.Index([s[4] for s in self.samples])
#     def __len__(self): return len(self.y)
#     def __getitem__(self, idx): return self.X[idx], self.R[idx], self.y[idx]

# # ---------- Mixture-of-Experts model (regime-aware head) ----------
# class PositionalEncoding(nn.Module):
#     def __init__(self, d_model, max_len=1024):
#         super().__init__()
#         pe = torch.zeros(max_len, d_model)
#         pos = torch.arange(0, max_len).unsqueeze(1).float()
#         div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
#         pe[:, 0::2] = torch.sin(pos * div)
#         pe[:, 1::2] = torch.cos(pos * div)
#         self.register_buffer('pe', pe.unsqueeze(0))
#     def forward(self, x):
#         S = x.size(1)
#         return x + self.pe[:, :S, :]

# class RegimeMoEClassifier(nn.Module):
#     def __init__(self, in_features, seq_len=30, d_model=64, nhead=4, num_layers=2,
#                  r_features=4, n_experts=3):
#         super().__init__()
#         self.seq_len = seq_len
#         self.input_proj = nn.Linear(in_features, d_model)
#         self.pos = PositionalEncoding(d_model, max_len=seq_len)
#         enc = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=128, batch_first=True)
#         self.transformer = nn.TransformerEncoder(enc, num_layers)

#         rep_dim = seq_len * d_model
#         self.backbone_head = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(rep_dim, 128),
#             nn.ReLU(),
#             nn.Dropout(0.2)
#         )
#         self.experts = nn.ModuleList([nn.Linear(128, 2) for _ in range(n_experts)])
#         self.gate = nn.Sequential(
#             nn.Linear(r_features, 16),
#             nn.ReLU(),
#             nn.Linear(16, n_experts)
#         )

#     def forward(self, x, r):
#         x = self.input_proj(x)
#         x = self.pos(x)
#         x = self.transformer(x)
#         h = self.backbone_head(x)
#         weights = torch.softmax(self.gate(r), dim=1)
#         logits_stack = torch.stack([exp(h) for exp in self.experts], dim=1)  # (B, n_exp, 2)
#         logits = (logits_stack * weights.unsqueeze(-1)).sum(dim=1)           # (B, 2)
#         return logits

# class TransformerClassifier(nn.Module):
#     def __init__(self, in_features, seq_len=30, d_model=64, nhead=4, num_layers=2):
#         super().__init__()
#         self.seq_len = seq_len
#         self.input_proj = nn.Linear(in_features, d_model)
#         self.pos = PositionalEncoding(d_model, max_len=seq_len)
#         enc = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=128, batch_first=True)
#         self.transformer = nn.TransformerEncoder(enc, num_layers)
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(seq_len*d_model, 128),
#             nn.ReLU(),
#             nn.Dropout(0.2),
#             nn.Linear(128, 2)
#         )
#     def forward(self, x):
#         x = self.input_proj(x)
#         x = self.pos(x)
#         x = self.transformer(x)
#         return self.classifier(x)

# # ---------- Loss: Focal ----------
# class FocalLoss(nn.Module):
#     def __init__(self, alpha=0.5, gamma=2.0):
#         super().__init__(); self.alpha, self.gamma = alpha, gamma
#     def forward(self, logits, targets):
#         ce = nn.functional.cross_entropy(logits, targets, reduction='none')
#         p = torch.softmax(logits, dim=1).gather(1, targets.view(-1,1)).squeeze()
#         loss = self.alpha * (1 - p).pow(self.gamma) * ce
#         return loss.mean()

# # ---------- Train / Early stopping ----------
# def train_with_early_stopping(model, Xtr, ytr, Xva, yva, epochs=150, lr=1e-4, patience=10,
#                               seed=SEED, batch_size=64, weights_tr: torch.Tensor=None,
#                               use_focal=True):
#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     model = model.to(device)
#     opt = optim.Adam(model.parameters(), lr=lr)
#     base_loss = FocalLoss() if use_focal else nn.CrossEntropyLoss(reduction='none')
#     g = torch.Generator().manual_seed(SEED)

#     if weights_tr is None:
#         weights_tr = torch.ones(len(Xtr), dtype=torch.float32)
#     ds = TensorDataset(Xtr, ytr, weights_tr)
#     tr_loader = DataLoader(ds, batch_size=batch_size, shuffle=True, generator=g)

#     best_w, best_val = None, float('inf'); bad = 0
#     for ep in range(1, epochs+1):
#         model.train(); tot = 0.0; batches = 0
#         for xb, yb, wb in tr_loader:
#             xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
#             opt.zero_grad()
#             logits = model(xb)
#             if isinstance(base_loss, FocalLoss):
#                 loss = base_loss(logits, yb)
#             else:
#                 loss_vec = base_loss(logits, yb)
#                 loss = (loss_vec * wb).mean()
#             loss.backward(); opt.step()
#             tot += loss.item(); batches += 1
#         model.eval()
#         with torch.no_grad():
#             v_logits = model(Xva.to(device))
#             vloss = (FocalLoss().to(device)(v_logits, yva.to(device))
#                      if isinstance(base_loss, FocalLoss)
#                      else nn.CrossEntropyLoss()(v_logits, yva.to(device)))
#         print(f"Epoch {ep:03d} | train_loss={tot/max(batches,1):.4f} | val_loss={vloss:.4f}")
#         if vloss < best_val:
#             best_val, bad = vloss.item() if torch.is_tensor(vloss) else float(vloss), 0
#             best_w = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
#         else:
#             bad += 1
#             if bad > patience:
#                 print(f"Early stopping at epoch {ep}")
#                 break
#     if best_w is not None:
#         model.load_state_dict(best_w)
#     return model

# # ---------- Threshold selection (precision) ----------
# def pick_threshold_for_precision(p, y, min_signals=12):
#     thrs = np.linspace(0.60, 0.90, 61)
#     best = (0.0, 0.50, 0)
#     for t in thrs:
#         yhat = (p >= t).astype(int)
#         n = int(yhat.sum())
#         if n < min_signals:
#             continue
#         prec = precision_score(y, yhat, zero_division=0)
#         if prec > best[0]:
#             best = (prec, float(t), n)
#     return best

# def pick_threshold_from_val(model, Xva, yva):
#     device = next(model.parameters()).device
#     model.eval()
#     with torch.no_grad():
#         p = torch.softmax(model(Xva.to(device)), dim=1)[:, 1].cpu().numpy()
#     thr_grid = np.linspace(0.45, 0.65, 41)
#     scores = [f1_score(yva.numpy(), (p >= t).astype(int)) for t in thr_grid]
#     t_best = float(thr_grid[int(np.argmax(scores))])
#     return t_best

# # ---------- Platt calibration ----------
# class PlattCalibrator:
#     def __init__(self):
#         self.lr = None
#     def fit(self, p_val: np.ndarray, y_val: np.ndarray):
#         y_unique = np.unique(y_val)
#         if len(y_unique) < 2:
#             self.lr = None
#             return self
#         x = np.clip(p_val.reshape(-1,1), 1e-6, 1-1e-6)
#         self.lr = LogisticRegression(solver='lbfgs')
#         self.lr.fit(x, y_val)
#         return self
#     def transform(self, p: np.ndarray) -> np.ndarray:
#         if self.lr is None:
#             return p
#         x = np.clip(p.reshape(-1,1), 1e-6, 1-1e-6)
#         return self.lr.predict_proba(x)[:,1]

# def platt_fit_apply(p_val: np.ndarray, y_val: np.ndarray, p_test: np.ndarray):
#     cal = PlattCalibrator().fit(p_val, y_val)
#     return cal.transform(p_test), cal

# # ---------- Helper: robust label -> numpy ----------
# def _as_numpy_1d(y):
#     if isinstance(y, np.ndarray):
#         arr = y
#     elif hasattr(y, "detach"):
#         arr = y.detach().cpu().numpy()
#     elif hasattr(y, "numpy"):
#         arr = y.numpy()
#     else:
#         arr = np.asarray(y)
#     return arr.astype(int).reshape(-1)

# # ---------- Eval helpers ----------
# def eval_classification(model, X, y, thr):
#     y_np = _as_numpy_1d(y)
#     device = next(model.parameters()).device
#     model.eval()
#     with torch.no_grad():
#         p = torch.softmax(model(X.to(device)), dim=1)[:, 1].cpu().numpy()
#     yhat = (p >= thr).astype(int)
#     return {
#         "acc": accuracy_score(y_np, yhat),
#         "prec": precision_score(y_np, yhat, zero_division=0),
#         "rec": recall_score(y_np, yhat, zero_division=0),
#         "f1": f1_score(y_np, yhat, zero_division=0),
#         "probs": p,
#         "yhat": yhat
#     }

# # ---------- "Today" inference ----------
# def infer_today_prob(df_full_feats: pd.DataFrame, features, scaler: RobustScaler, model, seq_len=30):
#     window = df_full_feats[features].iloc[-seq_len:].copy()
#     if len(window) < seq_len:
#         return None, None
#     window_scaled = scaler.transform(window)
#     if isinstance(window_scaled, pd.DataFrame):
#         window_scaled = window_scaled.to_numpy()
#     X_live = torch.tensor(window_scaled, dtype=torch.float32).unsqueeze(0)
#     device = next(model.parameters()).device
#     model.eval()
#     with torch.no_grad():
#         prob_up = torch.softmax(model(X_live.to(device)), dim=1)[:, 1].item()
#     return df_full_feats.index[-1], float(prob_up)

# # --- Live inference on arbitrary dates (single-ticker model) ---
# def infer_live_probs_basic(df_full_feats: pd.DataFrame, features, scaler: RobustScaler,
#                            model, seq_len: int, dates: pd.DatetimeIndex) -> pd.Series:
#     dates = pd.DatetimeIndex(dates)
#     device = next(model.parameters()).device
#     out = []
#     for d in dates:
#         if d not in df_full_feats.index:
#             out.append(np.nan); continue
#         end_loc = df_full_feats.index.get_loc(d)
#         if isinstance(end_loc, slice):
#             end_loc = end_loc.stop - 1
#         start = end_loc - (seq_len - 1)
#         if start < 0:
#             out.append(np.nan); continue
#         window = df_full_feats[features].iloc[start:end_loc+1]
#         X = scaler.transform(window)
#         if isinstance(X, pd.DataFrame):
#             X = X.to_numpy()
#         X = torch.tensor(X, dtype=torch.float32).unsqueeze(0).to(device)
#         with torch.no_grad():
#             p = torch.softmax(model(X), dim=1)[:, 1].item()
#         out.append(p)
#     return pd.Series(out, index=dates, name="prob_up").dropna()

# # --- Live inference on arbitrary dates (cross-sectional MoE model + regime) ---
# def infer_live_probs_cs(df_full_feats: pd.DataFrame, features, scaler: RobustScaler,
#                         model, regime_df: pd.DataFrame, seq_len: int,
#                         dates: pd.DatetimeIndex) -> pd.Series:
#     dates = pd.DatetimeIndex(dates)
#     device = next(model.parameters()).device
#     out = []
#     for d in dates:
#         if d not in df_full_feats.index:
#             out.append(np.nan); continue
#         end_loc = df_full_feats.index.get_loc(d)
#         if isinstance(end_loc, slice):
#             end_loc = end_loc.stop - 1
#         start = end_loc - (seq_len - 1)
#         if start < 0:
#             out.append(np.nan); continue
#         window = df_full_feats[features].iloc[start:end_loc+1]
#         X = scaler.transform(window)
#         if isinstance(X, pd.DataFrame):
#             X = X.to_numpy()
#         X = torch.tensor(X, dtype=torch.float32).unsqueeze(0).to(device)
#         r = regime_df.reindex([d]).iloc[0].values.reshape(1, -1)
#         R = torch.tensor(r, dtype=torch.float32).to(device)
#         with torch.no_grad():
#             p = torch.softmax(model(X, R), dim=1)[:, 1].item()
#         out.append(p)
#     return pd.Series(out, index=dates, name="prob_up").dropna()

# # ---------- Band (hysteresis) ----------
# def state_from_band(probs: np.ndarray, idx: pd.Index, thr_entry=0.78, thr_exit=0.48):
#     state = np.zeros_like(probs, dtype=bool)
#     for i, p in enumerate(probs):
#         if i == 0:
#             state[i] = (p >= thr_entry)
#         else:
#             state[i] = (p >= (thr_exit if state[i-1] else thr_entry))
#     return pd.Series(state, index=idx, name='state')

# def eval_classification_band(probs, y, idx, thr_entry, thr_exit):
#     y_np = _as_numpy_1d(y)
#     s = state_from_band(probs, idx, thr_entry, thr_exit)
#     yhat = s.astype(int).values
#     return {
#         "acc": accuracy_score(y_np, yhat),
#         "prec": precision_score(y_np, yhat, zero_division=0),
#         "rec": recall_score(y_np, yhat, zero_division=0),
#         "f1": f1_score(y_np, yhat, zero_division=0),
#         "probs": probs,
#         "yhat": yhat,
#         "state": s
#     }

# # ---------- Band artifacts + entry-only gating + vol gate helpers ----------
# def _safe_band_path(ticker: str, out_dir: str) -> str:
#     os.makedirs(out_dir, exist_ok=True)
#     return os.path.join(out_dir, f"{_safe_ticker_for_path(ticker)}_band.json")

# def save_band(ticker: str, best: dict, out_dir: str):
#     """
#     best = {"entry": float, "exit": float, ...}
#     """
#     path = _safe_band_path(ticker, out_dir)
#     with open(path, "w") as f:
#         json.dump(best, f, indent=2)

# def load_band(ticker: str, out_dir: str):
#     path = _safe_band_path(ticker, out_dir)
#     with open(path, "r") as f:
#         j = json.load(f)
#     return float(j["entry"]), float(j["exit"]), j

# def entry_only_gate(prev_final: bool, raw_curr: bool,
#                     trend_ok_today: bool, vol_ok_today: bool,
#                     override_used: bool) -> bool:
#     """
#     Gates checked only on entries (flat -> long).
#     """
#     is_entry = (not prev_final) and raw_curr
#     if is_entry:
#         return bool(raw_curr and ((trend_ok_today and vol_ok_today) or override_used))
#     else:
#         return bool(raw_curr)  # once in, gates don't force exit

# def apply_entry_only_gates(raw_state: pd.Series,
#                            trend_gate: pd.Series,
#                            vol_gate: pd.Series,
#                            override: pd.Series) -> pd.Series:
#     """
#     Build final state where gates can only *block new entries*.
#     """
#     idx = raw_state.index
#     trend_gate = trend_gate.reindex(idx).fillna(False)
#     vol_gate   = vol_gate.reindex(idx).fillna(False)
#     override   = override.reindex(idx).fillna(False)

#     out = []
#     prev_final = False
#     for i, d in enumerate(idx):
#         raw_curr = bool(raw_state.iloc[i])
#         if (not prev_final) and raw_curr:
#             passed = bool((trend_gate.iloc[i] and vol_gate.iloc[i]) or override.iloc[i])
#             final = raw_curr and passed
#         else:
#             final = raw_curr
#         out.append(final)
#         prev_final = final
#     return pd.Series(out, index=idx, name="state")

# def realized_vol(close: pd.Series, lookback: int = 60) -> pd.Series:
#     """Annualized rolling realized volatility from daily pct returns."""
#     ret = close.pct_change()
#     return (ret.rolling(lookback, min_periods=lookback//2).std() * np.sqrt(252.0)).rename("rv")

# def fit_vol_gate_threshold(vol_train: pd.Series, q_high: float = 0.80) -> float:
#     vol_train = vol_train.dropna()
#     return float(vol_train.quantile(q_high)) if len(vol_train) else np.inf


# # ---------- Backtest ----------
# def backtest_state(price_s: pd.Series, state: pd.Series,
#                    init_cash=10_000, fees=0.001, slippage=0.001,
#                    execution: str = "next_open"):  # "next_open" or "same_close"
#     state = state.astype(bool)
#     prev = state.shift(1, fill_value=False)
#     entries = (state & ~prev)
#     exits   = (~state & prev)

#     if execution == "next_open":
#         entries = entries.shift(1, fill_value=False)
#         exits   = exits.shift(1, fill_value=False)

#     idx = price_s.index.union(entries.index).union(exits.index)
#     price_s = price_s.reindex(idx).astype(float)
#     entries = entries.reindex(idx, fill_value=False).astype(bool)
#     exits   = exits.reindex(idx, fill_value=False).astype(bool)

#     if entries.any():
#         last_true_idx = entries.index[entries].max()
#         if not exits.any() or exits.index[exits].max() < last_true_idx:
#             exits.iloc[-1] = True

#     pf = vbt.Portfolio.from_signals(price_s, entries, exits,
#                                     init_cash=init_cash, fees=fees, slippage=slippage)

#     def _get_returns_and_curve(_pf):
#         ret_s, curve_s = None, None
#         for cand in ('returns',):
#             obj = getattr(_pf, cand, None)
#             if obj is None: continue
#             try:
#                 ret_s = obj() if callable(obj) else obj
#                 if isinstance(ret_s, (pd.Series, pd.DataFrame)):
#                     ret_s = ret_s.squeeze()
#                     break
#             except Exception:
#                 pass
#         for cand in ('value', 'asset_value', 'portfolio_value'):
#             obj = getattr(_pf, cand, None)
#             if obj is None: continue
#             try:
#                 curve_s = obj() if callable(obj) else obj
#                 if isinstance(curve_s, (pd.Series, pd.DataFrame)):
#                     curve_s = curve_s.squeeze()
#                     break
#             except Exception:
#                 pass
#         if (ret_s is None) and (curve_s is not None):
#             try:
#                 ret_s = curve_s.pct_change().dropna()
#             except Exception:
#                 pass
#         return ret_s, curve_s

#     try:
#         stats = pf.stats(freq='1D')
#     except TypeError:
#         stats = pf.stats()
#         rets, curve = _get_returns_and_curve(pf)
#         if ('Volatility [%]' not in stats.index) or pd.isna(stats.get('Volatility [%]', np.nan)):
#             if isinstance(rets, pd.Series) and len(rets) > 0:
#                 vol_pct = float(rets.std(ddof=0) * math.sqrt(252.0) * 100.0)
#                 stats.loc['Volatility [%]'] = vol_pct
#         if ('Max Drawdown [%]' not in stats.index) or pd.isna(stats.get('Max Drawdown [%]', np.nan)):
#             try:
#                 if isinstance(curve, pd.Series) and len(curve) > 1:
#                     dd = (curve / curve.cummax() - 1.0).min() * 100.0
#                     stats.loc['Max Drawdown [%]'] = float(dd)
#             except Exception:
#                 pass
#         wr = stats.get('Win Rate [%]', np.nan)
#         if pd.isna(wr):
#             try:
#                 rec = pf.trades.records_readable
#                 wins = int((rec['PnL'] > 0).sum()); total = int(len(rec))
#                 stats.loc['Win Rate [%]'] = (100.0 * wins / max(total, 1)) if total > 0 else np.nan
#             except Exception:
#                 pass

#     return pf, stats

# # ---------- Auto-tune band on validation ----------
# def pick_band_on_val(p_va, idx_va, close_series, y_va=None, min_trades=5,
#                      entry_grid=np.linspace(0.62, 0.90, 29), gap=0.25,
#                      init_cash=10_000, fees=0.001, slippage=0.001,
#                      objective="winrate"):
#     best = None
#     close_va = close_series.reindex(idx_va).dropna()
#     for e in entry_grid:
#         x = max(0.40, e - gap)
#         state = state_from_band(p_va, idx_va, thr_entry=e, thr_exit=x)
#         pf, stats = backtest_state(close_va, state, init_cash=init_cash, fees=fees, slippage=slippage)
#         trades = float(stats.get('Total Trades', 0.0))
#         print(f"Testing band: entry={e:.3f}, exit={x:.3f}, trades={trades}")
#         if trades < min_trades:
#             continue

#         if objective == "winrate":
#             wr = stats.get('Win Rate [%]', np.nan)
#             if pd.isna(wr):
#                 try:
#                     rec = pf.trades.records_readable
#                     wins = int((rec['PnL'] > 0).sum()); total = int(len(rec))
#                     wr = 100.0 * wins / max(total, 1)
#                 except Exception:
#                     wr = -np.inf
#             score = wr
#         elif objective == "precision":
#             if y_va is None:
#                 continue
#             m_va = eval_classification_band(p_va, y_va, idx_va, e, x)
#             score = 100.0 * m_va['prec']
#         else:  # return
#             score = float(stats['Total Return [%]'])
#         cand = (score, e, x, trades, stats)
#         if (best is None) or (cand[0] > best[0]):
#             best = cand
#     return best

# # ---------- Risk-aware band search with constraints ----------
# def _risk_score_from_stats(stats, lam=0.3, gamma=0.2):
#     ret = float(stats.get('Total Return [%]', 0.0))
#     vol = float(stats.get('Volatility [%]', stats.get('Volatility', 0.0)))
#     mdd = float(stats.get('Max Drawdown [%]', stats.get('Max Drawdown', 0.0)))
#     if np.isnan(ret): ret = 0.0
#     if np.isnan(vol): vol = 0.0
#     if np.isnan(mdd): mdd = 0.0
#     return ret - lam * vol - gamma * abs(mdd)

# def _rr_precheck_by_atr(df_feat: pd.DataFrame, entries_state: pd.Series,
#                         target_mult=1.5, stop_mult=1.0, atr_col='ATRp',
#                         max_fail_rate=0.5):
#     entries = entries_state[entries_state].index
#     if len(entries) < 3:
#         return False
#     atrp = df_feat[atr_col].reindex(entries).dropna()
#     if len(atrp) == 0:
#         return False
#     ok1 = (target_mult > stop_mult * 1.1)
#     ok2 = (atrp.median() < 0.06)
#     ok3 = (atrp.quantile(0.9) < 0.12)
#     fail_rate = 1.0 - float((atrp < 0.08).mean())
#     return bool(ok1 and ok2 and ok3 and (fail_rate <= max_fail_rate))

# def pick_band_on_val_riskaware(p_va, idx_va, close_series, df_feat_val,
#                                min_trades=20, enforce_entry_min=0.60,
#                                gap=0.25, lam=0.3, gamma=0.2,
#                                use_trend_gate=True, use_vol_gate=True,
#                                trend_gate_s=None, vol_gate_s=None,
#                                gate_relax_penalty=10.0,
#                                init_cash=10_000, fees=0.001, slippage=0.001):
#     best = None
#     close_va = close_series.reindex(idx_va).dropna()
#     if len(close_va) == 0:
#         return None
#     gate_penalty = 0.0
#     if not use_trend_gate or not use_vol_gate:
#         gate_penalty += gate_relax_penalty

#     for e in np.linspace(max(0.50, enforce_entry_min-0.05), 0.90, 36):
#         x = max(0.40, e - gap)
#         state = state_from_band(p_va, idx_va, thr_entry=e, thr_exit=x)

#         if e < enforce_entry_min:
#             entries = (state & ~state.shift(1, fill_value=False))
#             if not _rr_precheck_by_atr(df_feat_val, entries, target_mult=1.6, stop_mult=1.0):
#                 continue

#         gated_state = state.copy()
#         if use_trend_gate and trend_gate_s is not None:
#             gated_state &= trend_gate_s.reindex(gated_state.index).fillna(False)
#         if use_vol_gate and vol_gate_s is not None:
#             gated_state &= vol_gate_s.reindex(gated_state.index).fillna(False)

#         pf, stats = backtest_state(close_va, gated_state,
#                                    init_cash=init_cash, fees=fees, slippage=slippage,
#                                    execution="next_open")
#         trades = float(stats.get('Total Trades', 0.0))
#         if trades < min_trades:
#             continue
#         score = _risk_score_from_stats(stats, lam=lam, gamma=gamma) - gate_penalty
#         cand = (score, e, x, trades, stats)
#         print(f"[riskaware] entry={e:.3f} exit={x:.3f} trades={trades:.0f} score={score:.2f}")
#         if (best is None) or (cand[0] > best[0]):
#             best = cand
#     return best

# # ---------- Baseline: MA momentum + ATR-style exit ----------
# def baseline_ma_atr_signals(df_feat: pd.DataFrame,
#                             fast=50, slow=200,
#                             atr_mult_exit=1.5):
#     ma_fast = df_feat['MA50']
#     ma_slow = df_feat['MA200']
#     atr = df_feat['ATRp'] * df_feat['Close']
#     momentum_state = (ma_fast > ma_slow) & (ma_slow.pct_change(20) > 0)

#     exit_line = ma_fast - atr_mult_exit * (atr.fillna(0.0))
#     early_exit = df_feat['Close'] < exit_line

#     st = pd.Series(False, index=df_feat.index)
#     prev = False
#     for i in range(len(df_feat)):
#         if not prev and momentum_state.iloc[i]:
#             prev = True
#             st.iloc[i] = True
#         elif prev:
#             if (not momentum_state.iloc[i]) or early_exit.iloc[i]:
#                 prev = False
#                 st.iloc[i] = False
#             else:
#                 st.iloc[i] = True
#     return st.rename('baseline_state')

# # ---------- Pretty helpers for per-ticker reporting ----------
# def _print_prob_stats(name, probs):
#     if probs is None or len(probs) == 0:
#         print(f"[{name}] No probabilities to summarize.")
#         return
#     print(f"[{name}] prob_up range: min={np.min(probs):.3f} | max={np.max(probs):.3f} | mean={np.mean(probs):.3f}")

# def _print_signal_summary(ticker, state_series: pd.Series):
#     if state_series is None or len(state_series) == 0:
#         print(f"[{ticker}] No state to summarize.")
#         return
#     entries = (state_series & ~state_series.shift(1, fill_value=False))
#     exits   = (~state_series & state_series.shift(1, fill_value=False))
#     long_ratio = float(state_series.mean())
#     print(f"[{ticker}] signals: entries={int(entries.sum())}, exits={int(exits.sum())}, long-day%={100*long_ratio:.1f}%")

# def _print_bt_subset(ticker, stats):
#     fields = [
#         'Start', 'End', 'Period', 'Start Value', 'End Value',
#         'Total Return [%]', 'Max Drawdown [%]', 'Volatility [%]',
#         'Total Trades', 'Win Rate [%]', 'Profit Factor', 'Expectancy'
#     ]
#     print(f"\n--- {ticker} backtest summary (freq=1D) ---")
#     for f in fields:
#         if f in stats.index:
#             print(f"{f:>20}: {stats[f]}")

# # ---------- NEW: After-close decision helpers ----------
# def _next_business_day(d: pd.Timestamp) -> pd.Timestamp:
#     try:
#         return (pd.Timestamp(d) + BDay(1)).normalize()
#     except Exception:
#         return pd.Timestamp(d) + pd.Timedelta(days=1)

# def _format_after_close_message(ticker: str,
#                                 live_date: pd.Timestamp,
#                                 prev_final: bool,
#                                 curr_final: bool,
#                                 curr_raw: bool,
#                                 prob_today: float,
#                                 thr_entry: float,
#                                 thr_exit: float,
#                                 trend_ok_today: bool,
#                                 vol_ok_today: bool,
#                                 topn_kept_today: bool,
#                                 override_used: bool,
#                                 live_close: float) -> str:
#     """Return a single-line instruction tagged with the latest close date and close price."""
#     date_str = pd.Timestamp(live_date).date()
#     px_txt = f" | close={live_close:,.2f}"

#     # Transition flat -> long: enter at the NEXT session's open
#     if (not prev_final) and curr_final:
#         return (f"Enter Next Opening ({ticker}) ({date_str})  | prob_up={prob_today:.2%} | "
#                 f"band≥{thr_entry:.3f} | gates trend={'OK' if trend_ok_today else 'X'}, "
#                 f"vol={'OK' if vol_ok_today else 'X'}{px_txt}")

#     # Stay long
#     if prev_final and curr_final:
#         return f"Hold Long Position ({ticker}) ({date_str})  | prob_up={prob_today:.2%}{px_txt}"

#     # Transition long -> flat: exit at the NEXT session's open
#     if prev_final and (not curr_final):
#         reason = "band exit" if (not curr_raw) else "gate exit"
#         return (f"Exit Next Opening ({ticker}) ({date_str})  | {reason}, "
#                 f"prob_up={prob_today:.2%}, exit<{thr_exit:.3f}{px_txt}")

#     # Band passed but entry-only gates blocked the new position
#     if curr_raw and (not curr_final):
#         cause = []
#         if not trend_ok_today: cause.append("trend gate")
#         if not vol_ok_today:   cause.append("vol gate")
#         if not topn_kept_today:cause.append("top-N cap")
#         if override_used:      cause = []  # override bypasses gates
#         cause_txt = " & ".join(cause) if cause else "policy filter"
#         return (f"Flat: Band passed but {cause_txt} blocked ({ticker}) ({date_str})  | "
#                 f"prob_up={prob_today:.2%}{px_txt}")

#     # Nothing to do
#     return (f"Flat: No long (below entry band) ({ticker}) ({date_str})  | "
#             f"prob_up={prob_today:.2%}, entry≥{thr_entry:.3f}{px_txt}")



# # ===========================
# # PART 2 / 2  (FIXED & MERGED)
# # ===========================

# # --- HOTFIX: ensure triple-barrier target truly mutates the passed DataFrame in place ---
# def add_targets_triple_barrier_inplace(df: pd.DataFrame, H=5, atr_mult=1.0):
#     """
#     Robust daily triple-barrier labeling (in-place).
#     - Upper/lower barriers: Close ± atr_mult * ATR(14)
#     - Horizon H (in bars) ahead.
#     - If both barriers are touched on the same bar, label is set to NaN (tie / unknown order).
#     """
#     atr = _atr(df, n=14)
#     up = df['Close'] + atr_mult * atr
#     dn = df['Close'] - atr_mult * atr

#     highs = df['High'].to_numpy()
#     lows  = df['Low'].to_numpy()
#     up_v  = up.to_numpy()
#     dn_v  = dn.to_numpy()

#     n = len(df)
#     tgt = np.full(n, np.nan, dtype=float)

#     for i in range(n - H):
#         hi_seq = highs[i+1:i+H+1]
#         lo_seq = lows[i+1:i+H+1]
#         up_hit_idx = np.where(hi_seq >= up_v[i])[0]
#         dn_hit_idx = np.where(lo_seq <= dn_v[i])[0]

#         if up_hit_idx.size == 0 and dn_hit_idx.size == 0:
#             continue
#         if up_hit_idx.size > 0 and (dn_hit_idx.size == 0 or up_hit_idx[0] < dn_hit_idx[0]):
#             tgt[i] = 1.0
#         elif dn_hit_idx.size > 0 and (up_hit_idx.size == 0 or dn_hit_idx[0] < up_hit_idx[0]):
#             tgt[i] = 0.0
#         else:
#             pass

#     df['Target'] = tgt
#     df.dropna(subset=['Target'], inplace=True)
#     df['Target'] = df['Target'].astype(int)
#     df['FwdRet_H'] = np.log(df['Close'].shift(-H) / df['Close'])

# # ---------- Market context download (SPY & VIX) ----------
# def _download_market(period="15y"):
#     # Use ADJUSTED OHLC for SPY to avoid split/dividend artifacts
#     spy = yf.download(
#         "SPY", period=period, interval="1d",
#         auto_adjust=True, prepost=False, actions=False,
#         threads=False, progress=False
#     )
#     if isinstance(spy.columns, pd.MultiIndex):
#         spy.columns = [c[0] for c in spy.columns]
#     spy = _normalize_daily_index_us(spy)
#     spy_close = spy["Close"].astype(float).rename("SPY_Close").ffill().bfill()

#     # VIX doesn't need adjustment; keep as-is
#     vix = yf.download(
#         "^VIX", period=period, interval="1d",
#         auto_adjust=False, prepost=False, actions=False,
#         threads=False, progress=False
#     )
#     if isinstance(vix.columns, pd.MultiIndex):
#         vix.columns = [c[0] for c in vix.columns]
#     vix = _normalize_daily_index_us(vix)
#     vix_close = vix["Close"].astype(float).rename("VIX_Close").ffill().bfill()

#     return spy_close, vix_close


# # ---------- Regime features ----------
# def compute_regime_features_by_date(per_ticker_df: dict, spy_close: pd.Series, vix_close: pd.Series):
#     """
#     Build regime features on the union of dates across all tickers (needs Close, MA50 in per_ticker_df).
#     """
#     all_idx = None
#     for df in per_ticker_df.values():
#         all_idx = df.index if all_idx is None else all_idx.union(df.index)
#     all_idx = pd.DatetimeIndex(sorted(all_idx))

#     spy = spy_close.reindex(all_idx).ffill()
#     vix = vix_close.reindex(all_idx).ffill()

#     spy_ma200 = spy.rolling(200).mean()
#     trend = ((spy > spy_ma200) & (spy_ma200.pct_change(20) > 0)).astype(float)

#     vix_z = (vix - vix.rolling(60).mean()) / (vix.rolling(60).std() + 1e-9)

#     # Breadth: % of tickers above their MA50
#     above_ma50 = []
#     for t, df in per_ticker_df.items():
#         ma50 = df['MA50'].reindex(all_idx)
#         close = df['Close'].reindex(all_idx)
#         above_ma50.append((close > ma50).astype(float))
#     breadth = (pd.concat(above_ma50, axis=1).mean(axis=1)
#                if len(above_ma50) > 0 else pd.Series(0.5, index=all_idx))

#     mkt_ret20 = np.log(spy / spy.shift(20))

#     reg = pd.DataFrame({
#         "REG_trend_up": trend,
#         "REG_vix_z": vix_z.clip(-4, 4),
#         "REG_breadth": breadth,
#         "REG_mkt_ret20": mkt_ret20.fillna(0.0)
#     }, index=all_idx).ffill().fillna(0.0)
#     return reg

# # ---------- Cross-sectional quantile calibration & top-N ----------
# def cs_quantile_calibrate(dates: pd.Index, tickers: list, probs: np.ndarray, blend=0.5):
#     df = pd.DataFrame({"date": dates, "ticker": tickers, "p": probs})
#     out = np.zeros_like(probs, dtype=float)
#     for d, grp in df.groupby("date"):
#         n = len(grp)
#         if n == 0:
#             continue
#         ranks = grp['p'].rank(method='average')  # 1..n
#         q = (ranks - 0.5) / n
#         raw = grp['p'].values
#         cal = blend * q.values + (1.0 - blend) * raw
#         out[grp.index.values] = cal
#     return out

# def top_n_mask_by_date(dates: pd.Index, probs: np.ndarray, N: int):
#     df = pd.DataFrame({"date": dates, "p": probs})
#     keep = np.zeros_like(probs, dtype=bool)
#     for d, grp in df.groupby('date'):
#         if N <= 0:
#             continue
#         idx_sorted = grp['p'].sort_values(ascending=False).index[:N]
#         keep[idx_sorted.values] = True
#     return keep

# # ---------- Universe training & calibration ----------
# def run_universe_pipeline(tickers: list,
#                           seq_len=30,
#                           epochs=120,
#                           patience=8,
#                           data_period="10y",
#                           split_mode="by_date",
#                           val_years=1, test_years=2,
#                           target_mode="riskadj_sign",
#                           H=10, eps=0.0,
#                           H_min=5, H_max=10,
#                           use_band=True,
#                           thr_entry=0.65,
#                           thr_exit=0.40,
#                           auto_band_from_val=True,
#                           band_objective="riskaware",   # 'riskaware', 'precision', 'winrate', 'return'
#                           min_trades_val=20,
#                           lam=0.3, gamma=0.2,
#                           use_trend_gate=True,
#                           use_vol_gate=True,
#                           atrp_min=0.004, atrp_max=0.10,
#                           use_focal_loss=True,
#                           weight_by_future_abs=True,
#                           evaluate_baseline=True,
#                           override_gate_prob=None,
#                           top_n_per_date: int = 0,
#                           init_cash=10_000, fees=0.001, slippage=0.001):
#     print("\n=== Cross-Sectional Pipeline ===")
#     mkt_period = "max" if str(data_period).lower() == "max" else (
#         data_period if (isinstance(data_period, str) and data_period.endswith("y")
#                         and int(re.sub(r'[^0-9]', '', data_period) or 0) >= 15) else "15y")
#     spy_close, vix_close = _download_market(period=mkt_period)

#     # 1) Build per-ticker **full features** and **labeled** frames
#     per_full = {}   # features only (for regime/gates & latest day)
#     per = {}        # labeled frames (for training)
#     for t in tickers:
#         try:
#             df_raw = yf.download(t, period=data_period, interval="1d",
#                                  auto_adjust=True, prepost=False, actions=False,
#                                  threads=False, progress=False)
#             if isinstance(df_raw.columns, pd.MultiIndex):
#                 df_raw.columns = [c[0] for c in df_raw.columns]
#             df_raw = df_raw[['Open','High','Low','Close','Volume']].ffill().bfill()
#             df_raw = _normalize_daily_index_us(df_raw)
#             if len(df_raw) < 300:
#                 print(f"{t}: not enough history"); continue
#             df_feat = compute_features(df_raw)
#             per_full[t] = df_feat.copy()

#             # Label targets on a copy
#             if target_mode == "riskadj_sign":
#                 df_lab = add_targets_risk_adjusted_sign_inplace(df_feat, H_min=H_min, H_max=H_max, eps=eps)
#             elif target_mode == "excess_sign":
#                 df_lab = add_targets_excess_sign_inplace(df_feat, market_close=spy_close, H=H, eps=eps)
#             elif target_mode == "horizon":
#                 df_lab = df_feat.copy()
#                 add_targets_horizon_inplace(df_lab, H=max(H_min,5), eps=eps)
#             else:
#                 df_lab = df_feat.copy()
#                 add_targets_triple_barrier_inplace(df_lab, H=max(H_min,5), atr_mult=1.0)

#             per[t] = df_lab
#         except Exception as e:
#             print(f"{t}: download/feature error -> {e}")

#     if len(per) == 0:
#         raise RuntimeError("No usable tickers for cross-sectional training.")

#     # 2) Regime features on union of dates
#     regime_df = compute_regime_features_by_date(per_full, spy_close, vix_close)

#     # 3) Split each name
#     splits = {}
#     for t, df in per.items():
#         try:
#             if split_mode == "by_date":
#                 tr, va, te = time_splits_by_date(df, test_years=test_years, val_years=val_years, purge_days=max(2, H_max))
#             else:
#                 tr, va, te = time_splits(df, purge_days=max(2, H_max))
#             splits[t] = (tr, va, te)
#         except Exception as e:
#             print(f"{t}: split error -> {e}")

#     # 4) Features list
#     features = [
#         'LogReturn','RSI','MACD','SignalLine','BB_upper','BB_lower',
#         'r1','r5','r10','ATRp','pctB','distMA50'
#     ]

#     # 5) Fit scaler on stacked TRAIN across names, then transform each split
#     train_stack = pd.concat([splits[t][0][features] for t in splits if len(splits[t][0])>0], axis=0)
#     scaler = RobustScaler()
#     try:
#         scaler.set_output(transform="pandas")
#     except Exception:
#         pass
#     scaler.fit(train_stack)

#     per_scaled = {}
#     for t, (tr, va, te) in splits.items():
#         tr_scaled = tr.copy(); va_scaled = va.copy(); te_scaled = te.copy()
#         tr_scaled[features] = scaler.transform(tr_scaled[features])
#         va_scaled[features] = scaler.transform(va_scaled[features])
#         te_scaled[features] = scaler.transform(te_scaled[features])
#         per_scaled[t] = (tr_scaled, va_scaled, te_scaled)

#     # 6) Build per-name datasets with regime inputs
#     per_ds = {}
#     for t, (tr, va, te) in per_scaled.items():
#         dtr = TimeSeriesDatasetWithRegime(tr, features, regime_df, seq_len)
#         dva = TimeSeriesDatasetWithRegime(va, features, regime_df, seq_len)
#         dte = TimeSeriesDatasetWithRegime(te, features, regime_df, seq_len)
#         if len(dtr)==0 or len(dva)==0 or len(dte)==0:
#             continue
#         per_ds[t] = {"train": dtr, "val": dva, "test": dte}

#     if len(per_ds) == 0:
#         raise RuntimeError("Datasets empty after seq_len.")

#     # 7) Assemble cross-sectional datasets
#     dtr_xs = CrossSectionalDataset({t: per_ds[t]["train"] for t in per_ds})
#     dva_xs = CrossSectionalDataset({t: per_ds[t]["val"] for t in per_ds})
#     dte_xs = CrossSectionalDataset({t: per_ds[t]["test"] for t in per_ds})

#     device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     model = RegimeMoEClassifier(in_features=len(features), seq_len=seq_len,
#                                 r_features=4, n_experts=3).to(device)

#     # 8) Train
#     def _train_loop(model, dtr, dva, epochs, patience, use_focal):
#         opt = optim.Adam(model.parameters(), lr=1e-4)
#         base_loss = FocalLoss() if use_focal else nn.CrossEntropyLoss()
#         g = torch.Generator().manual_seed(SEED)
#         tr_loader = DataLoader(TensorDataset(dtr.X, dtr.R, dtr.y), batch_size=64, shuffle=True, generator=g)
#         best_w, best_val = None, float('inf'); bad = 0
#         for ep in range(1, epochs+1):
#             model.train(); tot=0.0; batches=0
#             for xb, rb, yb in tr_loader:
#                 xb, rb, yb = xb.to(device), rb.to(device), yb.to(device)
#                 opt.zero_grad()
#                 logits = model(xb, rb)
#                 loss = base_loss(logits, yb)
#                 loss.backward(); opt.step()
#                 tot += loss.item(); batches += 1
#             model.eval()
#             with torch.no_grad():
#                 v_logits = model(dva.X.to(device), dva.R.to(device))
#                 vloss = base_loss(v_logits, dva.y.to(device))
#             print(f"[XS] Epoch {ep:03d} | train_loss={tot/max(batches,1):.4f} | val_loss={vloss:.4f}")
#             if vloss < best_val:
#                 best_val, bad = float(vloss), 0
#                 best_w = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
#             else:
#                 bad += 1
#                 if bad > patience:
#                     print(f"[XS] Early stopping at epoch {ep}")
#                     break
#         if best_w is not None:
#             model.load_state_dict(best_w)
#         return model

#     model = _train_loop(model, dtr_xs, dva_xs, epochs, patience, use_focal_loss)

#     # 9) Infer probs and per-date calibration
#     def _infer_probs(model, ds):
#         model.eval()
#         with torch.no_grad():
#             p = torch.softmax(model(ds.X.to(device), ds.R.to(device)), dim=1)[:,1].cpu().numpy()
#         return p

#     p_tr = _infer_probs(model, dtr_xs)
#     p_va = _infer_probs(model, dva_xs)
#     p_te = _infer_probs(model, dte_xs)

#     cal_va = cs_quantile_calibrate(dva_xs.index, dva_xs.tickers, p_va, blend=0.5)
#     cal_te = cs_quantile_calibrate(dte_xs.index, dte_xs.tickers, p_te, blend=0.5)

#     keep_mask_te = np.ones_like(cal_te, dtype=bool)
#     if top_n_per_date and top_n_per_date > 0:
#         keep_mask_te = top_n_mask_by_date(dte_xs.index, cal_te, top_n_per_date)

#     # --- Collect after-close messages here (per-ticker latest close) ---
#     after_close_msgs = []

#     # 10) Per-name evaluation, band selection, baseline comparison + reporting
#     per_results = {}
#     report_date = latest_us_trading_date()
#     for t, d in per_ds.items():
#         dva = d["val"]; dte = d["test"]

#         mask_va = (np.array(dva_xs.tickers) == t)
#         mask_te = (np.array(dte_xs.tickers) == t)
#         pva = cal_va[mask_va]
#         pte_t = cal_te[mask_te]
#         keep_ticker = keep_mask_te[mask_te]

#         df_full = per_full[t]     # FULL features (includes yesterday)
#         df_labeled = per[t]       # labeled (ends earlier)

#         # gates computed on FULL frame so they exist on yesterday too
#         trend_ok_full = (df_full['Close'] > df_full['MA200']) & (df_full['MA200'].pct_change(20) > 0)

#         # Quantile-based realized vol gate: fit on TRAIN, apply everywhere
#         rv_full  = realized_vol(df_full['Close'], lookback=60)
#         rv_train = rv_full.reindex(splits[t][0].index).dropna()  # TRAIN index for this ticker
#         vol_thr_t = fit_vol_gate_threshold(rv_train, q_high=0.80)
#         vol_ok_full = (rv_full <= vol_thr_t)

#         trend_gate_va = trend_ok_full.reindex(dva.index).fillna(False)
#         vol_gate_va   = vol_ok_full.reindex(dva.index).fillna(False)

#         # Band search
#         band_dir = os.path.join("artifacts", "bands")
#         thr_entry_t, thr_exit_t = thr_entry, thr_exit  # defaults

#         # Try to load frozen band first
#         try:
#             loaded_entry, loaded_exit, _meta = load_band(t, band_dir)
#             thr_entry_t, thr_exit_t = loaded_entry, loaded_exit
#             print(f"[{t}] Using FROZEN band from disk: entry={thr_entry_t:.3f} exit={thr_exit_t:.3f}")
#             auto_band_from_val = False  # prevent retuning on TEST
#         except Exception:
#             pass

#         if auto_band_from_val:
#             if band_objective == "riskaware":
#                 picked = pick_band_on_val_riskaware(
#                     p_va=pva, idx_va=dva.index, close_series=df_full['Open'],
#                     df_feat_val=df_full, min_trades=min_trades_val,
#                     enforce_entry_min=max(0.60, thr_entry),
#                     gap=max(0.20, thr_entry - thr_exit),
#                     lam=lam, gamma=gamma,
#                     use_trend_gate=use_trend_gate, use_vol_gate=use_vol_gate,
#                     trend_gate_s=trend_gate_va, vol_gate_s=vol_gate_va,
#                     gate_relax_penalty=10.0,
#                     init_cash=init_cash, fees=fees, slippage=slippage
#                 )
#                 if picked is not None:
#                     score, thr_entry_t, thr_exit_t, trades, _ = picked
#                     print(f"[{t}] picked band (riskaware): entry={thr_entry_t:.3f} exit={thr_exit_t:.3f} trades={int(trades)} score={score:.2f}")
#             else:
#                 picked = pick_band_on_val(
#                     p_va=pva, idx_va=dva.index, close_series=df_full['Open'],
#                     y_va=dva.y.numpy(), min_trades=min_trades_val,
#                     entry_grid=np.linspace(max(0.52, thr_entry-0.10), 0.90, 41),
#                     gap=max(0.20, thr_entry - thr_exit),
#                     init_cash=10_000, fees=fees, slippage=slippage,
#                     objective=band_objective
#                 )
#                 if picked is not None:
#                     score, thr_entry_t, thr_exit_t, trades, _ = picked
#                     print(f"[{t}] picked band ({band_objective}): entry={thr_entry_t:.3f} exit={thr_exit_t:.3f} trades={int(trades)} score={score:.2f}")

#         # Persist the chosen band (frozen from VAL) for paper/live
#         try:
#             save_band(t, {"entry": float(thr_entry_t), "exit": float(thr_exit_t)}, band_dir)
#         except Exception as e:
#             print(f"[{t}] Warning: couldn't save band -> {e}")

#         # Build TEST state with gates (TEST dates live on labeled frame indices)
#         s_test = state_from_band(pte_t, dte.index, thr_entry=thr_entry_t, thr_exit=thr_exit_t)

#         override = pd.Series(False, index=s_test.index)
#         if override_gate_prob is not None:
#             p_series = pd.Series(pte_t, index=dte.index)
#             override = p_series >= override_gate_prob

#         trend_gate = trend_ok_full.reindex(s_test.index).fillna(False) if use_trend_gate else pd.Series(True, index=s_test.index)
#         vol_gate   = vol_ok_full.reindex(s_test.index).fillna(False)   if use_vol_gate   else pd.Series(True, index=s_test.index)

#         # Entry-only gating on TEST
#         gated_state_test = apply_entry_only_gates(s_test, trend_gate, vol_gate, override)
#         if top_n_per_date and top_n_per_date > 0:
#             mask_series = pd.Series(keep_ticker, index=dte.index)
#             gated_state_test &= mask_series
#         gated_state_test = gated_state_test.rename('state')

#         price_test = df_full['Open'].reindex(gated_state_test.index).astype(float)
#         pf_m, st_m = backtest_state(price_test, gated_state_test,
#                                     init_cash=init_cash, fees=fees, slippage=slippage,
#                                     execution="next_open")

#         # Baseline
#         pf_b, st_b = None, None
#         if evaluate_baseline:
#             base_state = baseline_ma_atr_signals(df_full)
#             base_state = base_state.reindex(gated_state_test.index).fillna(False)
#             pf_b, st_b = backtest_state(price_test, base_state,
#                                         init_cash=init_cash, fees=fees, slippage=slippage,
#                                         execution="next_open")

#         # Reporting
#         _print_prob_stats(t, pte_t)
#         print(f"[{t}] band used: entry={thr_entry_t:.3f} | exit={thr_exit_t:.3f}")
#         print(f"[{t}] gates pass rates -> trend={trend_gate.mean():.3f} | vol={vol_gate.mean():.3f}")
#         _print_signal_summary(t, gated_state_test)
#         _print_bt_subset(t, st_m)
#         if evaluate_baseline and st_b is not None:
#             _print_bt_subset(f"{t} (baseline MA+ATR)", st_b)

#         per_results[t] = {
#             "band": (thr_entry_t, thr_exit_t),
#             "model_pf": pf_m, "model_stats": st_m,
#             "baseline_pf": pf_b, "baseline_stats": st_b
#         }

#         # --- AFTER-CLOSE: evaluate strictly at yesterday's US/Eastern trading day ---
#         eval_date = report_date
#         if eval_date not in df_full.index:
#             try:
#                 eval_date = df_full.index[df_full.index.get_loc(report_date, method='pad')]
#             except Exception:
#                 eval_date = df_full.index.max()

#         live_close = float(df_full['Close'].loc[eval_date])

#         if eval_date in dte.index:
#             loc = dte.index.get_loc(eval_date)
        
#             # --- LIVE recompute even inside TEST ---
#             p_live = infer_live_probs_cs(
#                 df_full, features, scaler, model, regime_df, seq_len,
#                 pd.DatetimeIndex([eval_date])
#             )
#             prob_today = float(p_live.iloc[0])
#             # --- END LIVE recompute ---
        
#             raw_curr   = bool(s_test.iloc[loc])
#             curr_final = bool(gated_state_test.iloc[loc])
#             prev_final = bool(gated_state_test.shift(1, fill_value=False).iloc[loc])
#             trend_ok_today = bool(trend_ok_full.reindex([eval_date]).fillna(False).iloc[0])
#             vol_ok_today   = bool(vol_ok_full.reindex([eval_date]).fillna(False).iloc[0])
#             override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
#             topn_today     = True
#             if top_n_per_date and top_n_per_date > 0:
#                 topn_today = bool(pd.Series(keep_ticker, index=dte.index).iloc[loc])

#         else:
#             # live inference for yesterday (beyond TEST end)
#             p_tail = infer_live_probs_cs(df_full, features, scaler, model, regime_df, seq_len,
#                                          pd.DatetimeIndex([eval_date]))
#             prob_today = float(p_tail.iloc[0])
#             raw_prev = bool(s_test.iloc[-1]) if len(s_test) else False
#             raw_curr = bool(prob_today >= (thr_exit_t if raw_prev else thr_entry_t))
#             trend_ok_today = bool(trend_ok_full.reindex([eval_date]).fillna(False).iloc[0])
#             vol_ok_today   = bool(vol_ok_full.reindex([eval_date]).fillna(False).iloc[0])
#             override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
#             prev_final = bool(gated_state_test.iloc[-1]) if len(gated_state_test) else False
#             curr_final = entry_only_gate(prev_final, raw_curr, trend_ok_today, vol_ok_today, override_used)
#             topn_today = True  # can't compute cross-sec cap for tail

#         msg = _format_after_close_message(
#             ticker=t,
#             live_date=eval_date,        # this is yesterday (or last <= yesterday)
#             prev_final=prev_final,
#             curr_final=curr_final,
#             curr_raw=raw_curr,
#             prob_today=prob_today,
#             thr_entry=thr_entry_t,
#             thr_exit=thr_exit_t,
#             trend_ok_today=trend_ok_today,
#             vol_ok_today=vol_ok_today,
#             topn_kept_today=bool(topn_today),
#             override_used=override_used,
#             live_close=live_close
#         )
#         after_close_msgs.append(msg)

#     # 11) Aggregate & print + consolidated latest-close sheet
#     total_start, total_end, ok_count = 0.0, 0.0, 0
#     beats = 0
#     for t, res in per_results.items():
#         st_m = res["model_stats"]
#         sv = float(st_m.get("Start Value", np.nan))
#         ev = float(st_m.get("End Value", np.nan))
#         if not (np.isnan(sv) or np.isnan(ev)):
#             total_start += sv; total_end += ev; ok_count += 1
#         if evaluate_baseline and res["baseline_stats"] is not None:
#             mret = float(st_m.get("Total Return [%]", 0.0))
#             bret = float(res["baseline_stats"].get("Total Return [%]", 0.0))
#             if mret > max(bret, 0.0):
#                 beats += 1
#         print(f"[{t}] model_ret={st_m.get('Total Return [%]', np.nan):.2f}% | "
#               f"vol={st_m.get('Volatility [%]', np.nan):.2f}% | "
#               f"mdd={st_m.get('Max Drawdown [%]', np.nan):.2f}% | "
#               f"baseline_ret={(res['baseline_stats'].get('Total Return [%]', np.nan) if res['baseline_stats'] is not None else np.nan):.2f}%")

#     if ok_count > 0:
#         pnl = total_end - total_start
#         ret_pct = (total_end / total_start - 1.0) * 100.0 if total_start > 0 else float("nan")
#         print("\n=== Cross-Sectional Aggregate summary ===")
#         print(f"Tickers processed: {ok_count} | Beat baseline on {beats}")
#         print(f"Total Start Value: ${total_start:,.2f}")
#         print(f"Total End Value:   ${total_end:,.2f}")
#         print(f"Total PnL:         ${pnl:,.2f}  ({ret_pct:,.2f}%)")

#     if len(after_close_msgs) > 0:
#         print(f"\n=== After-Close Signals — {report_date.date()} (US/Eastern) ===")
#         for line in after_close_msgs:
#             print(line)

#     return {"model": model, "scaler": scaler, "per_results": per_results, "regime_df": regime_df}

# # ---------- HOTFIX: redefine compute_features to fix a minor type() check typo ----------
# def compute_features(df: pd.DataFrame) -> pd.DataFrame:
#     df = df.copy().ffill().bfill()
#     df = df[['Open', 'High', 'Low', 'Close', 'Volume']].astype(float)

#     delta = df['Close'].diff()
#     gain = delta.where(delta > 0, 0).rolling(14).mean()
#     loss = -delta.where(delta < 0, 0).rolling(14).mean()
#     rs = gain / loss.where(loss != 0, np.nan)
#     df['RSI'] = 100 - (100 / (1 + rs))

#     ema12 = df['Close'].ewm(span=12, adjust=False).mean()
#     ema26 = df['Close'].ewm(span=26, adjust=False).mean()
#     df['MACD'] = ema12 - ema26
#     df['SignalLine'] = df['MACD'].ewm(span=9, adjust=False).mean()

#     ma20 = df['Close'].rolling(20).mean()
#     std20 = df['Close'].rolling(20).std()
#     df['BB_upper'] = ma20 + 2 * std20
#     df['BB_lower'] = ma20 - 2 * std20

#     df['LogReturn'] = np.log(df['Close'] / df['Close'].shift(1))
#     df['r1'] = df['Close'].pct_change(1)
#     df['r5'] = df['Close'].pct_change(5)
#     df['r10'] = df['Close'].pct_change(10)

#     atr = _atr(df, n=14)
#     df['ATRp'] = atr / df['Close']

#     rng = df['BB_upper'] - df['BB_lower']
#     numerator = df['Close'] - df['BB_lower']
#     df['pctB'] = numerator / rng.where(rng != 0, np.nan)

#     df['MA50'] = df['Close'].rolling(50).mean()
#     df['distMA50'] = df['Close'] / df['MA50'] - 1.0
#     df['MA200'] = df['Close'].rolling(200).mean()

#     for col in ['RSI', 'MACD', 'SignalLine', 'BB_upper', 'BB_lower', 'LogReturn',
#                 'r1', 'r5', 'r10', 'ATRp', 'pctB', 'MA50', 'distMA50', 'MA200']:
#         if isinstance(df[col], pd.DataFrame):
#             raise ValueError(f"Column {col} is a DataFrame with shape {df[col].shape}, expected Series")
#         elif not isinstance(df[col], pd.Series):
#             raise ValueError(f"Column {col} is not a Series, got {type(df[col])}")

#     df.dropna(inplace=True)
#     return df


# # ---------- Main pipeline (single ticker) ----------
# def run_ticker_pipeline(ticker: str,
#                         seq_len=30,
#                         epochs=150,
#                         patience=10,
#                         purge_days=2,
#                         init_cash=10_000,
#                         fees=0.001,
#                         slippage=0.001,
#                         data_period="10y",
#                         split_mode="by_date",
#                         val_years=1,
#                         test_years=2,
#                         target_mode="horizon",
#                         H=3, eps=0.003,
#                         tb_H=5, tb_atr_mult=1.0,
#                         use_band=True,
#                         thr_entry=0.65,
#                         thr_exit=0.40,
#                         auto_band_from_val=False,
#                         band_objective="winrate",
#                         min_trades_val=5,
#                         use_trend_gate=True,
#                         use_vol_gate=True,
#                         atrp_min=0.004, atrp_max=0.10,
#                         use_focal_loss=True,
#                         weight_by_future_abs=True,
#                         device_str=None,
#                         # --- NEW OPTIONS ---
#                         risk_H_min=5, risk_H_max=10,
#                         evaluate_baseline=False,
#                         lam=0.3, gamma=0.2,
#                         calibrate_platt=False,
#                         override_gate_prob=None):
#     print(f"\n=== {ticker} ===")

#     purge_days = max(purge_days, H, risk_H_max)

#     # 1) Download + features (normalized to US/Eastern calendar dates)
#     df_raw = yf.download(
#         ticker, period=data_period, interval="1d",
#         auto_adjust=True, prepost=False, actions=False,
#         threads=False, progress=False
#     )
#     print(f"Raw data shape: {df_raw.shape}, columns: {df_raw.columns}")
#     print(f"Raw data date range: {df_raw.index.min()} to {df_raw.index.max()}")
#     if isinstance(df_raw.columns, pd.MultiIndex):
#         df_raw.columns = [col[0] for col in df_raw.columns]
#     df_raw = df_raw[['Open', 'High', 'Low', 'Close', 'Volume']].ffill().bfill()
#     df_raw = _normalize_daily_index_us(df_raw)  # normalize to US/Eastern dates
#     if df_raw.empty or len(df_raw) < 300:
#         raise ValueError(f"Not enough data for {ticker}")
#     df_feat = compute_features(df_raw)

#     # 2) Splits
#     if split_mode == "by_date":
#         train, val, test = time_splits_by_date(df_feat, test_years=test_years, val_years=val_years, purge_days=purge_days)
#     else:
#         train, val, test = time_splits(df_feat, val_frac=0.15, test_frac=0.15, purge_days=purge_days)

#     # 3) Targets
#     if target_mode == "horizon":
#         for part in (train, val, test):
#             add_targets_horizon_inplace(part, H=H, eps=eps)
#     elif target_mode == "triple_barrier":
#         for part in (train, val, test):
#             add_targets_triple_barrier_inplace(part, H=tb_H, atr_mult=tb_atr_mult)
#     elif target_mode == "riskadj_sign":
#         for name, part in zip(['train','val','test'], (train, val, test)):
#             df_tmp = add_targets_risk_adjusted_sign_inplace(part, H_min=risk_H_min, H_max=risk_H_max, eps=eps)
#             if name == 'train':   train = df_tmp
#             elif name == 'val':   val = df_tmp
#             else:                 test = df_tmp
#     elif target_mode == "excess_sign":
#         spy_close, _vix = _download_market(period=data_period)
#         for name, part in zip(['train','val','test'], (train, val, test)):
#             df_tmp = add_targets_excess_sign_inplace(part, market_close=spy_close, H=max(H, risk_H_max), eps=eps)
#             if name == 'train':   train = df_tmp
#             elif name == 'val':   val = df_tmp
#             else:                 test = df_tmp
#     else:
#         raise ValueError(f"Unknown target_mode: {target_mode}")

#     # 4) Features
#     features = [
#         'LogReturn','RSI','MACD','SignalLine','BB_upper','BB_lower',
#         'r1','r5','r10','ATRp','pctB','distMA50'
#     ]

#     # 5) Scale on TRAIN only
#     scaler, train, val, test = scale_by_train(train, val, test, features)

#     # 6) Datasets
#     dtr = TimeSeriesDataset(train, features, seq_len)
#     dva = TimeSeriesDataset(val, features, seq_len)
#     dte = TimeSeriesDataset(test, features, seq_len)
#     if len(dtr) == 0 or len(dva) == 0 or len(dte) == 0:
#         raise ValueError("One of the splits is too short after seq_len; adjust years/fractions/seq_len.")

#     Xtr, ytr = dtr.X, dtr.y
#     Xva, yva = dva.X, dva.y
#     Xte, yte = dte.X, dte.y

#     # 6b) Sample weights
#     wtr = None
#     if weight_by_future_abs and 'FwdRet_H' in train.columns:
#         wtr = torch.tensor(
#             np.clip(train.loc[dtr.index, 'FwdRet_H'].abs().values, 1e-6, None),
#             dtype=torch.float32
#         )
#         wtr = wtr / (wtr.mean() + 1e-9)

#     # 7) Train
#     device = torch.device(device_str) if device_str else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#     model = TransformerClassifier(in_features=len(features), seq_len=seq_len).to(device)
#     model = train_with_early_stopping(
#         model, Xtr, ytr, Xva, yva,
#         epochs=epochs, patience=patience,
#         weights_tr=wtr, use_focal=use_focal_loss
#     )

#     # 8) Probs for each split
#     with torch.no_grad():
#         p_tr = torch.softmax(model(dtr.X.to(device)), dim=1)[:, 1].cpu().numpy()
#         p_va = torch.softmax(model(dva.X.to(device)), dim=1)[:, 1].cpu().numpy()
#         p_te_raw = torch.softmax(model(dte.X.to(device)), dim=1)[:, 1].cpu().numpy()
#         print(f"Test set prob_up range (raw): min={p_te_raw.min():.3f}, max={p_te_raw.max():.3f}, mean={p_te_raw.mean():.3f}")

#     # C1) Optional Platt calibration (keep calibrator for live)
#     p_te = p_te_raw.copy()
#     platt_cal = None
#     if calibrate_platt:
#         p_te, platt_cal = platt_fit_apply(p_va, yva.numpy(), p_te_raw)
#         print("Applied Platt calibration to TEST probabilities (fit on VAL).")

#     # 9) Metrics & decision rule
#     if use_band:
#         # C2) Use frozen band if present, save if newly picked
#         band_dir = os.path.join("artifacts", "bands")
#         try:
#             thr_entry, thr_exit, _meta = load_band(ticker, band_dir)
#             print(f"[{ticker}] Using FROZEN band from disk: entry={thr_entry:.3f} exit={thr_exit:.3f}")
#             auto_band_from_val = False
#         except Exception:
#             pass

#         if auto_band_from_val:
#             trend_ok_val = (df_feat['Close'] > df_feat['MA200']) & (df_feat['MA200'].pct_change(20) > 0)
#             # keep ATRp-based gate here just for VAL band-pick (as before)
#             vol_ok_val   = df_feat['ATRp'].between(atrp_min, atrp_max)
#             if band_objective == "riskaware":
#                 picked = pick_band_on_val_riskaware(
#                     p_va, dva.index, df_feat['Open'],
#                     df_feat_val=df_feat,
#                     min_trades=max(min_trades_val, 20),
#                     enforce_entry_min=thr_entry,
#                     gap=max(0.12, thr_entry - thr_exit),
#                     lam=lam, gamma=gamma,
#                     use_trend_gate=use_trend_gate, use_vol_gate=use_vol_gate,
#                     trend_gate_s=trend_ok_val.reindex(dva.index),
#                     vol_gate_s=vol_ok_val.reindex(dva.index),
#                     gate_relax_penalty=10.0,
#                     init_cash=init_cash, fees=fees, slippage=slippage
#                 )
#             else:
#                 picked = pick_band_on_val(
#                     p_va, dva.index, df_feat['Open'],
#                     y_va=yva.numpy(), min_trades=min_trades_val,
#                     entry_grid=np.linspace(max(0.52, thr_entry-0.10), 0.90, 41),
#                     gap=max(0.20, thr_entry - thr_exit),
#                     init_cash=init_cash, fees=fees, slippage=slippage,
#                     objective=band_objective
#                 )
#             if picked is not None:
#                 score, thr_entry, thr_exit, trades, _stats = picked
#                 print(f"[Auto-band:{band_objective}] score={score:.2f} | thr_entry={thr_entry:.3f} thr_exit={thr_exit:.3f} | trades={int(trades)}")
#             else:
#                 print("[Auto-band] No band met min trade count on validation; using provided band.")

#         # Persist the chosen band
#         try:
#             save_band(ticker, {"entry": float(thr_entry), "exit": float(thr_exit)}, band_dir)
#         except Exception as e:
#             print(f"[{ticker}] Warning: couldn't save band -> {e}")

#         print(f"Using band: entry >= {thr_entry:.2f}, exit < {thr_exit:.2f}")
#         m_tr = eval_classification_band(p_tr, ytr, dtr.index, thr_entry, thr_exit)
#         m_va = eval_classification_band(p_va, yva, dva.index, thr_entry, thr_exit)
#         m_te = eval_classification_band(p_te, yte, dte.index, thr_entry, thr_exit)
#         raw_state_test = m_te["state"]
#         state_test = raw_state_test.copy()
#         thr = None
#     else:
#         prec, thr, cnt = pick_threshold_for_precision(p_va, yva.numpy(), min_signals=12)
#         if prec == 0.0:
#             thr = pick_threshold_from_val(model, Xva, yva)
#         thr = max(thr, 0.60)
#         print(f"Chosen threshold for precision: {thr:.3f}")
#         m_tr = eval_classification(model, Xtr, ytr, thr)
#         m_va = eval_classification(model, Xva, yva, thr)
#         m_te = eval_classification(model, Xte, yte, thr)
#         raw_state_test = pd.Series(m_te['probs'] >= thr, index=dte.index, name='state')
#         state_test = raw_state_test.copy()

#     print(f"Train  acc={m_tr['acc']:.3f} prec={m_tr['prec']:.3f} rec={m_tr['rec']:.3f} f1={m_tr['f1']:.3f}")
#     print(f"Val    acc={m_va['acc']:.3f} prec={m_va['prec']:.3f} rec={m_va['rec']:.3f} f1={m_va['f1']:.3f}")
#     print(f"Test   acc={m_te['acc']:.3f} prec={m_te['prec']:.3f} rec={m_te['rec']:.3f} f1={m_te['f1']:.3f}")

#     # 10) Regime & volatility gating on TEST + optional override
#     trend_ok = (df_feat['Close'] > df_feat['MA200']) & (df_feat['MA200'].pct_change(20) > 0)
#     trend_gate = trend_ok.reindex(state_test.index).fillna(False) if use_trend_gate else pd.Series(True, index=state_test.index)

#     # C3) Quantile-based realized vol threshold fitted on TRAIN
#     rv_full  = realized_vol(df_feat['Close'], lookback=60)
#     rv_train = rv_full.reindex(train.index).dropna()
#     vol_thr  = fit_vol_gate_threshold(rv_train, q_high=0.80)
#     vol_ok   = (rv_full <= vol_thr)
#     vol_gate = vol_ok.reindex(state_test.index).fillna(False) if use_vol_gate else pd.Series(True, index=state_test.index)

#     override = pd.Series(False, index=state_test.index)
#     if override_gate_prob is not None:
#         active_probs = m_te['probs']
#         override = pd.Series(active_probs, index=state_test.index) >= float(override_gate_prob)

#     # C4) Entry-only gating on TEST
#     gated_state_test = apply_entry_only_gates(state_test, trend_gate, vol_gate, override).rename('state')
#     print(f"Trend gate pass rate: {trend_gate.mean():.3f}")
#     print(f"Vol gate  pass rate: {vol_gate.mean():.3f}")
#     _print_signal_summary(ticker, gated_state_test)

#     # 11) Backtest on TEST (next-bar execution at next OPEN)
#     price_test = df_feat['Open'].reindex(gated_state_test.index).astype(float)
#     pf, stats = backtest_state(
#         price_test, gated_state_test,
#         init_cash=init_cash, fees=fees, slippage=slippage,
#         execution="next_open"
#     )
#     print("\n--- Backtest (TEST split, next-bar OPEN, after costs; gated) ---")
#     print(stats)
#     _print_bt_subset(ticker, stats)

#     # Baseline comparison (optional)
#     if evaluate_baseline:
#         base_state = baseline_ma_atr_signals(df_feat).reindex(gated_state_test.index).fillna(False)
#         pf_b, st_b = backtest_state(
#             price_test, base_state,
#             init_cash=init_cash, fees=fees, slippage=slippage,
#             execution="next_open"
#         )
#         print("\n--- Baseline (MA+ATR) backtest ---")
#         print(st_b)
#         try:
#             mret = float(stats.get('Total Return [%]', 0.0))
#             bret = float(st_b.get('Total Return [%]', 0.0))
#             print(f"Model beats baseline? {mret:.2f}% vs {bret:.2f}% -> {mret > max(bret, 0.0)}")
#         except Exception:
#             pass

#     # 12) After-close single-line instruction — ALWAYS yesterday's US/Eastern close (or nearest <=)
#     msg = None

#     report_date = latest_us_trading_date()
#     if report_date in df_feat.index:
#         eval_date = report_date
#     else:
#         try:
#             eval_date = df_feat.index[df_feat.index.get_loc(report_date, method='pad')]
#         except Exception:
#             eval_date = df_feat.index.max()

#     live_close = float(df_feat['Close'].loc[eval_date])

#     if use_band:
#         if eval_date in dte.index:
#             # Recompute prob live for that exact date (still inside TEST)
#             loc = dte.index.get_loc(eval_date)
#             p_live = infer_live_probs_basic(
#                 df_full_feats=df_feat, features=features, scaler=scaler, model=model,
#                 seq_len=seq_len, dates=pd.DatetimeIndex([eval_date])
#             )
#             prob_today = float(p_live.iloc[0])

#             # C5) Apply Platt calibration to live prob if enabled
#             if calibrate_platt and (platt_cal is not None):
#                 prob_today = float(platt_cal.transform(np.array([prob_today]))[0])

#             # Keep states from precomputed TEST artifacts
#             raw_curr      = bool(raw_state_test.iloc[loc])
#             curr_final    = bool(gated_state_test.iloc[loc])
#             prev_final    = bool(gated_state_test.shift(1, fill_value=False).iloc[loc])
#             # gate checks at eval_date
#             trend_ok_today = bool(trend_ok.reindex([eval_date]).fillna(False).iloc[0]) if use_trend_gate else True
#             vol_ok_today   = bool(vol_ok.reindex([eval_date]).fillna(False).iloc[0])   if use_vol_gate   else True
#             override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
#         else:
#             # Tail beyond TEST: infer prob, roll band by one, then entry-only gate
#             p_tail = infer_live_probs_basic(
#                 df_full_feats=df_feat, features=features, scaler=scaler, model=model,
#                 seq_len=seq_len, dates=pd.DatetimeIndex([eval_date])
#             )
#             prob_today = float(p_tail.iloc[0])

#             # C5) Apply Platt to tail prob if enabled
#             if calibrate_platt and (platt_cal is not None):
#                 prob_today = float(platt_cal.transform(np.array([prob_today]))[0])

#             prev_raw       = bool(raw_state_test.iloc[-1]) if len(raw_state_test) else False
#             raw_curr       = bool(prob_today >= (thr_exit if prev_raw else thr_entry))
#             prev_final     = bool(gated_state_test.iloc[-1]) if len(gated_state_test) else False
#             trend_ok_today = bool(trend_ok.reindex([eval_date]).fillna(False).iloc[0]) if use_trend_gate else True
#             vol_ok_today   = bool(vol_ok.reindex([eval_date]).fillna(False).iloc[0])   if use_vol_gate   else True
#             override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
#             # C6) Entry-only decision for tail
#             curr_final     = entry_only_gate(prev_final, raw_curr, trend_ok_today, vol_ok_today, override_used)
#     else:
#         # Threshold mode
#         if eval_date in dte.index:
#             p_live = infer_live_probs_basic(
#                 df_full_feats=df_feat, features=features, scaler=scaler, model=model,
#                 seq_len=seq_len, dates=pd.DatetimeIndex([eval_date])
#             )
#             prob_today = float(p_live.iloc[0])
#             if calibrate_platt and (platt_cal is not None):
#                 prob_today = float(platt_cal.transform(np.array([prob_today]))[0])
#         else:
#             p_tail = infer_live_probs_basic(
#                 df_full_feats=df_feat, features=features, scaler=scaler, model=model,
#                 seq_len=seq_len, dates=pd.DatetimeIndex([eval_date])
#             )
#             prob_today = float(p_tail.iloc[0])
#             if calibrate_platt and (platt_cal is not None):
#                 prob_today = float(platt_cal.transform(np.array([prob_today]))[0])

#         raw_curr       = bool(prob_today >= (thr if thr is not None else 0.5))
#         prev_final     = bool(gated_state_test.iloc[-1]) if len(gated_state_test) else False
#         trend_ok_today = bool(trend_ok.reindex([eval_date]).fillna(False).iloc[0]) if use_trend_gate else True
#         vol_ok_today   = bool(vol_ok.reindex([eval_date]).fillna(False).iloc[0])   if use_vol_gate   else True
#         override_used  = (override_gate_prob is not None) and (prob_today >= float(override_gate_prob))
#         curr_final     = bool(raw_curr and ((trend_ok_today and vol_ok_today) or override_used))

#     msg = _format_after_close_message(
#         ticker=ticker,
#         live_date=eval_date,
#         prev_final=prev_final,
#         curr_final=curr_final,
#         curr_raw=raw_curr,
#         prob_today=float(prob_today),
#         thr_entry=float(thr_entry),
#         thr_exit=float(thr_exit),
#         trend_ok_today=bool(trend_ok_today),
#         vol_ok_today=bool(vol_ok_today),
#         topn_kept_today=True,
#         override_used=bool(override_used),
#         live_close=live_close
#     )

#     print("\n=== After-Close Signal (yesterday close) ===")
#     print(msg)
#     try:
#         if msg.startswith("Enter Next Opening"):
#             notify("Transformer Signal: ENTRY", msg)
#         elif msg.startswith("Exit Next Opening"):
#             notify("Transformer Signal: EXIT", msg)
#     except Exception:
#         pass


#     # 13) "Today" inference (convenience)
#     live_date_inf, live_prob = infer_today_prob(df_feat, features, scaler, model, seq_len=seq_len)
#     if live_date_inf is not None:
#         # C7) Apply Platt to today's prob
#         if calibrate_platt and (platt_cal is not None) and (live_prob is not None):
#             live_prob = float(platt_cal.transform(np.array([live_prob]))[0])

#         next_ix = df_feat.index[df_feat.index > live_date_inf]
#         next_date = next_ix[0] if len(next_ix) else None

#         prev_idx = gated_state_test.index[gated_state_test.index < live_date_inf]
#         st_yday = bool(gated_state_test.loc[prev_idx[-1]]) if len(prev_idx) else False

#         if use_band:
#             st_today_raw = (live_prob >= (thr_exit if st_yday else thr_entry))
#         else:
#             st_today_raw = (live_prob >= (thr if thr is not None else 0.5))

#         trend_ok_today = bool(trend_ok.reindex([live_date_inf]).fillna(False).iloc[0]) if use_trend_gate else True
#         vol_ok_today   = bool(vol_ok.reindex([live_date_inf]).fillna(False).iloc[0])   if use_vol_gate   else True
#         override_used  = (override_gate_prob is not None) and (live_prob >= float(override_gate_prob))

#         st_today = entry_only_gate(prev_final=st_yday, raw_curr=st_today_raw,
#                                    trend_ok_today=trend_ok_today, vol_ok_today=vol_ok_today,
#                                    override_used=override_used)

#         entry_today = (not st_yday) and st_today
#         exit_today  = st_yday and (not st_today)
#         close_today = float(df_feat['Close'].reindex([live_date_inf]).iloc[0])

#         when = f"next session ({next_date.date()})" if next_date is not None else "next session (no further bar available)"
#         if entry_today:
#             msg2 = f"{ticker}: ENTRY signal for {when} | prob_up={live_prob:.2%} | today_close=${close_today:,.2f}"
#             notify("Transformer Signal: ENTRY", msg2); print(msg2)
#         elif exit_today:
#             msg2 = f"{ticker}: EXIT signal for {when} | prob_up={live_prob:.2%} | today_close=${close_today:,.2f}"
#             notify("Transformer Signal: EXIT", msg2); print(msg2)
#         else:
#             state = "LONG" if st_today else "FLAT"
#             print(f"{ticker}: No new signal ({when}). State {state}. prob_up={live_prob:.2%} | today_close=${close_today:,.2f}")

#     return {
#         "model": model,
#         "scaler": scaler,
#         "threshold": thr if not use_band else None,
#         "band": (thr_entry, thr_exit) if use_band else None,
#         "datasets": {"train": dtr, "val": dva, "test": dte},
#         "metrics": {"train": m_tr, "val": m_va, "test": m_te},
#         "backtest": {"portfolio": pf, "stats": stats},
#         "features_df": df_feat
#     }


# # ---------- Run ----------
# if __name__ == "__main__":

#     total_start = 0.0
#     total_end = 0.0
#     total_beats = 0
#     ok_count = 0

#     tickers = [
#         "MSFT", "NVDA", "META",
#         "LRCX", "ICE",
#         "ORLY", "GS", "BLK", "PGR", "LLY",
#         "MRK", "ISRG", "HCA", "ETN",
#         "AAPL", "AMZN", "GOOGL", "CRM",
#         "CME", "TSLA", "COST", "V", "MCK",
#         "CAT", "ROK", "CVX",
#     ]


#     USE_CROSS_SECTIONAL = True  # set True to run the cross-sectional pipeline with calibration

#     if USE_CROSS_SECTIONAL:
#         # Cross-sectional training + calibrated bands + baseline comparison + richer per-ticker reports
#         run_universe_pipeline(
#             tickers=tickers,
#             seq_len=30,
#             epochs=120,
#             patience=8,
#             data_period="max",
#             split_mode="by_date",
#             val_years=3, test_years=5,
#             target_mode="riskadj_sign",  # or "excess_sign"
#             H=10, eps=0.0,
#             H_min=5, H_max=10,
#             use_band=True,
#             thr_entry=0.60,
#             thr_exit=0.45,
#             auto_band_from_val=True,
#             band_objective="precision",   # also supports: "riskaware", "winrate", "return"
#             min_trades_val=10,
#             lam=0.3, gamma=0.2,
#             use_trend_gate=True,
#             use_vol_gate=True,
#             atrp_min=0.004, atrp_max=0.10,
#             evaluate_baseline=True,
#             override_gate_prob=None,  # e.g., 0.92 to bypass gates on very high probs
#             top_n_per_date=0,         # set >0 to cap positions per day across names
#             init_cash=10_000, fees=0.001, slippage=0.001
#         )
#         print("\n[MAIN] Cross-sectional run complete.")
#     else:
#         # Single-name pipeline — prints the detailed report and an after-close single-line signal
#         for t in tickers:
#             try:
#                 res = run_ticker_pipeline(
#                     t,
#                     seq_len=30,
#                     epochs=120,
#                     patience=8,
#                     purge_days=3,
#                     init_cash=10_000,
#                     fees=0.001,
#                     slippage=0.001,
#                     data_period="10y",
#                     split_mode="by_date",
#                     val_years=1,
#                     test_years=2,
#                     target_mode="riskadj_sign",   # try: "riskadj_sign" / "excess_sign" / "horizon"
#                     H=3, eps=0.0,
#                     use_band=True,
#                     thr_entry=0.55,               # auto-band may raise this
#                     thr_exit=0.42,
#                     auto_band_from_val=True,      # enable to use band search
#                     band_objective="precision",   # "riskaware" / "winrate" / "precision" / "return"
#                     min_trades_val=10,
#                     use_trend_gate=True,
#                     use_vol_gate=True,
#                     atrp_min=0.004,
#                     atrp_max=0.10,
#                     use_focal_loss=False,
#                     weight_by_future_abs=True,
#                     evaluate_baseline=True,
#                     lam=0.3, gamma=0.2,
#                     calibrate_platt=True,
#                     override_gate_prob=None
#                 )

#                 # Aggregate Start/End values
#                 stats = res["backtest"]["stats"]
#                 sv = float(stats.get("Start Value", np.nan))
#                 ev = float(stats.get("End Value", np.nan))
#                 if not (np.isnan(sv) or np.isnan(ev)):
#                     total_start += sv
#                     total_end += ev
#                     ok_count += 1

#             except Exception as e:
#                 print(f"{t}: error -> {e}")

#         # Final aggregate summary
#         if ok_count > 0:
#             pnl = total_end - total_start
#             ret_pct = (total_end / total_start - 1.0) * 100.0 if total_start > 0 else float("nan")
#             print("\n=== Aggregate summary ===")
#             print(f"Tickers processed: {ok_count}")
#             print(f"Total Start Value: ${total_start:,.2f}")
#             print(f"Total End Value:   ${total_end:,.2f}")
#             print(f"Total PnL:         ${pnl:,.2f}  ({ret_pct:,.2f}%)")
#         else:
#             print("\nNo successful backtests to summarize.")

#     # Final confirmation print so the script always ends with output
#     print("\n=== Script finished ===")
